# From Pixels to Provenance — full pipeline notebook

Automated galaxy-morphology classification in HST archival imaging (NUS-ISS Practice Module, Sangeeta Bahrani, A0339780R).

This notebook contains **the complete code of the 24 October 2026 release** (27 stages, `config.yaml`, tools). It runs the same way on Google Colab and in a local Jupyter session, on **any** set of HST `_drc` / `_drz` files: one pointing or several pointings at different RA/Dec.

**How to use it**

1. **Section 1 – Setup**: choose the project folder (on Colab: Google Drive, so outputs survive the session) and install the packages.
2. **Section 2 – Write the code**: run every cell once. Each `%%writefile` cell writes one file of the project (`src/p2p/...`, `tools/...`, `config.yaml`, `run_pipeline.py`). To change a setting, edit the `config.yaml` cell and run it again.
3. **Section 3 – Data**: put your FITS files in `data/raw/` and the GZH / Gaia / CANDELS tables in `data/catalogs/` (or download them with the tool cells).
4. **Section 4 – Run**: either run everything in one cell, or stage by stage. After each stage, `show_stage("…")` prints what the stage did, its key numbers and its figures.
5. **Section 5 – Results**: inspect `catalog.csv`.

A GPU (Colab: *Runtime → Change runtime type → GPU*) speeds up Stages 1b, 2a and 4a–4d; the rest runs on CPU.

## 1. Setup

In [ ]:
import os, sys, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/pixels2provenance"     # change if you like
else:
    PROJECT_DIR = os.path.abspath("pixels2provenance_nb")        # next to this notebook

for d in ["src/p2p/stages", "tools", "data/raw", "data/catalogs", "outputs"]:
    os.makedirs(os.path.join(PROJECT_DIR, d), exist_ok=True)
os.chdir(PROJECT_DIR)
print("Project folder:", os.getcwd())

The `%%writefile` cells below write relative paths, so the working directory must be the project folder. Re-run the cell above whenever you restart the kernel.

In [ ]:
%%writefile requirements.txt
# Python 3.10 - 3.13
numpy>=1.24
scipy>=1.10
pandas>=2.0
matplotlib>=3.7
pyyaml>=6.0
joblib>=1.3
astropy>=6.0
photutils>=2.0          # 2.x and 3.x both supported (src/p2p/compat.py)
statmorph>=0.5          # if the build fails: pip install --no-build-isolation statmorph
scikit-learn>=1.3
hdbscan>=0.8.33
xgboost>=2.0
torch>=2.1              # CPU build is enough; CUDA / Apple MPS are used automatically if present
torchvision>=0.16
astroquery>=0.4.7       # only for downloading from MAST / querying Gaia online

In [ ]:
# Install the packages (Colab already has numpy, scipy, pandas, matplotlib, scikit-learn, torch, torchvision)
%pip install -q -r requirements.txt
# statmorph sometimes needs this on fresh environments:
# %pip install -q --no-build-isolation statmorph

In [ ]:
import importlib
for m in ["numpy", "scipy", "pandas", "astropy", "photutils", "statmorph", "sklearn", "hdbscan", "xgboost", "torch", "torchvision"]:
    try:
        mod = importlib.import_module(m)
        print(f"{m:12s} {getattr(mod, '__version__', '?')}")
    except Exception as e:
        print(f"{m:12s} MISSING  ({e})")
import torch
print("CUDA available:", torch.cuda.is_available())

## 2. Write the project code

Run every cell in this section once. The files are identical to the release on your Mac (`src/`, `tools/`, `config.yaml`, `run_pipeline.py`).

### 2.1 Configuration — `config.yaml`
All parameters of every stage. Edit here and re-run this cell to change a setting.

In [ ]:
%%writefile config.yaml
# =====================================================================
# From Pixels to Provenance - pipeline configuration
# Every stage reads its settings from this file. Paths are relative to
# the project folder (the folder that holds this file).
# =====================================================================

project:
  name: "From Pixels to Provenance"
  seed: 42
  n_jobs: -1                 # CPU workers for statmorph / clustering (-1 = all cores)
  device: auto               # auto | cpu | cuda | mps   (deep-learning stages)

paths:
  raw: data/raw              # put your *_drc.fits / *_drz.fits files here
  catalogs: data/catalogs    # GZH / Gaia / CANDELS tables (CSV, CSV.GZ or FITS)
  outputs: outputs           # every stage writes into outputs/<stage folder>/

# ---------------------------------------------------------------- Stage 0
partition:
  mode: auto                 # auto = derive regions from the FITS footprints | manual
  collar_arcmin: 0.1         # buffer ring around every region (proposal: 2' for large mosaics;
                             # a single ACS pointing is only ~3.4' wide: 0.1' = 6" keeps stripes usable)
  fractions: {train: 0.70, dev: 0.15, test: 0.15}
  min_pointings_for_whole_assignment: 3   # fewer pointings -> split each pointing into stripes
  external_patterns: []      # filename / TARGNAME substrings that mark CANDELS (external) pointings,
                             # e.g. ["EGS", "egs"]. Everything else is GZH train/dev/test.
  phase7_region: auto        # auto = external pointings if any, else the held-out TEST split
                             # (one field: Phase 7 is evaluated on galaxies never trained on) | external | test
  manual_regions: []         # used when mode: manual, e.g.
  #  - {name: COSMOS_A, ra: [150.05, 150.15], dec: [2.15, 2.25], split: train}

# ---------------------------------------------------------------- Stage 1a
acquisition:
  file_patterns: ["*_drc.fits", "*_drz.fits", "*_drc.fits.gz", "*_drz.fits.gz"]
  group_radius_arcsec: 60    # files whose centres are closer than this form one pointing ...
  group_min_overlap: 0.5     # ... and so do files whose footprints overlap by >= 50% of the smaller one
                             # (bands of one field taken by different HST programmes)
  bands: auto                # auto | list such as [F435W, F606W, F814W] (blue -> red)
  max_bands: 3
  detection_band: auto       # auto = reddest band in each pointing
  download:                  # optional: fetch files from MAST (needs internet + astroquery)
    enabled: false
    targets:
      - {ra: 53.12, dec: -27.80, radius_deg: 0.03, instrument: "ACS/WFC", filters: [F435W, F606W, F814W]}
    max_files_per_target: 3

# ---------------------------------------------------------------- Stage 1b
frame_qc:
  tile: 128
  train_stride: 128
  score_stride: 64           # 50% overlap so artefacts on a tile edge are not split
  min_tile_coverage: 0.9
  max_train_tiles: 4000
  latent_dim: 128
  epochs: 8                  # minimum epochs ...
  min_steps: 1000            # ... and at least this many optimisation steps
  batch_size: 64
  threshold_percentile: 99.0
  exclude_flagged: false     # true = drop sources in flagged tiles from training. Default false:
                             # with no labelled artefacts the CAE also flags rare bright galaxies.

# ---------------------------------------------------------------- Stage 1c
detection:
  background_box: 64
  background_filter: 3
  kernel_fwhm_px: 3.0
  nsigma: 2.0
  npixels: 10
  deblend_nlevels: 32
  deblend_contrast: 0.001
  correlated_noise_factor: 1.0   # drizzled noise is correlated; >1 inflates the rms (see README)
  # ---- band registration (bands from different HST programmes can disagree by ~1") ----
  register_bands: true           # measure each band's offset against the detection band and remove it
  register_search_arcsec: 3.0    # largest offset searched for
  max_band_offset_arcsec: 0.05   # Stage 1c FAILS if a band is still offset by more than this
  register_min_significance: 5.0 # the offset peak must hold >= 5x the pairs expected by chance
  band_offsets_arcsec: {}        # known offsets instead of measured ones, band minus detection band,
                                 # [dRA cos(dec), dDec] in arcsec, e.g. {F475W: [0.154, 0.866]}
  # ---- cosmic rays (single-exposure products cannot reject them) ----
  cr_rejection: two_band         # two_band = a source must be confirmed by a second band | none
  confirm_snr: 3.0               # S/N in another covering band that confirms a source as real ...
  confirm_max_shift_px: 1.5      # ... with that band's light centred within 1.5 px of the detection,
  confirm_max_shift_frac: 0.5    # or within 0.5 x the equivalent radius for large galaxies
  cr_pixel_nsigma: 5.0           # a cutout pixel is a cosmic-ray hit if it is > 5 sigma above ...
  cr_pixel_ratio: 4.0            # ... AND > 4 x what every other band predicts from the source colour
  cr_mask_grow_px: 0             # extra growth of the masked area (each hit is already grown by 1 px)
  mag_min_snr: 2.0               # mag_<band> (and colours) only where the band detects the source at S/N >= 2
  max_sources_per_pointing: 8000 # confirmed (real) sources: brightest kept if more are detected
  max_cr_candidates_per_pointing: 1000   # cosmic-ray candidates kept (as Noise), capped separately
  cutout_size: 128
  cutout_pixscale_arcsec: auto   # auto = detection band's pixel scale
  min_valid_fraction: 0.90
  core_radius_px: 16

# ---------------------------------------------------------------- Stage 1d
psf_match:
  stamp: 25
  min_stars: 5
  max_stars: 40
  star_min_snr: 50
  star_max_ellipticity: 0.3  # drizzled ACS stars measure up to ~0.23 in a single exposure
  star_isolation_px: 5       # no other segment this close to a PSF star's centre (further out: masked)
  star_min_snr_per_band: 30  # a star shapes a band's PSF only if it has S/N >= 30 in that band
  require_stars: true        # stop when no PSF star passes the cuts (false = continue with the Gaussian fallback)
  fallback_fwhm_arcsec: {ACS: 0.10, WFC3_UVIS: 0.08, WFC3_IR: 0.15, default: 0.10}  # approximate
  kernel: gaussian           # gaussian (robust default) | photutils (Fourier ratio, needs clean PSFs)
  window: 0.35               # photutils TopHatWindow beta (kernel: photutils only)

# ---------------------------------------------------------------- Stage 1e / 5b
morphology:
  min_snr: 3.0               # fainter sources get NaN morphology + flag 4
  sersic_maxiter: 500
  sersic_n_bounds: [0.05, 8.0]   # a fit ending at or beyond these limits gets flag_sersic = 1
  sersic_rhalf_max_px: 64        # ... and so does r_half beyond half the 128-px cutout
  pass2_mode: psf_fit        # psf_fit = statmorph Sersic fit convolved with the ePSF
                             # deconvolve = Richardson-Lucy deconvolution, then statmorph
  rl_iterations: 10

# ---------------------------------------------------------------- Stage 2a-2e
representation:
  latent_dim: 64
  epochs: 15                 # minimum epochs ...
  min_steps: 1500            # ... and at least this many optimisation steps (few cutouts -> more epochs)
  batch_size: 64
  n_pca: 4                   # MAXIMUM number of CAE-latent principal components for the clustering space
  pca_min_variance: 0.02     # ...and drop any of them explaining < 2% of the latent variance (see pca_scree.png)
  spatial_weight: 0.5        # weight of the image block vs the physical block
  tabular_features: [fwhm_over_psf, band_consistency, gini, m20, concentration, asymmetry, smoothness,
                     sersic_n, r_half, ellipticity, mag, snr, colour_1, colour_2]
clustering:
  k_min: 2
  k_max: 14
  covariance_type: diag            # diag | full (full needs many more objects per component)
  hdbscan_min_cluster_size: auto   # auto = max(10, n_train // 50)
  hdbscan_min_samples: 5
baseline:
  point_source_tolerance: 0.2     # |FWHM / stellar-locus FWHM - 1| <= this -> stellar locus
  star_locus_min_snr: 150          # size separates stars from compact galaxies only above this S/N
                                   # (GOODS-S single exposures: ~ mag 22.3; check stage2d size_magnitude.png)
  class_star_threshold: 0.5
  noise_snr_max: 3.0
  sub_psf_factor: 0.7              # FWHM below 0.7 x the stellar locus -> sharper than the PSF
  single_band_max: 0.05            # only used if sources.csv has no cr_candidate column (older runs)
semantic_audit:
  gaia_radius_arcsec: 0.5
  gzh_radius_arcsec: 0.5
  gaia_mag_limit: 21.0             # compared with the detection-band AB magnitude (approximate)
  montage_n: 25
gating:
  physical_gate: true              # object-level checks a cluster name must not contradict (Stage 2e docstring)
  transitional_p: 0.80
  outlier_max_p: 0.50
  recon_percentile: 99.0

# ---------------------------------------------------------------- Catalogues
catalogs:
  gzh:
    file: auto                     # auto = first file in data/catalogs matching *gz_hubble* or *gzh*
    match_radius_arcsec: 0.5
    estimate_offset: true          # remove the median astrometric offset before matching
    columns:                       # exact Galaxy Zoo: Hubble (Willett et al. 2017) names
      ra: RA
      dec: DEC
      smooth: t01_smooth_or_features_a01_smooth_debiased_fraction
      features: t01_smooth_or_features_a02_features_or_disk_debiased_fraction
      artifact: t01_smooth_or_features_a03_star_or_artifact_weighted_fraction
      edgeon: t02_edgeon_a01_yes_weighted_fraction
      spiral: t04_spiral_a01_spiral_weighted_fraction
      odd: t06_odd_a01_yes_weighted_fraction
      odd_irregular: t08_odd_feature_a04_irregular_weighted_fraction
      odd_disturbed: t08_odd_feature_a03_disturbed_weighted_fraction
      odd_merger: t08_odd_feature_a06_merger_weighted_fraction
      clumpy: t12_clumpy_a01_yes_weighted_fraction
    thresholds: {smooth: 0.6, features: 0.5, edgeon: 0.5, spiral: 0.5, irregular: 0.5,
                 odd: 0.4, merger: 0.5}
    boundary_agreement: 0.5
  gaia:
    file: auto                     # auto = *gaia* in data/catalogs (e.g. gaia_dr3.csv); or query online (below)
    query_online: false            # true = astroquery Gaia cone search per pointing
    columns: {source_id: source_id, ra: ra, dec: dec, parallax: parallax, pmra: pmra,
              pmdec: pmdec, g_mag: phot_g_mean_mag, teff: teff_gspphot}
    epoch: 2016.0
  candels:
    file: auto                     # auto = *kartaltepe* / *candels* / *k15* in data/catalogs (e.g. candels_k15.csv)
    columns:                       # Kartaltepe et al. (2015) FRACTIONAL table = VizieR J/ApJS/221/11/table3.
                                   # Lists = alternative names (VizieR web / TAP / astroquery spell them differently).
                                   # The five class fractions are REQUIRED. Check: python tools/check_catalog_columns.py
      ra: [RAJ2000, _RAJ2000, RA, ra]
      dec: [DEJ2000, _DEJ2000, DEC, dec]
      spheroid: ["f(Sph)", f_Sph_, fSph, f_spheroid]
      disk: ["f(Dk)", f_Dk_, fDk, f_disk]
      irregular: ["f(Ir)", f_Ir_, fIr, f_irr]
      point_source: ["f(PS)", f_PS_, fPS, f_ps]
      unclassifiable: ["f(UnCl)", f_UnCl_, fUnCl, f_unc]
      merger: ["f(M)", f_M_, fM, f_merger]
      n_classifiers: [NCl, Ncl, n_classifiers]   # used to keep the deepest row when a galaxy appears twice

# ---------------------------------------------------------------- Stage 4
supervised:
  pretrained: false          # true = ImageNet weights (needs internet the first time)
  epochs: 15
  batch_size: 32
  lr: 0.001
  weight_decay: 0.0001
  focal_gamma: 2.0
  merger_weight: 0.5
  tabular_only_epochs: 40
  ensemble_weights: [1.0, 1.0, 1.0]   # dual-branch, EfficientNet, XGBoost

# ---------------------------------------------------------------- Stage 8
catalogue:
  designation_prefix: "P2P J"
  dedup_radius_arcsec: 0.2

### 2.2 Runner — `run_pipeline.py`

In [ ]:
%%writefile run_pipeline.py
#!/usr/bin/env python
"""Run the From Pixels to Provenance pipeline.

Examples (from the project folder):
    python run_pipeline.py                  # every stage, in order
    python run_pipeline.py --list           # show the stages
    python run_pipeline.py --only 1c        # one stage (its inputs must exist)
    python run_pipeline.py --from 2a        # from 2a to the end
    python run_pipeline.py --from 1a --to 1e
"""
from __future__ import annotations

import argparse
import importlib
import json
import sys
import time
import traceback
import warnings
from pathlib import Path

ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT / "src"))

from p2p.context import STAGES, Ctx, MissingInput, StageSkipped, save_json  # noqa: E402


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--only", help="run a single stage, e.g. 1c")
    ap.add_argument("--from", dest="start", help="first stage to run")
    ap.add_argument("--to", dest="stop", help="last stage to run")
    ap.add_argument("--list", action="store_true", help="list the stages and exit")
    ap.add_argument("--config", default="config.yaml")
    ap.add_argument("--keep-going", action="store_true", help="continue after a failed stage")
    args = ap.parse_args()

    keys = [k for k, *_ in STAGES]
    if args.list:
        for k, mod, folder, title in STAGES:
            print(f"  {k:>3}  {title:45s} -> outputs/{folder}/   (src/p2p/stages/{mod}.py)")
        return
    if args.only:
        todo = [args.only]
    else:
        i0 = keys.index(args.start) if args.start else 0
        i1 = keys.index(args.stop) if args.stop else len(keys) - 1
        todo = keys[i0:i1 + 1]
    unknown = [k for k in todo if k not in keys]
    if unknown:
        sys.exit(f"Unknown stage(s): {unknown}. Use --list.")

    warnings.filterwarnings("ignore", category=RuntimeWarning)
    ctx = Ctx(ROOT, args.config)
    runlog_path = ctx.out / "run_log.json"
    runlog = json.loads(runlog_path.read_text()) if runlog_path.exists() else {}
    info = {k: (m, t) for k, m, _, t in STAGES}

    for k in todo:
        mod_name, title = info[k]
        ctx.log.info("=" * 78)
        ctx.log.info(f"Stage {k} - {title}")
        ctx.log.info("=" * 78)
        t0 = time.time()
        status = "ok"
        try:
            mod = importlib.import_module(f"p2p.stages.{mod_name}")
            summary = mod.run(ctx) or {}
        except StageSkipped as e:
            status, summary = "skipped", {"reason": str(e)}
            ctx.log.info(f"[Stage {k}] SKIPPED: {e}")
        except MissingInput as e:
            status, summary = "missing-input", {"reason": str(e)}
            ctx.log.info(f"[Stage {k}] CANNOT RUN: {e}")
            if not args.keep_going:
                _save(runlog, k, status, t0, summary, runlog_path)
                sys.exit(1)
        except Exception as e:
            status, summary = "failed", {"error": repr(e)}
            ctx.log.info(traceback.format_exc())
            if not args.keep_going:
                _save(runlog, k, status, t0, summary, runlog_path)
                sys.exit(1)
        _save(runlog, k, status, t0, summary, runlog_path)
        ctx.log.info(f"[Stage {k}] {status} in {time.time() - t0:.1f} s")
    ctx.log.info(f"Done. Outputs in {ctx.rel(ctx.out)} (final catalogue: stage8_catalogue/catalog.csv)")


def _save(runlog, k, status, t0, summary, path):
    runlog[k] = {"status": status, "seconds": round(time.time() - t0, 1),
                 "finished": time.strftime("%Y-%m-%d %H:%M:%S"), "summary": summary}
    save_json(runlog, path)


if __name__ == "__main__":
    main()

### 2.3 Shared package — `src/p2p/`
- `__init__.py`
- `catalogs.py`
- `compat.py`
- `context.py`
- `detection.py`
- `features.py`
- `fitsio.py`
- `inference.py`
- `morphology.py`
- `multiband.py`
- `nets.py`
- `utils.py`
- `xgb_worker.py`

In [ ]:
%%writefile src/p2p/__init__.py
"""From Pixels to Provenance - automated galaxy morphology from HST archival imaging."""
__version__ = "1.0.0"

In [ ]:
%%writefile src/p2p/catalogs.py
"""Loading the external catalogues (GZH, Gaia DR3, CANDELS) into a standard form.

Each loader returns a DataFrame with *standard* column names (ra, dec, smooth,
...) taken from the mapping in config.yaml, cut to the sky area of the
pointings so large tables (GZH has ~114k rows) stay small in memory."""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from .context import Ctx, StageSkipped

PATTERNS = {"gzh": ["*gz_hubble*", "*gzh*", "*galaxy_zoo*"],
            "gaia": ["*gaia*"],
            "candels": ["*kartaltepe*", "*candels*", "*k15*"]}


def find_catalog(ctx: Ctx, kind: str) -> Path | None:
    spec = ctx.cfg["catalogs"][kind].get("file", "auto")
    if spec and spec != "auto":
        p = ctx.root / spec if not Path(spec).is_absolute() else Path(spec)
        return p if p.exists() else None
    if not ctx.catalogs.exists():
        return None
    for pat in PATTERNS[kind]:
        hits = sorted(ctx.catalogs.glob(pat)) + sorted(ctx.catalogs.glob(f"*/{pat}"))     # also one folder down
        hits = [h for h in hits if h.is_file() and h.suffix.lower() in (".csv", ".gz", ".fits", ".fit", ".vot", ".xml", ".tsv")]
        if kind == "candels":                     # VizieR J/ApJS/221/11: table3 holds the class fractions
            hits = sorted(hits, key=lambda h: "table3" not in h.name.lower())
        if hits:
            return hits[0]
    return None


def _candidates(col) -> list[str]:
    """A mapping value may be one column name or a list of alternatives (first one found wins)."""
    return [str(c) for c in col] if isinstance(col, (list, tuple)) else [str(col)]


def _text_format(path: Path):
    """Separator and comment character of a text table. VizieR / Gaia-archive downloads may use ';' or
    tabs and '#' comment lines; GZH is a plain comma-separated file."""
    import gzip
    opener = gzip.open if path.name.lower().endswith(".gz") else open
    with opener(path, "rt", errors="replace") as f:
        first = ""
        for line in f:
            if line.strip() and not line.startswith("#"):
                first = line
                break
    sep = max([",", ";", "\t", "|"], key=first.count) if first else ","
    return (sep if first.count(sep) else ","), "#"


def read_table(path: Path, usecols=None) -> pd.DataFrame:
    name = path.name.lower()
    if name.endswith((".fits", ".fit", ".fits.gz")):
        from astropy.table import Table
        t = Table.read(path)
        names = [c for c in t.colnames if len(t[c].shape) <= 1]
        df = t[names].to_pandas()
        return df[[c for c in df.columns if c in usecols]] if usecols else df
    if name.endswith((".vot", ".xml", ".vot.gz")):
        from astropy.table import Table
        return Table.read(path, format="votable").to_pandas()
    sep, comment = _text_format(path)
    header = pd.read_csv(path, nrows=0, sep=sep, comment=comment).columns
    cols = [c for c in header if usecols is None or c in usecols]
    return pd.read_csv(path, usecols=cols, sep=sep, comment=comment, low_memory=False)


def read_mapped(path: Path, mapping: dict) -> pd.DataFrame:
    """Read only the mapped columns (case-insensitive) - keeps big tables light."""
    want = {c.lower() for v in mapping.values() for c in _candidates(v)}
    name = path.name.lower()
    if name.endswith((".csv", ".csv.gz", ".tsv", ".tsv.gz", ".txt", ".dat")):
        sep, comment = _text_format(path)
        header = pd.read_csv(path, nrows=0, sep=sep, comment=comment).columns
        cols = [c for c in header if c.strip().lower() in want]
        df = pd.read_csv(path, usecols=cols, sep=sep, comment=comment, low_memory=False)
        df.columns = [c.strip() for c in df.columns]
        return df
    df = read_table(path)
    return df[[c for c in df.columns if c.lower() in want]]


def _to_degrees(values, is_ra):
    """Numbers stay numbers; sexagesimal strings ('03:32:24.6' / '-27:48:31') are converted to degrees."""
    num = pd.to_numeric(values, errors="coerce")
    text = values.astype(str).str.strip()
    sexa = num.isna() & text.str.contains(r"^[+-]?\d+[: ]\d+[: ]\d", regex=True)
    if sexa.any():
        from astropy.coordinates import Angle
        import astropy.units as u
        unit = u.hourangle if is_ra else u.deg
        num = num.copy()
        num[sexa] = [Angle(t, unit=unit).degree for t in text[sexa]]
    return num


def standardise(df: pd.DataFrame, mapping: dict, kind: str, required=("ra", "dec")) -> pd.DataFrame:
    """Rename catalogue columns to standard names. Matching is case-insensitive and a mapping value
    may list alternatives. A missing REQUIRED column stops the run with the closest names; a missing
    optional one is filled with NaN. Rows without a usable position are dropped."""
    import difflib
    lower = {c.lower(): c for c in df.columns}
    out = pd.DataFrame(index=df.index)
    missing = []
    for std, col in mapping.items():
        src = next((lower[c.lower()] for c in _candidates(col) if c.lower() in lower), None)
        if src is None:
            missing.append((std, col))
            out[std] = np.nan
        elif std in ("ra", "dec"):
            out[std] = _to_degrees(df[src], is_ra=(std == "ra"))
        elif std in ("source_id", "id"):
            out[std] = df[src]
        else:
            out[std] = pd.to_numeric(df[src], errors="coerce")
    hard = [m for m in missing if m[0] in required]
    if hard:
        hints = {std: difflib.get_close_matches(str(_candidates(col)[0]), list(df.columns), n=3, cutoff=0.3)
                 for std, col in hard}
        raise ValueError(f"{kind}: required column(s) not found: {hard}. Closest names in the file: {hints}.\n"
                         f"Fix 'catalogs.{kind}.columns' in config.yaml (python tools/check_catalog_columns.py "
                         "lists every column).")
    if missing:
        print(f"[catalogs] {kind}: optional column(s) missing, filled with NaN: {missing}")
    bad = out["ra"].isna() | out["dec"].isna()
    if bad.mean() > 0.5:
        raise ValueError(f"{kind}: {bad.mean():.0%} of the rows have no readable RA/Dec - check the file format.")
    return out[~bad]


def sky_box(metas, margin_deg=0.02):
    pts = np.vstack([np.asarray(m["footprint"]) for m in metas])
    return (pts[:, 0].min() - margin_deg, pts[:, 0].max() + margin_deg,
            pts[:, 1].min() - margin_deg, pts[:, 1].max() + margin_deg)


def cut_to_box(df, box):
    r0, r1, d0, d1 = box
    return df[(df.ra >= r0) & (df.ra <= r1) & (df.dec >= d0) & (df.dec <= d1)].reset_index(drop=True)


# ---------------------------------------------------------------- GZH
def load_gzh(ctx: Ctx, metas, required=True) -> pd.DataFrame | None:
    path = find_catalog(ctx, "gzh")
    if path is None:
        if required:
            raise StageSkipped("No Galaxy Zoo: Hubble table in data/catalogs (e.g. gz_hubble_main.csv.gz). "
                               "See README section 'Catalogues'.")
        return None
    mapping = ctx.cfg["catalogs"]["gzh"]["columns"]
    df = standardise(read_mapped(path, mapping), mapping, "gzh")
    df["gzh_row"] = df.index.to_numpy()          # row number in the original GZH file
    return cut_to_box(df, sky_box(metas))


# ---------------------------------------------------------------- Gaia
def load_gaia(ctx: Ctx, metas) -> pd.DataFrame | None:
    cfg = ctx.cfg["catalogs"]["gaia"]
    path = find_catalog(ctx, "gaia")
    if path is not None:
        df = standardise(read_mapped(path, cfg["columns"]), cfg["columns"], "gaia")
        return cut_to_box(df, sky_box(metas))
    if cfg.get("query_online"):
        return query_gaia_online(ctx, metas)
    return None


def query_gaia_online(ctx: Ctx, metas) -> pd.DataFrame | None:
    try:
        from astroquery.gaia import Gaia
    except Exception:
        print("[catalogs] astroquery not installed - cannot query Gaia online.")
        return None
    r0, r1, d0, d1 = sky_box(metas)
    q = ("SELECT source_id, ra, dec, parallax, pmra, pmdec, phot_g_mean_mag, teff_gspphot "
         f"FROM gaiadr3.gaia_source WHERE ra BETWEEN {r0} AND {r1} AND dec BETWEEN {d0} AND {d1}")
    try:
        t = Gaia.launch_job_async(q).get_results().to_pandas()
    except Exception as e:
        print(f"[catalogs] Gaia query failed: {e}")
        return None
    out = ctx.catalogs / "gaia_dr3_query.csv"
    ctx.catalogs.mkdir(parents=True, exist_ok=True)
    t.to_csv(out, index=False)
    return standardise(t, ctx.cfg["catalogs"]["gaia"]["columns"], "gaia")


def propagate_gaia(gaia: pd.DataFrame, to_epoch: float, from_epoch: float = 2016.0) -> pd.DataFrame:
    """Move Gaia positions to the HST observation epoch with proper motions."""
    g = gaia.copy()
    if not np.isfinite(to_epoch):
        return g
    dt = to_epoch - from_epoch
    pmra = g["pmra"].fillna(0).to_numpy(float)          # mas/yr, includes cos(dec)
    pmdec = g["pmdec"].fillna(0).to_numpy(float)
    cosd = np.cos(np.deg2rad(g["dec"].to_numpy(float)))
    g["ra"] = g["ra"] + pmra * dt / 3.6e6 / np.maximum(cosd, 1e-6)
    g["dec"] = g["dec"] + pmdec * dt / 3.6e6
    return g


# ---------------------------------------------------------------- CANDELS
K15_CLASS_COLUMNS = ("spheroid", "disk", "irregular", "point_source", "unclassifiable")


def load_candels(ctx: Ctx, metas) -> pd.DataFrame | None:
    """Kartaltepe et al. (2015) FRACTIONAL table (VizieR J/ApJS/221/11/table3). The five class fractions
    are required: if they were silently NaN every galaxy would become 'spheroid'. The same galaxy can
    appear at two depths; the row with the most classifiers is kept."""
    path = find_catalog(ctx, "candels")
    if path is None:
        return None
    mapping = ctx.cfg["catalogs"]["candels"]["columns"]
    df = standardise(read_mapped(path, mapping), mapping, "candels", required=("ra", "dec") + K15_CLASS_COLUMNS)
    if "n_classifiers" in df and df.n_classifiers.notna().any():
        key = (df.ra * 3600 / 0.2).round().astype("int64").astype(str) + "_" + \
              (df.dec * 3600 / 0.2).round().astype("int64").astype(str)
        before = len(df)
        df = df.assign(_k=key).sort_values("n_classifiers", ascending=False).drop_duplicates("_k").drop(columns="_k")
        if before > len(df):
            print(f"[catalogs] candels: {before - len(df)} duplicate rows (same galaxy, other depth) removed")
    return cut_to_box(df.reset_index(drop=True), sky_box(metas))

In [ ]:
%%writefile src/p2p/compat.py
"""photutils renamed several arguments and attributes between 1.x/2.x and 3.x
(npixels -> n_pixels, nlevels -> n_levels, xcentroid -> x_centroid, ...).
These wrappers accept either version so the project runs on both."""
from __future__ import annotations

import inspect

import numpy as np

from photutils.segmentation import SourceCatalog, deblend_sources, detect_sources

_DEBLEND = inspect.signature(deblend_sources).parameters
_ALIASES = {"x_centroid": "xcentroid", "y_centroid": "ycentroid", "n_labels": "nlabels",
            "segment_area": "area", "semimajor_axis": "semimajor_sigma",
            "semiminor_axis": "semiminor_sigma", "kron_flux_err": "kron_fluxerr",
            "segment_flux_err": "segment_fluxerr"}


def detect(data, threshold, npixels, mask=None):
    return detect_sources(data, threshold, npixels, mask=mask)


def deblend(data, segm, npixels, nlevels=32, contrast=0.001):
    kw = {"contrast": contrast}
    kw["n_levels" if "n_levels" in _DEBLEND else "nlevels"] = nlevels
    if "progress_bar" in _DEBLEND:
        kw["progress_bar"] = False
    return deblend_sources(data, segm, npixels, **kw)


def attr(obj, name):
    if hasattr(obj, name):
        return getattr(obj, name)
    return getattr(obj, _ALIASES[name])


def values(obj, name):
    """Catalogue property as a plain float array (strips astropy units)."""
    v = attr(obj, name)
    v = getattr(v, "value", v)
    return np.asarray(v, dtype=float)


def n_labels(segm) -> int:
    if segm is None:
        return 0
    return int(attr(segm, "n_labels"))


__all__ = ["detect", "deblend", "attr", "values", "n_labels", "SourceCatalog"]

In [ ]:
%%writefile src/p2p/context.py
"""Shared run context: configuration, folders, logging and the stage registry.

Every stage module exposes ``run(ctx)``. A stage reads what earlier stages
wrote (through ``ctx.require`` / ``load_objects``) and writes its own files
into ``outputs/<stage folder>/`` so each step can be inspected on its own.
"""
from __future__ import annotations

import json
import logging
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

# (key, module name, output folder, title) in EXECUTION order.
# Phase 5 runs before Phase 4 so the supervised models train on the
# PSF-refined Pass-2 features they are deployed on.
STAGES = [
    ("0",  "stage0_partition",        "stage0_partition",        "Sky-region partitioning"),
    ("1a", "stage1a_acquisition",     "stage1a_acquisition",     "Archive acquisition / local scan"),
    ("1b", "stage1b_frame_qc",        "stage1b_frame_qc",        "Frame quality control (CAE)"),
    ("1c", "stage1c_detection",       "stage1c_detection",       "Detection, deblending, aligned cutouts"),
    ("1d", "stage1d_psf_match",       "stage1d_psf_match",       "Cross-filter PSF homogenisation"),
    ("1e", "stage1e_pass1_morphology", "stage1e_pass1_morphology", "Pass-1 photometry and morphology"),
    ("2a", "stage2a_representation",  "stage2a_representation",  "Representation learning + fusion"),
    ("2b", "stage2b_clustering",      "stage2b_clustering",      "Unsupervised clustering"),
    ("2c", "stage2c_baseline",        "stage2c_baseline",        "Baseline segregation + Gaia match"),
    ("2d", "stage2d_semantic_audit",  "stage2d_semantic_audit",  "Semantic audit"),
    ("2e", "stage2e_continuum_gating", "stage2e_continuum_gating", "Continuum handling + outlier gating"),
    ("3a", "stage3a_gzh_labels",      "stage3a_gzh_labels",      "GZH label anchoring"),
    ("3b", "stage3b_star_routing",    "stage3b_star_routing",    "Star calibration routing"),
    ("3c", "stage3c_noise_model",     "stage3c_noise_model",     "Noise false-positive model"),
    ("5a", "stage5a_empirical_psf",   "stage5a_empirical_psf",   "Empirical PSF construction"),
    ("5b", "stage5b_pass2_refinement", "stage5b_pass2_refinement", "Pass-2 morphology refinement"),
    ("4a", "stage4a_dual_branch",     "stage4a_dual_branch",     "Dual-branch multi-task training"),
    ("4b", "stage4b_ensemble",        "stage4b_ensemble",        "Ensemble consensus"),
    ("4c", "stage4c_calibration",     "stage4c_calibration",     "Probability calibration"),
    ("4d", "stage4d_ablation",        "stage4d_ablation",        "Controlled ablation"),
    ("6a", "stage6a_test_eval",       "stage6a_test_eval",       "In-domain test (run once)"),
    ("6b", "stage6b_uncertainty",     "stage6b_uncertainty",     "Uncertainty audit"),
    ("6c", "stage6c_xai",             "stage6c_xai",             "Explainable AI (Grad-CAM, SHAP)"),
    ("7a", "stage7a_taxonomy",        "stage7a_taxonomy",        "Taxonomy alignment + ceiling"),
    ("7b", "stage7b_crosscat",        "stage7b_crosscat",        "Cross-catalogue evaluation"),
    ("7c", "stage7c_generalisation",  "stage7c_generalisation",  "Generalisation disentanglement"),
    ("8",  "stage8_catalogue",        "stage8_catalogue",        "Unified catalogue (catalog.csv)"),
]
STAGE_DIR = {k: d for k, _, d, _ in STAGES}


class MissingInput(RuntimeError):
    """Raised when a stage needs a file that an earlier stage has not written yet."""


class StageSkipped(RuntimeError):
    """Raised by a stage that cannot run with the data provided (e.g. no CANDELS table)."""


class Ctx:
    def __init__(self, root: str | Path, config_file: str = "config.yaml"):
        self.root = Path(root).resolve()
        with open(self.root / config_file) as f:
            self.cfg = yaml.safe_load(f)
        p = self.cfg["paths"]
        self.raw = self.root / p["raw"]
        self.catalogs = self.root / p["catalogs"]
        self.out = self.root / p["outputs"]
        self.out.mkdir(parents=True, exist_ok=True)
        self.log = _make_logger(self.out / "pipeline.log")
        self.seed = int(self.cfg["project"].get("seed", 42))
        self.set_seed()

    # ---------------- folders and files ----------------
    def stage_dir(self, key: str) -> Path:
        d = self.out / STAGE_DIR[key]
        d.mkdir(parents=True, exist_ok=True)
        return d

    def path(self, key: str, name: str) -> Path:
        return self.out / STAGE_DIR[key] / name

    def require(self, key: str, name: str) -> Path:
        p = self.path(key, name)
        if not p.exists():
            raise MissingInput(f"{self.rel(p)} not found - run stage {key} first "
                               f"(python run_pipeline.py --only {key}).")
        return p

    # ---------------- small helpers ----------------
    def rel(self, p) -> str:
        """Path relative to the project folder when possible (outputs may live elsewhere)."""
        try:
            return str(Path(p).resolve().relative_to(self.root))
        except ValueError:
            return str(p)

    def set_seed(self):
        random.seed(self.seed)
        np.random.seed(self.seed)
        try:
            import torch
            torch.manual_seed(self.seed)
        except Exception:
            pass

    @property
    def n_jobs(self) -> int:
        n = int(self.cfg["project"].get("n_jobs", -1))
        return (os.cpu_count() or 1) if n < 0 else max(1, n)

    def device(self):
        import torch
        want = str(self.cfg["project"].get("device", "auto")).lower()
        if want == "auto":
            if torch.cuda.is_available():
                return torch.device("cuda")
            if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
                return torch.device("mps")
            return torch.device("cpu")
        return torch.device(want)


# ---------------- JSON / table helpers ----------------
def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=_json_default)
    return path


def load_json(path: Path):
    with open(path) as f:
        return json.load(f)


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if np.isnan(o) else float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    return str(o)


# Which file holds which per-object table. load_objects() merges them on object_id.
OBJECT_TABLES = {
    "sources":     ("1c", "sources.csv"),
    "pass1":       ("1e", "features_pass1.csv"),
    "clusters":    ("2b", "clusters.csv"),
    "baseline":    ("2c", "baseline.csv"),
    "gaia":        ("2c", "gaia_match.csv"),
    "evidence":    ("2d", "evidence.csv"),
    "segregation": ("2e", "segregation.csv"),
    "labels":      ("3a", "labels.csv"),
    "routing":     ("3b", "star_routing.csv"),
    "noise":       ("3c", "noise_scores.csv"),
    "pass2":       ("5b", "features_pass2.csv"),
}


def load_objects(ctx: Ctx, parts) -> pd.DataFrame:
    """Merge per-object tables on object_id (left join on the source list).
    A later part OVERWRITES columns of the same name for the rows it holds -
    this is how Pass-2 features replace Pass-1 features for the galaxy track."""
    df = None
    for part in parts:
        key, name = OBJECT_TABLES[part]
        t = pd.read_csv(ctx.require(key, name))
        if df is None:
            df = t
            continue
        overlap = [c for c in t.columns if c in df.columns and c != "object_id"]
        df = df.merge(t, on="object_id", how="left", suffixes=("", "__new"), indicator="_has")
        rows = df["_has"] == "both"
        for c in overlap:
            df.loc[rows, c] = df.loc[rows, c + "__new"]
            df = df.drop(columns=c + "__new")
        df = df.drop(columns="_has")
    return df


def _make_logger(path: Path) -> logging.Logger:
    log = logging.getLogger("p2p")
    if log.handlers:
        return log
    log.setLevel(logging.INFO)
    fmt = logging.Formatter("%(asctime)s  %(message)s", "%H:%M:%S")
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    fh = logging.FileHandler(path)
    fh.setFormatter(fmt)
    log.addHandler(sh)
    log.addHandler(fh)
    return log


class Timer:
    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *a):
        self.seconds = time.time() - self.t0

In [ ]:
%%writefile src/p2p/detection.py
"""Background modelling and source detection shared by Stage 1c and Stage 3c."""
from __future__ import annotations

import numpy as np
from photutils.background import Background2D, MedianBackground
from scipy import ndimage

from .compat import deblend, detect
from .utils import gaussian_psf


def subtract_background(sci, valid, box=64, filt=3):
    """2-D background (median estimator) fitted on covered pixels only, so chip
    gaps and frame edges (WHT = 0) do not drag the estimate. Returns the
    background-subtracted image (0 where invalid) and the background rms map."""
    data = np.where(valid, sci, 0.0).astype(np.float32)
    bkg = Background2D(data, box, filter_size=(filt, filt), coverage_mask=~valid,
                       bkg_estimator=MedianBackground(), fill_value=0.0)
    sub = np.where(valid, data - bkg.background, 0.0).astype(np.float32)
    rms = np.where(valid, bkg.background_rms, np.nan).astype(np.float32)
    rms = np.where(np.isfinite(rms) & (rms > 0), rms, np.nanmedian(rms))
    return sub, rms


def smooth(sub, fwhm_px=3.0):
    k = gaussian_psf(fwhm_px, size=int(2 * round(1.5 * fwhm_px) + 1))
    return ndimage.convolve(sub, k, mode="constant", cval=0.0)


def detect_and_deblend(sub, rms, valid, cfg, deblend_sources=True):
    """Threshold = nsigma x rms x correlated-noise factor on the smoothed image,
    then multi-threshold deblending (SExtractor-style)."""
    conv = smooth(sub, cfg["kernel_fwhm_px"])
    thr = cfg["nsigma"] * rms * cfg.get("correlated_noise_factor", 1.0)
    segm = detect(conv, thr, cfg["npixels"], mask=~valid)
    if segm is not None and deblend_sources:
        segm = deblend(conv, segm, cfg["npixels"], cfg["deblend_nlevels"], cfg["deblend_contrast"])
    return segm, conv

In [ ]:
%%writefile src/p2p/features.py
"""Tabular pre-processing (fit on TRAIN only) and model-ready tensors."""
from __future__ import annotations

import joblib
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

from .utils import asinh_stack, signed_log1p


class TabularPrep:
    """Median imputation -> signed log -> RobustScaler. Columns that are entirely
    NaN in the training data (e.g. colours when only one band exists) are dropped."""

    def __init__(self, columns):
        self.columns = list(columns)
        self.kept = None
        self.imputer = SimpleImputer(strategy="median")
        self.scaler = RobustScaler()

    def fit(self, df: pd.DataFrame):
        cols = [c for c in self.columns if c in df.columns]
        X = df[cols].apply(pd.to_numeric, errors="coerce").to_numpy(float)
        keep = ~np.all(np.isnan(X), axis=0)
        self.kept = [c for c, k in zip(cols, keep) if k]
        X = signed_log1p(self.imputer.fit_transform(df[self.kept].apply(pd.to_numeric, errors="coerce")))
        self.scaler.fit(X)
        return self

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        X = df.reindex(columns=self.kept).apply(pd.to_numeric, errors="coerce")
        return self.scaler.transform(signed_log1p(self.imputer.transform(X))).astype(np.float32)

    def save(self, path):
        joblib.dump(self, path)

    @staticmethod
    def load(path) -> "TabularPrep":
        return joblib.load(path)


def load_cutouts(ctx):
    """PSF-matched cutouts written by Stage 1d (memory-mapped, row = cutout_index)."""
    return np.load(ctx.require("1d", "cutouts.npy"), mmap_mode="r")


def image_tensor(cutouts, index):
    import torch
    batch = np.asarray(cutouts[np.asarray(index, int)], dtype=np.float32)
    return torch.tensor(asinh_stack(batch))


def make_tensors(df: pd.DataFrame, cutouts, prep: TabularPrep, with_labels=True):
    import torch
    d = {"img": image_tensor(cutouts, df["cutout_index"].to_numpy()),
         "tab": torch.tensor(prep.transform(df)),
         "object_id": df["object_id"].to_numpy()}
    if with_labels and "gz_label_id" in df and df["gz_label_id"].notna().all():
        d["y"] = torch.tensor(df["gz_label_id"].astype(int).to_numpy())
        d["ym"] = torch.tensor(df["gz_merger_flag"].fillna(0).astype(float).to_numpy(), dtype=torch.float32)
    return d


def galaxy_table(ctx):
    """Every galaxy-track object with Pass-2 features (overwriting Pass 1) and labels."""
    from .context import load_objects
    from .utils import as_bool
    df = load_objects(ctx, ["sources", "pass1", "segregation", "labels", "pass2"])
    df = df[(df.cluster_semantic == "Galaxy") & (df.anomaly_flag == 0)].copy()
    df["is_pseudo_label"] = as_bool(df.is_pseudo_label.fillna(True))
    df["edge_flag"] = as_bool(df.edge_flag)
    return df


def supervised_splits(ctx):
    """GZH-labelled galaxies per split. Truncated (edge_flag) sources are excluded,
    and QC-flagged ones too when frame_qc.exclude_flagged is true."""
    df = galaxy_table(ctx)
    lab = df[~df.is_pseudo_label & df.split_tag.isin(["train", "dev", "test"]) & ~df.edge_flag]
    if ctx.cfg["frame_qc"].get("exclude_flagged"):
        lab = lab[lab.frame_quality_flag == 1]
    lab = lab.assign(gz_label_id=lab.gz_label_id.astype(int))
    splits = {s: lab[lab.split_tag == s].reset_index(drop=True) for s in ("train", "dev", "test")}
    empty = [s for s, v in splits.items() if len(v) == 0]
    if empty:
        m = df[~df.is_pseudo_label]
        per = m.split_tag.fillna("collar").value_counts().to_dict()
        per_ok = m[~m.edge_flag].split_tag.fillna("collar").value_counts().to_dict()
        raise RuntimeError(
            f"No labelled galaxies in split(s) {empty}.\n"
            f"  GZH-matched galaxies per split: {per}\n"
            f"  ... after removing edge-truncated ones: {per_ok}\n"
            "  With one pointing, Stage 0 cuts it into stripes: look at outputs/stage0_partition/split_regions.png, "
            "lower partition.collar_arcmin, or add a second pointing.")
    return splits

In [ ]:
%%writefile src/p2p/fitsio.py
"""Reading HST drizzled products (_drc / _drz) and grouping them into pointings."""
from __future__ import annotations

import fnmatch
import warnings
from pathlib import Path

import numpy as np
from astropy.io import fits
from astropy.time import Time
from astropy.wcs import WCS, FITSFixedWarning
from astropy.wcs.utils import proj_plane_pixel_scales

warnings.simplefilter("ignore", FITSFixedWarning)


def find_products(raw_dir: Path, patterns) -> list[Path]:
    files = []
    for p in sorted(raw_dir.rglob("*")):
        if p.is_file() and any(fnmatch.fnmatch(p.name.lower(), pat.lower()) for pat in patterns):
            files.append(p)
    # prefer _drc over _drz for the same rootname
    by_root = {}
    for f in files:
        root = f.name.lower().replace("_drc", "").replace("_drz", "")
        if root not in by_root or "_drc" in f.name.lower():
            by_root[root] = f
    return sorted(by_root.values())


def filter_name(prim, sci) -> str:
    for hdr in (prim, sci):
        for k in ("FILTER", "FILTER1", "FILTER2"):
            v = str(hdr.get(k, "")).strip().upper()
            if v and not v.startswith("CLEAR") and v not in ("N/A", "NONE"):
                return v
    return "UNKNOWN"


def instrument_key(prim) -> str:
    inst = str(prim.get("INSTRUME", "")).upper()
    det = str(prim.get("DETECTOR", "")).upper()
    if inst == "WFC3":
        return f"WFC3_{det}" if det else "WFC3"
    return inst or "UNKNOWN"


def read_meta(path: Path) -> dict:
    """Header-only read: everything later stages need to know about a file."""
    with fits.open(path, memmap=True) as hdul:
        names = [h.name for h in hdul]
        prim = hdul[0].header
        sci_ext = "SCI" if "SCI" in names else (1 if len(hdul) > 1 else 0)
        sci = hdul[sci_ext].header
        w = WCS(sci).celestial
        nx, ny = int(sci.get("NAXIS1", 0)), int(sci.get("NAXIS2", 0))
    ok_wcs = w.has_celestial and str(sci.get("CTYPE1", "")).startswith("RA")
    scale = float(proj_plane_pixel_scales(w)[0] * 3600) if ok_wcs else np.nan
    fp = w.calc_footprint(axes=(nx, ny)) if ok_wcs else np.zeros((4, 2))
    centre = w.pixel_to_world_values((nx - 1) / 2, (ny - 1) / 2) if ok_wcs else (np.nan, np.nan)
    photflam = sci.get("PHOTFLAM", prim.get("PHOTFLAM"))
    photplam = sci.get("PHOTPLAM", prim.get("PHOTPLAM"))
    zp = float(-2.5 * np.log10(photflam) - 5 * np.log10(photplam) - 2.408) if photflam and photplam else np.nan
    date = prim.get("DATE-OBS") or sci.get("DATE-OBS")
    try:
        epoch = float(Time(date).decimalyear) if date else np.nan
    except Exception:
        epoch = np.nan
    return {
        "file": str(path), "name": path.name,
        "product_type": "drc" if "_drc" in path.name.lower() else ("drz" if "_drz" in path.name.lower() else "other"),
        "has_sci": "SCI" in names, "has_wht": "WHT" in names, "has_ctx": "CTX" in names,
        "wcs_ok": bool(ok_wcs), "naxis1": nx, "naxis2": ny,
        "instrument": instrument_key(prim), "telescope": str(prim.get("TELESCOP", "")),
        "filter": filter_name(prim, sci), "targname": str(prim.get("TARGNAME", "")),
        "proposid": prim.get("PROPOSID"), "exptime": float(prim.get("EXPTIME", sci.get("EXPTIME", np.nan))),
        "ndrizim": prim.get("NDRIZIM", sci.get("NDRIZIM")), "date_obs": str(date), "epoch": epoch,
        "bunit": str(sci.get("BUNIT", "")), "photflam": photflam, "photplam": photplam, "zp_ab": zp,
        "pixscale_arcsec": scale, "ra_centre": float(centre[0]), "dec_centre": float(centre[1]),
        "footprint": np.asarray(fp).tolist(),
    }


def load_image(path: str | Path):
    """SCI (float32), WHT (float32 or None) and the celestial WCS."""
    with fits.open(path, memmap=False) as hdul:
        names = [h.name for h in hdul]
        sci_ext = "SCI" if "SCI" in names else 1
        sci = np.asarray(hdul[sci_ext].data, dtype=np.float32)
        wht = np.asarray(hdul["WHT"].data, dtype=np.float32) if "WHT" in names else None
        w = WCS(hdul[sci_ext].header).celestial
    return sci, wht, w


def load_ctx(path: str | Path):
    """Drizzle context map (CTX): bit k set = input image k contributes to the pixel. None if absent
    or 3-D (more than 32 inputs)."""
    with fits.open(path, memmap=False) as hdul:
        names = [h.name for h in hdul]
        if "CTX" not in names or hdul["CTX"].data is None or hdul["CTX"].data.ndim != 2:
            return None
        return np.asarray(hdul["CTX"].data).astype(np.int64)


def valid_mask(sci, wht):
    m = np.isfinite(sci)
    if wht is not None:
        m &= wht > 0
    return m


def is_external(meta: dict, patterns) -> bool:
    text = (meta["name"] + " " + meta.get("targname", "")).lower()
    return any(p.lower() in text for p in patterns or [])


def _overlap_fraction(a: dict, b: dict) -> float:
    """Overlap of two footprints' RA/Dec boxes as a fraction of the SMALLER box (cos(dec)-corrected)."""
    pa, pb = np.asarray(a["footprint"]), np.asarray(b["footprint"])
    cosd = np.cos(np.deg2rad(0.5 * (pa[:, 1].mean() + pb[:, 1].mean())))
    w = min(pa[:, 0].max(), pb[:, 0].max()) - max(pa[:, 0].min(), pb[:, 0].min())
    h = min(pa[:, 1].max(), pb[:, 1].max()) - max(pa[:, 1].min(), pb[:, 1].min())
    if w <= 0 or h <= 0:
        return 0.0
    area = lambda p: np.ptp(p[:, 0]) * cosd * np.ptp(p[:, 1])
    return float(w * cosd * h / max(min(area(pa), area(pb)), 1e-12))


def group_pointings(metas: list[dict], radius_arcsec: float, min_overlap: float = 0.5) -> list[list[int]]:
    """Union-find: files whose centres are closer than radius_arcsec, OR whose footprints overlap by at
    least min_overlap of the smaller one, form one pointing. The overlap rule matters when the bands of
    one field come from different HST programmes, with centres far apart but the same sky."""
    n = len(metas)
    parent = list(range(n))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    ra = np.array([m["ra_centre"] for m in metas])
    dec = np.array([m["dec_centre"] for m in metas])
    for i in range(n):
        for j in range(i + 1, n):
            d = np.hypot((ra[i] - ra[j]) * np.cos(np.deg2rad(dec[i])), dec[i] - dec[j]) * 3600
            if d < radius_arcsec or _overlap_fraction(metas[i], metas[j]) >= min_overlap:
                parent[find(i)] = find(j)
    groups = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)
    # stable order: by RA of the group centre
    out = list(groups.values())
    out.sort(key=lambda g: (np.mean(ra[g]), np.mean(dec[g])))
    return out


def bbox_of(metas_in_group: list[dict]):
    pts = np.vstack([np.asarray(m["footprint"]) for m in metas_in_group])
    return float(pts[:, 0].min()), float(pts[:, 0].max()), float(pts[:, 1].min()), float(pts[:, 1].max())

In [ ]:
%%writefile src/p2p/inference.py
"""Loading the trained models and the frozen, calibrated inference path
(used by Stages 4b-4d, 6, 7 and 8)."""
from __future__ import annotations

import joblib
import numpy as np
import torch

from .context import Ctx, load_json
from .features import TabularPrep
from .nets import N_CLASSES, DualBranchNet, EfficientNetMember, logit, predict_model, softmax_np


class TabularMember:
    """XGBoost (or the sklearn fallback) on the Pass-2 table. XGBoost always runs in a separate
    process (p2p.xgb_worker): next to PyTorch on macOS it can crash Python with a segmentation
    fault. If that process fails, sklearn's HistGradientBoosting is used instead. Classes missing
    from the training data are handled by mapping back to all 5 columns."""

    def __init__(self, seed=0, log=print):
        self.seed = seed
        self.classes_ = None
        self.model = None          # sklearn model (fallback) - XGBoost lives in self.raw instead
        self.raw, self.best = None, None
        self.kind = None
        self._log = log

    def fit(self, X, y, Xdev=None, ydev=None):
        from . import xgb_worker
        self.classes_ = np.unique(y)
        remap = {c: i for i, c in enumerate(self.classes_)}
        yy = np.array([remap[c] for c in y])
        counts = np.bincount(yy)
        w = (len(yy) / (len(counts) * np.maximum(counts, 1)))[yy]
        if len(self.classes_) == 1:                        # nothing to learn: always that class
            self.kind = "constant"
            return self
        Xe = ye = None
        if Xdev is not None:
            keep = np.isin(ydev, self.classes_)
            if keep.any():
                Xe, ye = np.asarray(Xdev)[keep], np.array([remap[c] for c in np.asarray(ydev)[keep]])
        try:
            self.raw, self.best = xgb_worker.run_isolated(xgb_worker.fit, np.asarray(X, np.float32), yy, w, Xe, ye,
                                                          self.seed)
            self.kind = "xgboost"
        except Exception as e:                              # ImportError, or the child process crashed
            self._log(f"[TabularMember] XGBoost unavailable ({type(e).__name__}: {e}); using sklearn "
                      "HistGradientBoosting instead")
            from sklearn.ensemble import HistGradientBoostingClassifier
            self.model = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300,
                                                        class_weight="balanced", random_state=self.seed).fit(X, yy)
            self.kind = "sklearn-hgb"
        return self

    def __getstate__(self):
        d = dict(self.__dict__)
        d.pop("_log", None)                                 # the logger is not saved with the model
        return d

    def __setstate__(self, d):
        self.__dict__.update(d)
        self._log = print

    def predict_proba(self, X):
        from . import xgb_worker
        X = np.asarray(X, np.float32)
        out = np.zeros((len(X), N_CLASSES))
        if self.kind == "constant":
            out[:, self.classes_[0]] = 1.0
            return out
        p = xgb_worker.run_isolated(xgb_worker.predict, self.raw, self.best, X) if self.kind == "xgboost" \
            else self.model.predict_proba(X)
        out[:, self.classes_] = p
        return out / out.sum(1, keepdims=True)

    def contribs(self, X):
        """TreeSHAP values (n, classes, features) - XGBoost only."""
        from . import xgb_worker
        return xgb_worker.run_isolated(xgb_worker.contribs, self.raw, self.best, np.asarray(X, np.float32))


def load_models(ctx: Ctx):
    meta = load_json(ctx.require("4a", "model_meta.json"))
    prep = TabularPrep.load(ctx.require("4a", "tab_prep.joblib"))
    dual = DualBranchNet(meta["n_tab"], in_ch=meta["in_ch"], mode="hybrid")
    dual.load_state_dict(torch.load(ctx.require("4a", "dual_branch.pt"), map_location="cpu"))
    m = {"meta": meta, "prep": prep, "dual": dual}
    if ctx.path("4b", "efficientnet.pt").exists():
        eff = EfficientNetMember(in_ch=meta["in_ch"])
        eff.load_state_dict(torch.load(ctx.path("4b", "efficientnet.pt"), map_location="cpu"))
        m["effnet"] = eff
        m["tabular"] = joblib.load(ctx.require("4b", "tabular_member.joblib"))
    return m


def ensemble_predict(ctx: Ctx, data, models):
    dev = ctx.device()
    d = predict_model(models["dual"], data, dev)
    out = {"members": {"dual": d["morph"]}, "merger": d["merger"]}
    if "effnet" in models:
        out["members"]["effnet"] = predict_model(models["effnet"], data, dev)["morph"]
        out["members"]["tabular"] = models["tabular"].predict_proba(data["tab"].numpy())
        w = np.asarray(ctx.cfg["supervised"]["ensemble_weights"], float)
        keys = ["dual", "effnet", "tabular"]
        P = sum(wi * out["members"][k] for wi, k in zip(w / w.sum(), keys))
    else:
        P = d["morph"]
    out["morph"] = P / P.sum(1, keepdims=True)
    return out


def calibrated_predict(ctx: Ctx, data, models):
    """The FROZEN inference path: soft vote -> temperature scaling (Stage 4c)."""
    out = ensemble_predict(ctx, data, models)
    p = ctx.path("4c", "temperature.json")
    if p.exists():
        T = load_json(p)
        out["morph"] = softmax_np(np.log(np.clip(out["morph"], 1e-12, 1)) / T["T_morph"])
        z = T.get("merger_platt_a", 1.0) * logit(out["merger"]) + T.get("merger_platt_b", 0.0)
        out["merger"] = 1 / (1 + np.exp(-z))
    return out

In [ ]:
%%writefile src/p2p/morphology.py
"""Morphology measurement shared by Stage 1e (Pass 1) and Stage 5b (Pass 2).

statmorph (Rodriguez-Gomez et al. 2019) measures, for one source:
  non-parametric : Gini, M20, concentration C, asymmetry A, smoothness S,
                   multimode / intensity / deviation (MID), shape asymmetry
  parametric     : single Sersic fit (n, r_eff, ellipticity), convolved with
                   the PSF passed in -> this is where Pass 1 and Pass 2 differ.
If statmorph is missing or fails, simple NumPy estimators are used (flag = 4).
Masked pixels: other sources (segmentation stamp value 2) and cosmic-ray pixels
(value 3, Stage 1c). statmorph's -99 failure codes are stored as NaN, and a Sersic
fit that ends at a parameter limit (morphology.sersic_n_bounds,
sersic_rhalf_max_px) gets flag_sersic = 1.
"""
from __future__ import annotations

import warnings

import numpy as np
from scipy.signal import fftconvolve

try:
    import statmorph
    _HAVE_STATMORPH = True
except Exception:
    _HAVE_STATMORPH = False

FEATURES = ["gini", "m20", "concentration", "asymmetry", "smoothness", "sersic_n", "sersic_rhalf",
            "sersic_ellip", "r_half", "r20", "r80", "r_petro", "ellipticity_sm", "shape_asymmetry",
            "multimode", "intensity", "deviation", "gini_m20_merger", "sn_per_pixel", "flag", "flag_sersic"]

_STATMORPH_MAP = {"gini": "gini", "m20": "m20", "concentration": "concentration", "asymmetry": "asymmetry",
                  "smoothness": "smoothness", "sersic_n": "sersic_n", "sersic_rhalf": "sersic_rhalf",
                  "sersic_ellip": "sersic_ellip", "r_half": "rhalf_circ", "r20": "r20", "r80": "r80",
                  "r_petro": "rpetro_circ", "ellipticity_sm": "ellipticity_asymmetry",
                  "shape_asymmetry": "shape_asymmetry", "multimode": "multimode", "intensity": "intensity",
                  "deviation": "deviation", "gini_m20_merger": "gini_m20_merger", "sn_per_pixel": "sn_per_pixel",
                  "flag": "flag", "flag_sersic": "flag_sersic"}


def morph_image(cube):
    """Sum of the PSF-matched bands -> one image with the best signal-to-noise."""
    return np.asarray(cube, np.float32).sum(0)


def richardson_lucy(img, psf, iters=10):
    obs = np.clip(img, 0, None) + 1e-6
    est = np.full_like(obs, obs.mean())
    psf_m = psf[::-1, ::-1]
    for _ in range(iters):
        est *= fftconvolve(obs / (fftconvolve(est, psf, mode="same") + 1e-12), psf_m, mode="same")
    return est


def _numpy_fallback(img, target):
    pix = np.sort(np.abs(img[target]))
    n, tot = pix.size, pix.sum()
    gini = float(np.sum((2 * np.arange(1, n + 1) - n - 1) * pix) / (tot * (n - 1))) if n > 1 and tot > 0 else np.nan
    h, w = img.shape
    yy, xx = np.mgrid[:h, :w]
    r = np.hypot(yy - (h - 1) / 2, xx - (w - 1) / 2).ravel()
    o = np.argsort(r)
    cog = np.cumsum(np.clip(np.where(target, img, 0).ravel()[o], 0, None))
    out = {k: np.nan for k in FEATURES}
    if cog[-1] > 0:
        cog = cog / cog[-1]
        r20, r50, r80 = (r[o][np.searchsorted(cog, q)] for q in (0.2, 0.5, 0.8))
        out.update(r20=r20, r_half=r50, r80=r80, concentration=5 * np.log10(r80 / r20) if r20 > 0 else np.nan)
    rot = np.where(target, img, 0)[::-1, ::-1]
    den = np.abs(np.where(target, img, 0)).sum()
    out.update(gini=gini, asymmetry=float(np.abs(np.where(target, img, 0) - rot).sum() / (2 * den)) if den > 0 else np.nan,
               flag=4, flag_sersic=1)
    return out


def clean_result(res, n_bounds=(0.05, 8.0), rhalf_max=64.0):
    """-99 (statmorph's 'could not measure') -> NaN; a Sersic fit stuck at a limit -> flag_sersic = 1."""
    out = {k: (np.nan if (k not in ("flag", "flag_sersic") and v is not None and np.isfinite(v) and v <= -98.9)
               else v) for k, v in res.items()}
    n, rh = out.get("sersic_n", np.nan), out.get("sersic_rhalf", np.nan)
    stuck = (np.isfinite(n) and (n <= n_bounds[0] or n >= n_bounds[1])) or (np.isfinite(rh) and rh > rhalf_max)
    if stuck or not np.isfinite(n):
        out["flag_sersic"] = max(1.0, float(np.nan_to_num(out.get("flag_sersic", 1.0), nan=1.0)))
    return out


def measure_one(cube, seg, gain, psf=None, mode="psf_fit", rl_iters=10, sersic_maxiter=500,
                n_bounds=(0.05, 8.0), rhalf_max=64.0):
    return clean_result(_measure_one(cube, seg, gain, psf, mode, rl_iters, sersic_maxiter), n_bounds, rhalf_max)


def _measure_one(cube, seg, gain, psf=None, mode="psf_fit", rl_iters=10, sersic_maxiter=500):
    img = morph_image(cube)
    target = seg == 1                                   # cosmic-ray pixels (3) are NOT part of the target
    if target.sum() < 5:
        return {**{k: np.nan for k in FEATURES}, "flag": 4, "flag_sersic": 1}
    mask = (seg == 2) | (seg == 3) | ((img == 0) & ~target)
    use_psf = psf
    if mode == "deconvolve" and psf is not None:
        dil = np.zeros_like(target)
        from scipy.ndimage import binary_dilation
        dil = binary_dilation(target, iterations=3)
        img = np.where(dil, richardson_lucy(np.where(dil, img, 0), psf, rl_iters), img)
        use_psf = None
    if not _HAVE_STATMORPH:
        return _numpy_fallback(img, target)
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            m = statmorph.source_morphology(img, target.astype(int), mask=mask, gain=gain, psf=use_psf,
                                            sersic_maxiter=sersic_maxiter)[0]
        return {k: float(getattr(m, v)) if getattr(m, v, None) is not None else np.nan
                for k, v in _STATMORPH_MAP.items()}
    except Exception:
        return _numpy_fallback(img, target)


def measure_batch(cut_path, seg_path, indices, gains, psfs, mode, rl_iters, sersic_maxiter,
                  n_bounds=(0.05, 8.0), rhalf_max=64.0):
    """Worker for joblib: opens the memory-mapped arrays itself (nothing big is pickled)."""
    try:
        from astropy import log as _alog
        _alog.setLevel("ERROR")                    # statmorph reports through astropy's logger
    except Exception:
        pass
    warnings.simplefilter("ignore")
    cut = np.load(cut_path, mmap_mode="r")
    seg = np.load(seg_path, mmap_mode="r")
    out = []
    for i, g, p in zip(indices, gains, psfs):
        out.append(measure_one(cut[i].astype(np.float32), np.asarray(seg[i]), g, p, mode, rl_iters, sersic_maxiter,
                               n_bounds, rhalf_max))
    return out


def run_parallel(ctx, cut_path, seg_path, indices, gains, psfs, mode="psf_fit"):
    from joblib import Parallel, delayed
    cfg = ctx.cfg["morphology"]
    n_jobs = ctx.n_jobs
    chunks = np.array_split(np.arange(len(indices)), max(1, min(len(indices), n_jobs * 4)))
    jobs = [delayed(measure_batch)(str(cut_path), str(seg_path), [indices[j] for j in ch], [gains[j] for j in ch],
                                   [psfs[j] for j in ch], mode, cfg["rl_iterations"], cfg["sersic_maxiter"],
                                   tuple(cfg.get("sersic_n_bounds", (0.05, 8.0))),
                                   float(cfg.get("sersic_rhalf_max_px", 64.0)))
            for ch in chunks if len(ch)]
    res = Parallel(n_jobs=n_jobs, verbose=5)(jobs)
    return [r for part in res for r in part]

In [ ]:
%%writefile src/p2p/multiband.py
"""Multi-band checks shared by Stage 1c (and read by later stages through sources.csv).

Three problems of single-exposure HST products are handled here:

1. Band registration. Bands taken by different programmes can carry different
   astrometric solutions (e.g. an 'a priori' pointing-only WCS against one refined
   on a star catalogue). `registration_catalog` detects sources on a MEDIAN-FILTERED
   image: a single exposure holds tens of thousands of cosmic rays, which are
   brighter than most galaxies and would fill any 'brightest sources' list, but
   they are only 1-3 px wide and a 5x5 median erases them while galaxies survive.
   `measure_offset` finds the shift from the pairs of the two lists and reports how
   far the peak stands above the random-pair level; `shift_wcs` removes it.
2. Cosmic rays (source level). Cosmic rays hit each exposure at random places,
   so a real source is also seen in a second band of the same sky, a cosmic ray
   almost never. `segment_photometry` measures every detection segment in every
   band (S/N from the background rms) and the centroid of the band's significant
   pixels on a 3x3 MEDIAN-FILTERED copy (cosmic rays erased, compact sources kept).
   A band CONFIRMS a source when its S/N >= detection.confirm_snr AND its light is
   centred where the detection-band light is (within detection.confirm_max_shift_px):
   in a single exposure a cosmic ray of the other band often lands inside a large
   footprint (high S/N at the wrong place), and cosmic rays swallowed by a galaxy's
   detection footprint would pull a raw centroid away.
3. Cosmic rays (pixel level). Inside a cutout, a pixel that is far brighter in
   one band than the other bands predict (given the source's own colour) is a
   cosmic-ray hit in that band. `cr_pixel_mask` finds them; Stage 1c replaces
   them by the prediction and marks them 3 in the segmentation stamp, so
   statmorph masks them.
"""
from __future__ import annotations

import numpy as np
from scipy import ndimage
from scipy.spatial import cKDTree


# ---------------------------------------------------------------- 1. registration
def tangent_arcsec(ra, dec, ra0, dec0):
    """Small-field tangent-plane offsets (arcsec) from (ra0, dec0): x = dRA cos(dec), y = dDec."""
    ra, dec = np.asarray(ra, float), np.asarray(dec, float)
    return (ra - ra0) * np.cos(np.deg2rad(dec0)) * 3600.0, (dec - dec0) * 3600.0


def registration_catalog(sub, rms, valid, wcs, cfg, median_px=5, nsigma=5.0, npixels=25, n_max=3000, max_ellip=0.8):
    """RA/Dec of the brightest extended sources of a median-filtered image (cosmic rays removed)."""
    from .compat import SourceCatalog, n_labels, values
    from .detection import detect_and_deblend
    med = ndimage.median_filter(np.where(valid, sub, 0.0).astype(np.float32), size=median_px)
    sky = med[valid]
    if sky.size > 2_000_000:
        sky = sky[:: sky.size // 2_000_000]
    mad = 1.4826 * np.median(np.abs(sky - np.median(sky)))           # noise of the median image
    rms_m = rms * (mad / max(float(np.nanmedian(rms)), 1e-12))
    segm, conv = detect_and_deblend(med, rms_m, valid, {**cfg, "nsigma": nsigma, "npixels": npixels},
                                    deblend_sources=False)
    if n_labels(segm) == 0:
        return np.zeros(0), np.zeros(0)
    cat = SourceCatalog(med, segm, convolved_data=conv)
    x, y = values(cat, "x_centroid"), values(cat, "y_centroid")
    flux, ell = values(cat, "segment_flux"), values(cat, "ellipticity")
    ok = np.where(np.isfinite(x) & np.isfinite(flux) & (ell < max_ellip))[0]
    ok = ok[np.argsort(-flux[ok])][:n_max]
    return wcs.pixel_to_world_values(x[ok], y[ok])


def registration_sources(x, y, snr, semimajor, ellipticity, min_snr=20.0, min_sigma_px=1.2, max_ellip=0.7, n_max=3000):
    """Indices of sources good for registration: bright and EXTENDED (sharp cosmic rays and long
    thin tracks would match random partners in a crowded single exposure)."""
    ok = (np.isfinite(x) & np.isfinite(y) & (np.asarray(snr) >= min_snr) & (np.asarray(semimajor) >= min_sigma_px)
          & (np.asarray(ellipticity) < max_ellip))
    idx = np.where(ok)[0]
    return idx[np.argsort(-np.asarray(snr)[idx])][:n_max]


def measure_offset(ref_xy, band_xy, search_arcsec=3.0, bin_arcsec=0.05, refine_arcsec=0.15):
    """Median offset (band - reference, arcsec) from all pairs within `search_arcsec`.
    The peak of the 2-D histogram of pair offsets is robust to the random pairs of a crowded
    field; the median of the pairs within `refine_arcsec` of the peak refines it.
    Returns (dx, dy, n_pairs, scatter_arcsec, significance), significance = n_pairs / the number of
    random pairs expected in the same refine circle (a real offset gives >> 5)."""
    if len(ref_xy) == 0 or len(band_xy) == 0:
        return np.nan, np.nan, 0, np.nan, 0.0
    tree = cKDTree(band_xy)
    pairs = tree.query_ball_point(ref_xy, search_arcsec)
    d = np.array([band_xy[j] - ref_xy[i] for i, js in enumerate(pairs) for j in js], float).reshape(-1, 2)
    if len(d) < 5:
        return np.nan, np.nan, 0, np.nan, 0.0
    edges = np.arange(-search_arcsec, search_arcsec + bin_arcsec, bin_arcsec)
    H, ex, ey = np.histogram2d(d[:, 0], d[:, 1], bins=[edges, edges])
    H = ndimage.uniform_filter(H, 3)                     # 3x3 smoothing: the peak is a few bins wide
    i, j = np.unravel_index(np.argmax(H), H.shape)
    peak = np.array([0.5 * (ex[i] + ex[i + 1]), 0.5 * (ey[j] + ey[j + 1])])
    for _ in range(3):                                   # iterate the refinement around the median
        near = np.hypot(*(d - peak).T) <= refine_arcsec
        if near.sum() < 3:
            break
        peak = np.median(d[near], axis=0)
    near = np.hypot(*(d - peak).T) <= refine_arcsec
    scatter = float(1.4826 * np.median(np.hypot(*(d[near] - peak).T))) if near.any() else np.nan
    expected = len(d) * refine_arcsec ** 2 / search_arcsec ** 2          # random pairs in the refine circle
    return float(peak[0]), float(peak[1]), int(near.sum()), scatter, float(near.sum() / max(expected, 1e-9))


def shift_wcs(wcs, dx_arcsec, dy_arcsec):
    """Copy of `wcs` whose sky positions are moved by (-dx, -dy) arcsec: the band's positions were
    (dx, dy) too large (dx = dRA cos(dec)). Shifting CRVAL moves a small field rigidly."""
    w = wcs.deepcopy()
    dec0 = w.wcs.crval[1]
    w.wcs.crval = [w.wcs.crval[0] - dx_arcsec / 3600.0 / np.cos(np.deg2rad(dec0)), dec0 - dy_arcsec / 3600.0]
    w.wcs.set()
    return w


# ---------------------------------------------------------------- 2. source-level confirmation
def median_image(sub, valid, size=3):
    """Median-filtered copy and the factor by which its noise is lower than the input's (from the MAD)."""
    img = np.where(valid, sub, 0.0).astype(np.float32)
    med = ndimage.median_filter(img, size=size)
    a, m = img[valid], med[valid]
    step = max(1, a.size // 2_000_000)
    mad = lambda x: 1.4826 * np.median(np.abs(x - np.median(x)))
    return med, float(mad(m[::step]) / max(mad(a[::step]), 1e-12))


def segment_photometry(labels, det_wcs, band, det_pixarea, max_label, corr=1.0, sig_level=2.0):
    """Flux, error and coverage of every detection segment measured in another band.

    labels   : detection-band segmentation image (0 = sky)
    band     : dict with 'sub', 'rms', 'val', 'wcs', 'pixarea' of the band
    Each detection pixel samples the band at the same sky position (nearest pixel); the flux is
    scaled by the pixel-area ratio. The centroid (detection-band pixel units) is weighted by the band's
    MEDIAN-FILTERED pixels (band['med'], noise factor band['med_noise']) above sig_level sigma only, so
    cosmic rays inside the footprint cannot move it. Returns arrays indexed by label (0..max_label):
    flux, err, coverage, cx, cy (NaN where the band has no significant pixel)."""
    ys, xs = np.nonzero(labels)
    lab = labels[ys, xs]
    n_all = np.bincount(lab, minlength=max_label + 1).astype(float)
    if band is None:
        z = np.zeros(max_label + 1)
        nan = np.full(max_label + 1, np.nan)
        return z, nan, z, nan, nan
    ra, dec = det_wcs.pixel_to_world_values(xs, ys)
    bx, by = band["wcs"].world_to_pixel_values(ra, dec)
    del ra, dec
    xi, yi = np.round(bx).astype(np.int64), np.round(by).astype(np.int64)
    h, w = band["sub"].shape
    ok = (xi >= 0) & (yi >= 0) & (xi < w) & (yi < h)
    ok[ok] = band["val"][yi[ok], xi[ok]]
    scale = det_pixarea / band["pixarea"]
    v = np.zeros(len(lab))
    var = np.zeros(len(lab))
    v[ok] = band["sub"][yi[ok], xi[ok]] * scale
    var[ok] = (band["rms"][yi[ok], xi[ok]] * scale * corr) ** 2
    flux = np.bincount(lab, weights=v, minlength=max_label + 1)
    err = np.sqrt(np.bincount(lab, weights=var, minlength=max_label + 1))
    cov = np.bincount(lab, weights=ok.astype(float), minlength=max_label + 1) / np.maximum(n_all, 1)
    vm = np.zeros(len(lab))
    vm[ok] = band["med"][yi[ok], xi[ok]] * scale
    wgt = np.where(vm > sig_level * np.sqrt(var) * band["med_noise"], vm, 0.0)
    wsum = np.bincount(lab, weights=wgt, minlength=max_label + 1)
    with np.errstate(invalid="ignore", divide="ignore"):
        cx = np.where(wsum > 0, np.bincount(lab, weights=wgt * xs, minlength=max_label + 1) / wsum, np.nan)
        cy = np.where(wsum > 0, np.bincount(lab, weights=wgt * ys, minlength=max_label + 1) / wsum, np.nan)
    return flux, np.where(err > 0, err, np.nan), cov, cx, cy


# ---------------------------------------------------------------- 3. pixel-level cosmic-ray mask
_RING = np.array([[1, 1, 1], [1, 0, 1], [1, 1, 1]], bool)


def cr_pixel_mask(cube, valid, seg, sigma, flux, snr, nsigma=5.0, ratio=4.0, min_ref_snr=3.0, sharp_ratio=2.0):
    """Cosmic-ray pixels of one cutout, per band, and the replacement values.

    cube  : (bands, S, S) raw stamps (e-/s per output pixel)
    valid : (bands, S, S) bool coverage
    seg   : (S, S) 0 sky / 1 target / 2 other source
    sigma : (bands, S, S) per-pixel noise (background + source Poisson)
    flux, snr : per-band segment flux and S/N of the TARGET (fallback colour for faint targets)
    A pixel of band b is a hit when it is > nsigma above the value that EVERY usable other band c
    predicts (v_c x R_bc) and more than `ratio` times that prediction. R_bc, the target's colour, is the
    MEDIAN pixel ratio v_b / v_c over the target pixels bright in c (> 5 sigma): segment fluxes are no good
    here, because cosmic-ray hits inside a segment inflate them and the whole galaxy would then look too
    bright in the other band. Band c is usable when the target is detected in it (snr >= min_ref_snr) and
    it covers the pixel. Pixels of other sources (seg == 2) are not tested: their colour is not the
    target's, and they are masked anyway.
    A hit must also be SHARP in its own band: more than nsigma above the median of its 8 neighbours and
    more than sharp_ratio times that median. Without this, the smooth red halo of ACS F814W around bright
    stars (scattered light the blue bands do not have) is 'too bright for its colour' and was masked.
    Returns (mask (bands, S, S) bool, replacement (bands, S, S))."""
    nb = cube.shape[0]
    mask = np.zeros(cube.shape, bool)
    repl = np.zeros(cube.shape, np.float32)
    test_area = seg != 2
    target = seg == 1
    for b in range(nb):
        preds, errs = [], []
        for c in range(nb):
            if c == b or not (np.isfinite(snr[c]) and snr[c] >= min_ref_snr and flux[c] > 0 and np.isfinite(flux[b])):
                continue
            bright = target & valid[b] & valid[c] & (cube[c] > 5 * sigma[c])
            if bright.sum() >= 5:
                r = float(np.median(cube[b][bright] / cube[c][bright]))
            else:
                r = max(flux[b], 0.0) / flux[c]
            r = max(r, 0.0)
            preds.append(np.where(valid[c], cube[c] * r, np.nan))
            errs.append(np.where(valid[c], sigma[c] * r, np.nan))
        if not preds:
            continue
        P, E = np.stack(preds), np.stack(errs)
        excess = cube[b][None] - P
        tot = np.sqrt(sigma[b][None] ** 2 + E ** 2)
        hit_c = (excess > nsigma * tot) & (cube[b][None] > ratio * np.clip(P, 0, None))
        usable = np.isfinite(P)
        hit = np.all(hit_c | ~usable, axis=0) & usable.any(axis=0)       # every usable band disagrees
        hit &= valid[b] & test_area & (cube[b] > nsigma * sigma[b])
        if hit.any():
            ring = ndimage.median_filter(cube[b], footprint=_RING, mode="nearest")
            hit &= (cube[b] - ring > nsigma * sigma[b]) & (cube[b] > sharp_ratio * np.clip(ring, 0, None))
        if hit.any():
            hit = ndimage.binary_dilation(hit, iterations=1) & valid[b] & test_area   # CR wings
            mask[b] = hit
            repl[b] = np.nanmedian(np.where(usable, P, np.nan), axis=0)
    return mask, np.nan_to_num(repl)


# ---------------------------------------------------------------- exposures per pixel
def popcount(ctx_values):
    """Number of set bits of the drizzle context (CTX) = number of input images at that pixel.
    For ACS/WFC the two chips of ONE exposure never overlap, so this counts exposures."""
    v = np.asarray(ctx_values).astype(np.uint64)
    n = np.zeros(v.shape, np.int64)
    while np.any(v):
        n += (v & np.uint64(1)).astype(np.int64)
        v = v >> np.uint64(1)
    return n

In [ ]:
%%writefile src/p2p/nets.py
"""Neural networks and training loops shared by Stages 1b, 2a and 4a-4d."""
from __future__ import annotations

import copy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from sklearn.metrics import average_precision_score, balanced_accuracy_score, f1_score

MORPH_CLASSES = ("Smooth", "FaceOnSpiral", "EdgeOn", "Irregular", "Ambiguous")
N_CLASSES = len(MORPH_CLASSES)


# ================================================================ autoencoders
class ConvAutoencoder(nn.Module):
    """Under-complete convolutional autoencoder, 128x128 input.
    Conv C->16->32->64 (128 -> 16 px), Linear bottleneck to latent_dim, mirror decoder.
    Stage 1b uses C=1 (one tile of one band); Stage 2a uses C = number of bands."""

    def __init__(self, in_ch=1, latent_dim=64, size=128):
        super().__init__()
        s = size // 8
        self.s = s
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(in_ch, 16, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU())
        self.enc_fc = nn.Linear(64 * s * s, latent_dim)
        self.dec_fc = nn.Linear(latent_dim, 64 * s * s)
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, 2, 1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, 2, 1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(16, in_ch, 3, 2, 1, output_padding=1), nn.Sigmoid())

    def encode(self, x):
        return self.enc_fc(self.encoder_conv(x).flatten(1))

    def decode(self, z):
        return self.decoder_conv(self.dec_fc(z).view(-1, 64, self.s, self.s))

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z


def train_autoencoder(model, X, epochs, batch_size, device, lr=1e-3, weight_decay=1e-5, log=print, tag="CAE"):
    """X: float32 array (N, C, H, W) already scaled to [0, 1]."""
    model.to(device).train()
    data = torch.tensor(X)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    for ep in range(epochs):
        perm = torch.randperm(len(data))
        run = 0.0
        for i in range(0, len(data), batch_size):
            xb = data[perm[i:i + batch_size]].to(device)
            recon, _ = model(xb)
            loss = F.mse_loss(recon, xb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            run += loss.item() * len(xb)
        if epochs <= 30 or (ep + 1) % max(1, epochs // 15) == 0 or ep + 1 == epochs:   # ~15 log lines
            log(f"  [{tag}] epoch {ep + 1:2d}/{epochs}  recon MSE = {run / len(data):.5f}")
    return model


@torch.no_grad()
def ae_errors_and_latents(model, X, device, batch=256):
    model.to(device).eval()
    errs, zs = [], []
    for i in range(0, len(X), batch):
        xb = torch.tensor(X[i:i + batch]).to(device)
        r, z = model(xb)
        errs.append(((xb - r) ** 2).mean(dim=(1, 2, 3)).cpu().numpy())
        zs.append(z.cpu().numpy())
    return np.concatenate(errs), np.concatenate(zs)


# ================================================================ supervised
class DualBranchNet(nn.Module):
    """ResNet-18 image branch (512-d) + Dense(n_tab -> 64) tabular branch, each
    LayerNorm-ed, concatenated, then Head A (5-class) and Head B (merger).
    mode='image' / 'tabular' removes a branch for the controlled ablation."""

    def __init__(self, n_tab, in_ch=3, n_classes=N_CLASSES, mode="hybrid", pretrained=False, dropout=0.3):
        super().__init__()
        self.mode = mode
        dim = 0
        if mode in ("hybrid", "image"):
            w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
            self.backbone = torchvision.models.resnet18(weights=w)
            if in_ch != 3:
                self.backbone.conv1 = nn.Conv2d(in_ch, 64, 7, 2, 3, bias=False)
            self.backbone.fc = nn.Identity()
            self.img_norm = nn.LayerNorm(512)
            dim += 512
        if mode in ("hybrid", "tabular"):
            self.tab_branch = nn.Sequential(nn.Linear(n_tab, 64), nn.ReLU())
            self.tab_norm = nn.LayerNorm(64)
            dim += 64
        self.drop = nn.Dropout(dropout)
        self.head_morph = nn.Linear(dim, n_classes)
        self.head_merger = nn.Linear(dim, 1)

    def forward(self, x_img, x_tab):
        parts = []
        if self.mode in ("hybrid", "image"):
            parts.append(self.img_norm(self.backbone(x_img)))
        if self.mode in ("hybrid", "tabular"):
            parts.append(self.tab_norm(self.tab_branch(x_tab)))
        z = self.drop(torch.cat(parts, dim=1))
        return {"morph": self.head_morph(z), "merger": self.head_merger(z).squeeze(1)}


class EfficientNetMember(nn.Module):
    def __init__(self, in_ch=3, n_classes=N_CLASSES, pretrained=False):
        super().__init__()
        w = torchvision.models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        self.net = torchvision.models.efficientnet_b0(weights=w)
        if in_ch != 3:
            old = self.net.features[0][0]
            self.net.features[0][0] = nn.Conv2d(in_ch, old.out_channels, 3, 2, 1, bias=False)
        self.net.classifier[1] = nn.Linear(self.net.classifier[1].in_features, n_classes)

    def forward(self, x_img, x_tab=None):
        return {"morph": self.net(x_img)}


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma

    def forward(self, logits, y):
        ce = F.cross_entropy(logits, y, reduction="none")
        pt = torch.exp(-ce)
        w = self.alpha[y] if self.alpha is not None else 1.0
        return (w * (1 - pt) ** self.gamma * ce).mean()


def augment(x):
    """90-degree rotations and flips: morphology is orientation-invariant."""
    out = []
    for xi in x:
        xi = torch.rot90(xi, int(np.random.randint(4)), dims=(1, 2))
        out.append(torch.flip(xi, dims=(2,)) if np.random.rand() < 0.5 else xi)
    return torch.stack(out)


@torch.no_grad()
def predict_model(model, data, device, batch=64):
    model.to(device).eval()
    morph, merger = [], []
    for i in range(0, len(data["img"]), batch):
        out = model(data["img"][i:i + batch].to(device), data["tab"][i:i + batch].to(device))
        morph.append(F.softmax(out["morph"], 1).cpu().numpy())
        if "merger" in out:
            merger.append(torch.sigmoid(out["merger"]).cpu().numpy())
    res = {"morph": np.concatenate(morph)}
    if merger:
        res["merger"] = np.concatenate(merger)
    return res


def _balanced_acc(y, pred):
    import warnings
    if len(set(np.asarray(y).tolist())) < 2:
        return np.nan
    with warnings.catch_warnings():                     # "y_pred contains classes not in y_true"
        warnings.simplefilter("ignore", UserWarning)
        return round(float(balanced_accuracy_score(y, pred)), 4)


def morph_metrics(y, P, ym=None, pm=None):
    y = np.asarray(y)
    pred = P.argmax(1)
    present = sorted(set(y.tolist()) | set(pred.tolist()))      # classes absent from both are not scored
    out = {"macro_f1": round(float(f1_score(y, pred, average="macro", labels=present, zero_division=0)), 4),
           "balanced_acc": _balanced_acc(y, pred),
           "n": int(len(y))}
    if pm is not None and ym is not None and 0 < np.asarray(ym).sum() < len(ym):
        out["merger_pr_auc"] = round(float(average_precision_score(ym, pm)), 4)
        out["merger_prevalence"] = round(float(np.mean(ym)), 4)
    return out


def fit_classifier(model, tr, dev, device, epochs=15, batch_size=32, lr=1e-3, weight_decay=1e-4,
                   gamma=2.0, merger_weight=0.5, seed=0, log=print, tag="model"):
    """Focal loss on Head A (+ weighted BCE on Head B if present). Every model gets the
    same fixed epoch budget and keeps its best dev-Macro-F1 checkpoint."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    model.to(device)
    y = tr["y"].numpy()
    counts = np.bincount(y, minlength=N_CLASSES)
    alpha = torch.tensor(len(y) / (N_CLASSES * np.maximum(counts, 1)), dtype=torch.float32).to(device)
    focal = FocalLoss(alpha, gamma)
    pos = float(tr["ym"].sum())
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor((len(y) - pos) / max(pos, 1.0)).to(device))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best, best_nll, best_state, n = -1.0, -np.inf, copy.deepcopy(model.state_dict()), len(y)
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n)
        run = 0.0
        for i in range(0, n, batch_size):
            b = perm[i:i + batch_size]
            if len(b) < 2:          # BatchNorm needs > 1 sample
                continue
            out = model(augment(tr["img"][b]).to(device), tr["tab"][b].to(device))
            loss = focal(out["morph"], tr["y"][b].to(device))
            if "merger" in out:
                loss = loss + merger_weight * bce(out["merger"], tr["ym"][b].to(device))
            opt.zero_grad()
            loss.backward()
            opt.step()
            run += loss.item() * len(b)
        sched.step()
        p = predict_model(model, dev, device)
        yd = dev["y"].numpy()
        m = morph_metrics(yd, p["morph"])
        dev_nll = float(-np.mean(np.log(np.clip(p["morph"][np.arange(len(yd)), yd], 1e-12, 1)))) if len(yd) else np.inf
        log(f"  [{tag}] epoch {ep + 1:2d}/{epochs}  loss={run / n:.4f}  dev macro-F1={m['macro_f1']:.3f}  "
            f"dev NLL={dev_nll:.3f}")
        # best dev Macro-F1; ties (common with a small dev split) go to the lower dev log-loss
        if (m["macro_f1"], -dev_nll) > (best, best_nll):
            best, best_nll, best_state = m["macro_f1"], -dev_nll, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model


# ================================================================ calibration
def softmax_np(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))

In [ ]:
%%writefile src/p2p/utils.py
"""Numerical helpers shared by several stages."""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree


# ---------------------------------------------------------------- images
def asinh_normalise(img, q_lo=1.0, q_hi=99.5, soft=0.1):
    """Percentile-clipped asinh stretch to [0, 1]; robust to hot pixels."""
    p = np.nan_to_num(np.asarray(img, dtype=np.float32))
    lo, hi = np.percentile(p, [q_lo, q_hi])
    x = np.clip((p - lo) / max(hi - lo, 1e-8), 0, 1)
    return (np.arcsinh(x / soft) / np.arcsinh(1.0 / soft)).astype(np.float32)


def asinh_stack(cubes):
    """(N, C, H, W) -> asinh-normalised float32, each band of each object separately."""
    cubes = np.asarray(cubes, dtype=np.float32)
    out = np.empty_like(cubes)
    for n in range(cubes.shape[0]):
        for c in range(cubes.shape[1]):
            out[n, c] = asinh_normalise(cubes[n, c]) if np.any(cubes[n, c]) else 0.0
    return out


def gaussian_psf(fwhm_px: float, size: int = 25):
    ax = np.arange(size) - (size - 1) / 2
    xx, yy = np.meshgrid(ax, ax)
    s = fwhm_px / 2.3548
    g = np.exp(-(xx ** 2 + yy ** 2) / (2 * s * s))
    return g / g.sum()


def fwhm_halfmax(psf):
    """FWHM (pixels) from the area above half maximum: A = pi FWHM^2 / 4."""
    psf = np.asarray(psf, float)
    return float(2.0 * np.sqrt((psf >= 0.5 * psf.max()).sum() / np.pi))


def fwhm_fit(psf, half=7):
    """FWHM (pixels) of a source at the stamp centre from a 2-D Gaussian fit - smooth and not quantised to
    whole pixels like the half-maximum area. Robust for single-exposure stamps: only the central
    (2*half+1)^2 pixels are fitted (distant neighbours / cosmic-ray remnants cannot capture the fit), the
    fit starts at the brightest pixel within 2 px of the centre, and sigma (0.2-8 px) and the centre
    (+-2 px) are bounded, so one axis cannot collapse to zero. Falls back to the half-maximum area
    measured on a 4x upsampled image."""
    psf = np.asarray(psf, float)
    n0 = psf.shape[0]
    c0 = n0 // 2
    h = min(half, c0)
    img = np.nan_to_num(psf[c0 - h:c0 + h + 1, c0 - h:c0 + h + 1])
    n = img.shape[0]
    c = (n - 1) / 2
    try:
        import warnings
        from astropy.modeling import fitting, models
        yy, xx = np.mgrid[:n, :n]
        near = np.hypot(yy - c, xx - c) <= 2
        py, px = np.unravel_index(np.argmax(np.where(near, img, -np.inf)), img.shape)
        if not img[py, px] > 0:
            raise ValueError("no positive peak near the centre")
        img = img / img[py, px]                   # amplitude ~1: e-/s values of ~0.01 stall the optimiser
        g0 = models.Gaussian2D(1.0, px, py, 1.0, 1.0,
                               bounds={"x_stddev": (0.2, 8.0), "y_stddev": (0.2, 8.0),
                                       "x_mean": (c - 2, c + 2), "y_mean": (c - 2, c + 2), "amplitude": (0, None)})
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            g = fitting.TRFLSQFitter()(g0, xx, yy, img)
        f = 2.3548 * np.sqrt(abs(g.x_stddev.value * g.y_stddev.value))
        if 0.5 < f < 2.3548 * 7.9:
            return float(f)
    except Exception:
        pass
    from scipy.ndimage import zoom
    up = zoom(np.clip(img, 0, None), 4, order=1)
    return fwhm_halfmax(up) / 4.0


def half_light_radius(img):
    """Radius (px) enclosing half the positive flux, about the image centre."""
    img = np.clip(np.asarray(img, float), 0, None)
    h, w = img.shape
    yy, xx = np.mgrid[:h, :w]
    r = np.hypot(yy - (h - 1) / 2, xx - (w - 1) / 2).ravel()
    o = np.argsort(r)
    cog = np.cumsum(img.ravel()[o])
    if cog[-1] <= 0:
        return np.nan
    return float(r[o][np.searchsorted(cog, 0.5 * cog[-1])])


def cutout_snr(img, core_r=4, sky_r=40):
    """Peak SNR of a stamp: core mean over the MAD noise of the outer annulus."""
    img = np.asarray(img, float)
    h, w = img.shape
    yy, xx = np.mgrid[:h, :w]
    r = np.hypot(yy - h / 2, xx - w / 2)
    sky = img[r > sky_r]
    sigma = 1.4826 * np.median(np.abs(sky - np.median(sky))) + 1e-12
    return float((img[r <= core_r].mean() - np.median(sky)) / sigma)


def signed_log1p(x):
    return np.sign(x) * np.log1p(np.abs(x))


# ---------------------------------------------------------------- sky matching
def unit_vectors(ra, dec):
    ra = np.deg2rad(np.asarray(ra, float))
    dec = np.deg2rad(np.asarray(dec, float))
    return np.column_stack([np.cos(dec) * np.cos(ra), np.cos(dec) * np.sin(ra), np.sin(dec)])


def sky_crossmatch(ra1, dec1, ra2, dec2, radius_arcsec=0.5):
    """Nearest neighbour of each catalogue-1 row in catalogue 2 (exact great-circle
    separations via a KD-tree on unit vectors). Returns idx2 (-1 = none), sep", matched."""
    n = len(ra1)
    if n == 0 or len(ra2) == 0:
        return np.full(n, -1), np.full(n, np.inf), np.zeros(n, bool)
    chord, idx = cKDTree(unit_vectors(ra2, dec2)).query(unit_vectors(ra1, dec1))
    sep = np.rad2deg(2.0 * np.arcsin(np.clip(chord / 2.0, 0, 1))) * 3600.0
    hit = sep <= radius_arcsec
    return np.where(hit, idx, -1), sep, hit


def match_with_offset(ra1, dec1, ra2, dec2, radius_arcsec=0.5, search_arcsec=1.5, estimate=True):
    """Remove the median catalogue-to-image offset before the final match
    (old catalogue coordinates can be off by a few tenths of an arcsecond).
    Matches are one-to-one: if two image objects claim one catalogue row,
    only the closer keeps it. Returns idx2, sep, hit, (dra", ddec")."""
    ra2 = np.asarray(ra2, float).copy()
    dec2 = np.asarray(dec2, float).copy()
    off = (0.0, 0.0)
    if estimate and len(ra1) and len(ra2):
        idx, sep, hit = sky_crossmatch(ra1, dec1, ra2, dec2, search_arcsec)
        if hit.sum() >= 5:
            cosd = np.cos(np.deg2rad(np.asarray(dec1)[hit]))
            dra = np.median((np.asarray(ra1)[hit] - ra2[idx[hit]]) * cosd) * 3600
            ddec = np.median((np.asarray(dec1)[hit] - dec2[idx[hit]]) * 3600)
            ra2 = ra2 + dra / 3600 / np.cos(np.deg2rad(dec2))
            dec2 = dec2 + ddec / 3600
            off = (float(dra), float(ddec))
    idx, sep, hit = sky_crossmatch(ra1, dec1, ra2, dec2, radius_arcsec)
    order = np.argsort(np.where(hit, sep, np.inf))
    taken = set()
    for i in order:
        if hit[i]:
            if idx[i] in taken:
                hit[i] = False
            else:
                taken.add(idx[i])
    idx = np.where(hit, idx, -1)
    return idx, sep, hit, off


# ---------------------------------------------------------------- metrics
def predictive_entropy(P):
    P = np.clip(np.asarray(P, float), 1e-12, 1)
    return -(P * np.log(P)).sum(1) / np.log(P.shape[1])


def expected_calibration_error(P, y, n_bins=15):
    P, y = np.asarray(P), np.asarray(y)
    conf, correct = P.max(1), (P.argmax(1) == y).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)


def bootstrap_ci(metric_fn, y, P, n=1000, seed=0, alpha=0.05):
    """Percentile bootstrap CI for a metric(y, P) - small test sets need error bars."""
    rng = np.random.default_rng(seed)
    y, P = np.asarray(y), np.asarray(P)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        try:
            vals.append(metric_fn(y[i], P[i]))
        except Exception:
            pass
    if not vals:
        return (np.nan, np.nan)
    vals = np.asarray(vals, float)
    if not np.isfinite(vals).any():
        return (np.nan, np.nan)
    return (float(np.nanpercentile(vals, 100 * alpha / 2)), float(np.nanpercentile(vals, 100 * (1 - alpha / 2))))


# ---------------------------------------------------------------- identity
def iau_designation(ra, dec, prefix="P2P J"):
    """IAU style: coordinates TRUNCATED, e.g. P2P J123456.7+621234."""
    ra_h = (np.asarray(ra, float) / 15.0) % 24
    h = np.floor(ra_h)
    m = np.floor((ra_h - h) * 60)
    s = np.floor((((ra_h - h) * 60 - m) * 60) * 10) / 10
    d = np.abs(np.asarray(dec, float))
    dd = np.floor(d)
    dm = np.floor((d - dd) * 60)
    ds = np.floor(((d - dd) * 60 - dm) * 60)
    sign = np.where(np.asarray(dec) >= 0, "+", "-")
    return [f"{prefix}{int(a):02d}{int(b):02d}{c:04.1f}{sg}{int(e):02d}{int(f):02d}{int(g):02d}"
            for a, b, c, sg, e, f, g in zip(h, m, s, sign, dd, dm, ds)]


def noise_evidence(df: pd.DataFrame, cfg_baseline: dict) -> np.ndarray:
    """Physical evidence that a detection is spurious: very low SNR, a profile sharper
    than the PSF (no real source can be), or a single-band detection. The single-band test is the
    S/N-based cr_candidate of Stage 1c (no other covering band sees the source at S/N >= confirm_snr);
    the older flux-ratio test (band_consistency < single_band_max) is used only when that column is
    missing - a ratio test calls a real red galaxy that is faint in a shallow blue band 'single-band'."""
    ev = (df["snr"].to_numpy(float) < cfg_baseline["noise_snr_max"]) | \
         (df["fwhm_over_psf"].to_numpy(float) < cfg_baseline["sub_psf_factor"])
    if "cr_candidate" in df:
        ev |= as_bool(df["cr_candidate"]).to_numpy()
    elif "band_consistency" in df:
        bc = df["band_consistency"].to_numpy(float)
        ev |= np.isfinite(bc) & (bc < cfg_baseline.get("single_band_max", 0.05))
    return ev


def as_bool(series) -> pd.Series:
    """CSV round-trips turn booleans into strings / NaN; normalise them."""
    return series.map(lambda v: str(v).strip().lower() in ("true", "1", "1.0")).astype(bool)

In [ ]:
%%writefile src/p2p/xgb_worker.py
"""XGBoost in its own process.

On macOS, PyTorch and XGBoost each load their own OpenMP runtime (libomp). Two
OpenMP runtimes in one Python process can crash it with a segmentation fault,
which no try/except can catch. So every XGBoost call - fit, predict, TreeSHAP -
runs in a short-lived child process that never imports torch. Only the trained
booster (raw JSON bytes) travels back to the main process.

This module must NOT import torch (directly or through p2p.nets / p2p.inference).
"""
from __future__ import annotations

import numpy as np

XGB_PARAMS = dict(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
                  eval_metric="mlogloss", n_jobs=2)


def run_isolated(fn, *args):
    """Run fn(*args) in a FRESH Python process (`python -m p2p.xgb_worker`) that imports nothing but
    this module - multiprocessing's 'spawn' would re-import the main script, and with it PyTorch.
    A crash of the child (e.g. a segmentation fault) raises RuntimeError here instead of killing the run."""
    import os
    import pickle
    import subprocess
    import sys
    import tempfile
    from pathlib import Path
    pkg_parent = str(Path(__file__).resolve().parents[1])            # the folder that holds p2p/
    with tempfile.TemporaryDirectory() as d:
        inp, out = Path(d, "in.pkl"), Path(d, "out.pkl")
        inp.write_bytes(pickle.dumps((fn.__name__, args)))
        env = dict(os.environ, PYTHONPATH=pkg_parent + os.pathsep + os.environ.get("PYTHONPATH", ""))
        r = subprocess.run([sys.executable, "-m", "p2p.xgb_worker", str(inp), str(out)], env=env,
                           capture_output=True, text=True)
        if r.returncode != 0 or not out.exists():
            raise RuntimeError(f"XGBoost process failed (exit code {r.returncode}): {r.stderr.strip()[-400:]}")
        return pickle.loads(out.read_bytes())


def loaded_modules():
    """Diagnostics: the heavy libraries loaded in the worker process (torch must not be one)."""
    import sys
    return sorted({m.split(".")[0] for m in sys.modules} & {"torch", "torchvision", "xgboost"})


def fit(X, y, w, Xdev, ydev, seed):
    from xgboost import XGBClassifier
    kw = dict(XGB_PARAMS, random_state=seed)
    if Xdev is not None and len(Xdev):
        model = XGBClassifier(**kw, early_stopping_rounds=30)
        model.fit(X, y, sample_weight=w, eval_set=[(Xdev, ydev)], verbose=False)
    else:
        model = XGBClassifier(**kw).fit(X, y, sample_weight=w)
    best = getattr(model, "best_iteration", None)
    return bytes(model.get_booster().save_raw("json")), (None if best is None else int(best))


def _booster(raw):
    import xgboost as xgb
    b = xgb.Booster()
    b.load_model(bytearray(raw))
    return b, xgb


def predict(raw, best, X):
    b, xgb = _booster(raw)
    rng = (0, best + 1) if best is not None else (0, 0)
    p = np.asarray(b.predict(xgb.DMatrix(np.asarray(X, np.float32)), iteration_range=rng))
    return np.column_stack([1 - p, p]) if p.ndim == 1 else p          # binary -> two columns


def contribs(raw, best, X):
    """Exact TreeSHAP values, shape (n, classes, features) without the bias column."""
    b, xgb = _booster(raw)
    rng = (0, best + 1) if best is not None else (0, 0)
    c = np.asarray(b.predict(xgb.DMatrix(np.asarray(X, np.float32)), pred_contribs=True, iteration_range=rng))
    if c.ndim == 2:                                                   # binary: (n, features + 1)
        c = c[:, None, :]
    return c[:, :, :-1]


if __name__ == "__main__":                     # the child process started by run_isolated
    import pickle
    import sys
    name, args = pickle.loads(open(sys.argv[1], "rb").read())
    result = globals()[name](*args)
    with open(sys.argv[2], "wb") as fh:
        pickle.dump(result, fh)

### 2.4 Stages — `src/p2p/stages/` (one file per stage)

In [ ]:
%%writefile src/p2p/stages/__init__.py
"""One module per pipeline stage. Each exposes run(ctx). See p2p.context.STAGES for the order."""

In [ ]:
%%writefile src/p2p/stages/stage0_partition.py
"""Stage 0 - Data governance: sky-region partitioning with a leakage collar.

Purpose : fix the train / dev / test / external split BY SKY REGION before any
          pixel is analysed, so overlapping pointings cannot leak one object
          into two buckets.
Method  : auto mode reads only the WCS headers of the files in data/raw and
          builds RA/Dec boxes:
            * >= N GZH pointings  -> whole pointings are assigned to buckets
                                     (largest-deficit rule, 70/15/15 by area);
            * fewer pointings     -> each pointing is cut into stripes along its
                                     longer axis so that each stripe holds 70/15/15
                                     of the area that HAS DATA (WHT > 0 in every band;
                                     the empty corners of a drizzled frame don't count).
          Pointings matching partition.external_patterns become 'external'
          (CANDELS cross-catalogue test). A collar (cos(dec)-corrected) is kept
          around every box; a source in a collar or in two boxes gets no split.
Output  : outputs/stage0_partition/split_rule.json (+ a footprint plot)
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..context import Ctx, load_json, save_json
from ..fitsio import bbox_of, find_products, group_pointings, is_external, read_meta


class SkyRegionPartitioner:
    def __init__(self, buffer_arcmin: float):
        self.buffer_deg = buffer_arcmin / 60.0
        self.regions: dict[str, dict] = {}

    def define_region(self, name, ra_min, ra_max, dec_min, dec_max, split):
        self.regions[name] = {"split_tag": split, "ra_range": [ra_min, ra_max], "dec_range": [dec_min, dec_max]}

    def assign_split(self, ra, dec):
        inner = None
        for reg in self.regions.values():
            r0, r1 = reg["ra_range"]
            d0, d1 = reg["dec_range"]
            cosd = np.cos(np.deg2rad(0.5 * (d0 + d1)))
            b_ra = self.buffer_deg / max(cosd, 1e-3)
            in_inner = (r0 <= ra <= r1) and (d0 <= dec <= d1)
            in_outer = (r0 - b_ra <= ra <= r1 + b_ra) and (d0 - self.buffer_deg <= dec <= d1 + self.buffer_deg)
            if in_inner:
                if inner is not None:
                    return None
                inner = reg["split_tag"]
            elif in_outer:
                return None
        return inner

    def assign_many(self, ra, dec):
        return [self.assign_split(float(r), float(d)) for r, d in zip(ra, dec)]

    def to_dict(self):
        return {"buffer_deg": self.buffer_deg, "regions": self.regions}

    @classmethod
    def from_file(cls, path):
        d = load_json(path)
        p = cls(d["buffer_deg"] * 60.0)
        p.regions = d["regions"]
        return p


def _area(box):
    r0, r1, d0, d1 = box
    return (r1 - r0) * np.cos(np.deg2rad(0.5 * (d0 + d1))) * (d1 - d0)


def _stripes(box, fractions):
    """Cut a box into consecutive stripes along its longer on-sky axis."""
    r0, r1, d0, d1 = box
    width_ra = (r1 - r0) * np.cos(np.deg2rad(0.5 * (d0 + d1)))
    along_ra = width_ra >= (d1 - d0)
    edges = np.concatenate([[0.0], np.cumsum(list(fractions.values()))])
    edges = edges / edges[-1]
    out = []
    for tag, a, b in zip(fractions.keys(), edges[:-1], edges[1:]):
        if along_ra:
            out.append((tag, (r0 + a * (r1 - r0), r0 + b * (r1 - r0), d0, d1)))
        else:
            out.append((tag, (r0, r1, d0 + a * (d1 - d0), d0 + b * (d1 - d0))))
    return out


def coverage_points(ms, stride=8):
    """Sky positions of pixels with data (WHT > 0) in EVERY file of a pointing, sampled
    every `stride` pixels. A drizzled ACS/WFC3 frame is a tilted quadrilateral of data
    inside a larger, zero-weight array, so the array's RA/Dec box overstates the data
    area - the corners of that box are empty. Only coverage is read here, never sources."""
    from ..fitsio import load_image, valid_mask
    ref = max(ms, key=lambda m: m["naxis1"] * m["naxis2"])
    sci, wht, w = load_image(ref["file"])
    yy, xx = np.mgrid[0:sci.shape[0]:stride, 0:sci.shape[1]:stride]
    ok = valid_mask(sci, wht)[yy, xx]
    ra, dec = w.pixel_to_world_values(xx[ok], yy[ok])
    keep = np.ones(len(ra), bool)
    for m in ms:
        if m is ref:
            continue
        s2, w2, wcs2 = load_image(m["file"])
        x2, y2 = wcs2.world_to_pixel_values(ra, dec)
        xi, yi = np.round(x2).astype(int), np.round(y2).astype(int)
        inb = (xi >= 0) & (yi >= 0) & (xi < s2.shape[1]) & (yi < s2.shape[0])
        v = np.zeros(len(ra), bool)
        v[inb] = valid_mask(s2, w2)[yi[inb], xi[inb]]
        keep &= v
    if keep.sum() < 0.05 * len(keep):            # bands hardly overlap: fall back to the largest file
        print(f"[Stage 0] WARNING: the bands of {ref['name']} overlap on only {keep.mean():.1%} of its area; "
              "stripes are cut from that file's coverage alone.")
        keep[:] = True
    return np.asarray(ra)[keep], np.asarray(dec)[keep]


def _stripes_equal_area(ra, dec, fractions):
    """Cut along the longer axis of the DATA area at the quantiles of the covered pixels,
    so each stripe holds its share (70/15/15) of the real data area, not of the box."""
    cosd = np.cos(np.deg2rad(np.median(dec)))
    along_ra = np.ptp(ra) * cosd >= np.ptp(dec)
    t = ra if along_ra else dec
    cum = np.cumsum(list(fractions.values()))
    cuts = np.concatenate([[t.min()], np.quantile(t, cum[:-1] / cum[-1]), [t.max()]])
    r0, r1, d0, d1 = ra.min(), ra.max(), dec.min(), dec.max()
    out = []
    for tag, a, b in zip(fractions.keys(), cuts[:-1], cuts[1:]):
        out.append((tag, (a, b, d0, d1) if along_ra else (r0, r1, a, b)))
    return out


def build_auto(metas, groups, cfg_part):
    fr = cfg_part["fractions"]
    part = SkyRegionPartitioner(cfg_part["collar_arcmin"])
    gzh, ext = [], []
    for gi, g in enumerate(groups):
        ms = [metas[i] for i in g]
        (ext if any(is_external(m, cfg_part.get("external_patterns")) for m in ms) else gzh).append((gi, bbox_of(ms)))
    for gi, box in ext:
        part.define_region(f"P{gi + 1:02d}_external", *box, split="external")
    if len(gzh) >= cfg_part.get("min_pointings_for_whole_assignment", 3):
        totals = {k: 0.0 for k in fr}
        total_area = sum(_area(b) for _, b in gzh)
        for gi, box in sorted(gzh, key=lambda t: -_area(t[1])):          # biggest first
            deficit = {k: fr[k] - totals[k] / total_area for k in fr}
            empty = [k for k in fr if totals[k] == 0]
            tag = empty[0] if empty and len(gzh) >= len(fr) else max(deficit, key=deficit.get)
            totals[tag] += _area(box)
            part.define_region(f"P{gi + 1:02d}", *box, split=tag)
        mode = "whole pointings"
    else:
        narrowest = np.inf
        cover = []
        for gi, box in gzh:
            try:
                ra, dec = coverage_points([metas[i] for i in groups[gi]])
                cover.append((ra, dec))
                stripes = _stripes_equal_area(ra, dec, fr)
            except Exception as e:                   # unreadable pixels: fall back to the header box
                print(f"[Stage 0] WARNING: could not read the coverage of pointing {gi + 1} ({e!r}); "
                      "cutting stripes from the header footprint.")
                stripes = _stripes(box, fr)
            for tag, sbox in stripes:
                part.define_region(f"P{gi + 1:02d}_{tag}", *sbox, split=tag)
                r0, r1, d0, d1 = sbox
                w = min((r1 - r0) * np.cos(np.deg2rad(0.5 * (d0 + d1))), d1 - d0) * 60   # arcmin
                narrowest = min(narrowest, w)
        mode = "stripes inside each pointing"
        if part.buffer_deg * 60 > 0.25 * narrowest:
            new = 0.25 * narrowest
            print(f"[Stage 0] WARNING: collar {part.buffer_deg * 60:.3f}' would swallow the narrowest stripe "
                  f"({narrowest:.3f}' wide); using {new:.3f}' instead.")
            part.buffer_deg = new / 60.0
        if cover:                                    # what each split REALLY got of the data area
            tags = pd.Series([t for ra, dec in cover for t in part.assign_many(ra, dec)]).fillna("collar")
            return part, mode, tags.value_counts(normalize=True).round(4).to_dict()
    return part, mode, None


def run(ctx: Ctx):
    cfg = ctx.cfg["partition"]
    out = ctx.stage_dir("0")
    if cfg["mode"] == "manual":
        part = SkyRegionPartitioner(cfg["collar_arcmin"])
        for r in cfg["manual_regions"]:
            part.define_region(r["name"], r["ra"][0], r["ra"][1], r["dec"][0], r["dec"][1], r["split"])
        mode, metas, groups, share = "manual", [], [], None
    else:
        files = find_products(ctx.raw, ctx.cfg["acquisition"]["file_patterns"])
        if not files:
            raise FileNotFoundError(f"No *_drc.fits / *_drz.fits files in {ctx.raw}. Put your HST files there "
                                    "(python tools/download_mast.py --ra <RA> --dec <Dec> --filters ...).")
        metas = [read_meta(f) for f in files]
        metas = [m for m in metas if m["wcs_ok"]]
        acq = ctx.cfg["acquisition"]
        groups = group_pointings(metas, acq["group_radius_arcsec"], acq.get("group_min_overlap", 0.5))
        for gi, g in enumerate(groups):
            ctx.log.info(f"[Stage 0] pointing P{gi + 1:02d}: {[metas[i]['filter'] for i in g]} "
                         f"({', '.join(metas[i]['name'] for i in g)})")
        part, mode, share = build_auto(metas, groups, cfg)

    save_json({**part.to_dict(), "mode": mode, "fractions": cfg["fractions"], "data_area_share": share},
              out / "split_rule.json")
    _plot(part, metas, out / "split_regions.png")
    tags = [r["split_tag"] for r in part.regions.values()]
    ctx.log.info(f"[Stage 0] {len(part.regions)} regions ({mode}); collar {part.buffer_deg * 60:.3f}' ; "
                 f"tags: { {t: tags.count(t) for t in set(tags)} }")
    if share:
        ctx.log.info(f"[Stage 0] share of the area WITH DATA per split (collar = no split): {share}")
    ctx.log.info(f"[Stage 0] Ruleset frozen -> {ctx.rel(out / 'split_rule.json')}")
    return {"mode": mode, "n_regions": len(part.regions)}


def _plot(part, metas, path):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        from matplotlib.patches import Polygon, Rectangle
    except Exception:
        return
    colours = {"train": "#2E5496", "dev": "#E69F00", "test": "#009E73", "external": "#CC79A7"}
    fig, ax = plt.subplots(figsize=(6, 5))
    for m in metas:
        ax.add_patch(Polygon(np.asarray(m["footprint"]), fill=False, ec="0.6", lw=0.6))
    for name, r in part.regions.items():
        (r0, r1), (d0, d1) = r["ra_range"], r["dec_range"]
        ax.add_patch(Rectangle((r0, d0), r1 - r0, d1 - d0, alpha=0.35, color=colours.get(r["split_tag"], "grey")))
        ax.text((r0 + r1) / 2, (d0 + d1) / 2, r["split_tag"], ha="center", va="center", fontsize=7)
    ax.autoscale()
    ax.invert_xaxis()
    ax.set_xlabel("RA [deg]")
    ax.set_ylabel("Dec [deg]")
    ax.set_title("Stage 0: sky regions (grey = file footprints)")
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage1a_acquisition.py
"""Stage 1a - Archive acquisition (local scan, optional MAST download).

Purpose : know exactly which science products the run uses and verify them.
Method  : (optional) download Level-3 _drc products from MAST with astroquery;
          then scan data/raw, prefer _drc over _drz, check each file has a SCI
          extension, a WHT map and a celestial WCS, read the photometric
          keywords (PHOTFLAM/PHOTPLAM -> AB zero-point), group files into
          pointings and choose the band set used for the 3-channel cutouts.
Output  : manifest.json (one entry per file) and pointings.json
          (pointing -> role, bands, detection band, files).
"""
from __future__ import annotations

from collections import Counter

import numpy as np

from ..context import Ctx, save_json
from ..fitsio import find_products, group_pointings, is_external, read_meta
from .stage0_partition import SkyRegionPartitioner


def download_from_mast(ctx: Ctx):
    """Optional: fetch _drc products for the targets in config.yaml (needs internet)."""
    dl = ctx.cfg["acquisition"]["download"]
    try:
        from astroquery.mast import Observations
    except Exception:
        ctx.log.info("[Stage 1a] astroquery missing - pip install astroquery to download.")
        return
    ctx.raw.mkdir(parents=True, exist_ok=True)
    for t in dl["targets"]:
        obs = Observations.query_criteria(coordinates=f"{t['ra']} {t['dec']}", radius=f"{t['radius_deg']} deg",
                                          obs_collection="HST", dataproduct_type="image", calib_level=3,
                                          instrument_name=t.get("instrument", "ACS/WFC"),
                                          filters=t.get("filters"))
        ctx.log.info(f"[Stage 1a] MAST: {len(obs)} observations near ({t['ra']}, {t['dec']})")
        if len(obs) == 0:
            continue
        prods = Observations.filter_products(Observations.get_product_list(obs), productType="SCIENCE",
                                             productSubGroupDescription="DRC", extension="fits")
        # one product per filter, smallest first
        prods.sort("size")
        seen, keep = set(), []
        for i, row in enumerate(prods):
            f = str(row["productFilename"])
            filt = next((x for x in t.get("filters", []) if x.lower() in f.lower()), f)
            if filt not in seen:
                seen.add(filt)
                keep.append(i)
            if len(keep) >= dl.get("max_files_per_target", 3):
                break
        if keep:
            man = Observations.download_products(prods[keep], download_dir=str(ctx.raw), flat=True)
            ctx.log.info(f"[Stage 1a] downloaded: {[str(p) for p in man['Local Path']]}")


def choose_bands(pointings, metas, cfg):
    want = cfg["bands"]
    if want != "auto":
        return [b.upper() for b in want]
    counts = Counter(metas[i]["filter"] for p in pointings for i in p)
    lam = {}
    for m in metas:
        if m["photplam"]:
            lam[m["filter"]] = float(m["photplam"])
    best = [b for b, _ in counts.most_common(cfg.get("max_bands", 3))]
    return sorted(best, key=lambda b: lam.get(b, 1e9))          # blue -> red


def run(ctx: Ctx):
    acfg = ctx.cfg["acquisition"]
    out = ctx.stage_dir("1a")
    if acfg["download"].get("enabled"):
        download_from_mast(ctx)
    files = find_products(ctx.raw, acfg["file_patterns"])
    if not files:
        raise FileNotFoundError(f"No HST products found in {ctx.raw}.")

    metas, rejected = [], []
    for f in files:
        m = read_meta(f)
        problems = [k for k in ("has_sci", "wcs_ok") if not m[k]]
        if not m["has_wht"]:
            problems.append("no WHT (coverage will be taken from finite SCI pixels)")
        m["problems"] = problems
        (rejected if ("has_sci" in problems or "wcs_ok" in problems) else metas).append(m)
    if not metas:
        raise RuntimeError(f"All files failed the structural check: {[r['name'] for r in rejected]}")

    groups = group_pointings(metas, acfg["group_radius_arcsec"], acfg.get("group_min_overlap", 0.5))
    bands = choose_bands(groups, metas, acfg)
    rules = SkyRegionPartitioner.from_file(ctx.require("0", "split_rule.json"))
    pointings = {}
    for gi, g in enumerate(groups):
        pid = f"P{gi + 1:02d}"
        ms = [metas[i] for i in g]
        by_band = {}
        for m in sorted(ms, key=lambda m: (m["product_type"] != "drc", -m["exptime"])):
            by_band.setdefault(m["filter"], m)          # deepest _drc per band
        used = {b: by_band[b]["file"] for b in bands if b in by_band}
        if not used:
            ctx.log.info(f"[Stage 1a] {pid}: none of the bands {bands} present - pointing skipped")
            continue
        if acfg["detection_band"] != "auto" and acfg["detection_band"] in used:
            det = acfg["detection_band"]
        else:
            det = max(used, key=lambda b: by_band[b]["photplam"] or 0)          # reddest
        role = "external" if any(is_external(m, ctx.cfg["partition"].get("external_patterns")) for m in ms) else "gzh"
        cen = (float(np.mean([m["ra_centre"] for m in ms])), float(np.mean([m["dec_centre"] for m in ms])))
        for m in ms:
            m["pointing"] = pid
        pointings[pid] = {"role": role, "bands": used, "missing_bands": [b for b in bands if b not in used],
                          "detection_band": det, "centre": cen,
                          "split_at_centre": rules.assign_split(*cen),
                          "instrument": by_band[det]["instrument"], "epoch": by_band[det]["epoch"],
                          "exptime_detection": by_band[det]["exptime"], "zp_ab": {b: by_band[b]["zp_ab"] for b in used},
                          "pixscale": {b: by_band[b]["pixscale_arcsec"] for b in used}}

    manifest = {m["name"]: {k: v for k, v in m.items()} for m in metas}
    save_json(manifest, out / "manifest.json")
    save_json({"bands": bands, "pointings": pointings, "rejected": [r["name"] for r in rejected]},
              out / "pointings.json")
    for pid, p in pointings.items():
        ctx.log.info(f"[Stage 1a] {pid} ({p['role']}): bands {list(p['bands'])} detection={p['detection_band']} "
                     f"missing={p['missing_bands']} instrument={p['instrument']}")
    ctx.log.info(f"[Stage 1a] {len(metas)} verified file(s), {len(rejected)} rejected; band set = {bands}")
    return {"files": len(metas), "pointings": len(pointings), "bands": bands}

In [ ]:
%%writefile src/p2p/stages/stage1b_frame_qc.py
"""Stage 1b - Frame quality control with a convolutional autoencoder.

Purpose : flag detector artefacts (satellite trails, bad drizzle regions, chip
          edges, hot-pixel clusters) that survive drizzle combination.
Method  : 128x128 sub-tiles of every SCI image are asinh-normalised; an
          under-complete CAE is trained on a random sample of well-covered
          tiles (artefacts are rare, so the model learns 'normal sky'); every
          tile is then scored with a 64-px stride (50% overlap, so an artefact
          on a tile edge is not split). The score is the reconstruction error
          with COMPACT sources masked (galaxies are rare too and would
          otherwise dominate); tiles above the 99th percentile of the scores of
          20 % HELD-OUT tiles (never trained on) are flagged.
Output  : tile_qc.csv (file, x0, y0, mse, flagged), qc_cae.pt, qc_threshold.json,
          qc_error_hist.png and one flag-map PNG per file.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import torch

from ..context import Ctx, load_json, save_json
from ..fitsio import load_image, valid_mask
from ..nets import ConvAutoencoder, train_autoencoder
from ..utils import asinh_normalise


def frame_sigma(sci, valid):
    d = sci[valid]
    if d.size > 2_000_000:
        d = d[:: d.size // 2_000_000]
    return float(1.4826 * np.median(np.abs(d - np.median(d)))) or 1.0


def qc_normalise(tile, sigma, cap=200.0):
    """ONE fixed stretch for every tile of a file (in units of the sky noise).
    A per-tile percentile stretch would blow empty-sky noise up to full range and
    squash a satellite trail - exactly the opposite of what an anomaly detector needs."""
    x = np.arcsinh(np.clip(tile / (3.0 * sigma), 0, None)) / np.arcsinh(cap)
    return np.clip(x, 0, 1).astype(np.float32)


def source_masked_mse(x, recon, tile, level=0.2, min_extent_frac=0.5, max_fill=0.25):
    """Reconstruction error with SOURCE-LIKE structure masked.

    Galaxies are rare too, so a plain MSE mostly flags bright galaxies. Every
    connected structure above `level` is masked unless it looks like an artefact:
      * long and thin  (extent >= half a tile, filling < 25% of its box) -> trail
      * very large     (area >= a quarter of the tile)                   -> bad patch
    so the score measures what the CAE cannot explain OUTSIDE ordinary sources."""
    from scipy import ndimage
    errs = np.empty(len(x), np.float32)
    for i in range(len(x)):
        img = x[i, 0]
        lab, n = ndimage.label(img > level)
        keep = np.ones_like(img, bool)
        for k, sl in enumerate(ndimage.find_objects(lab), start=1):
            if sl is None:
                continue
            h, w = sl[0].stop - sl[0].start, sl[1].stop - sl[1].start
            area = int((lab[sl] == k).sum())
            thin_long = max(h, w) >= min_extent_frac * tile and area / (h * w) < max_fill
            big = area >= 0.25 * tile * tile
            if not (thin_long or big):
                keep[sl] &= lab[sl] != k
        d = (img - recon[i, 0]) ** 2
        errs[i] = d[keep].mean() if keep.any() else 0.0
    return errs


@torch.no_grad()
def reconstruct(model, X, device, batch=256):
    model.to(device).eval()
    out = []
    for i in range(0, len(X), batch):
        r, _ = model(torch.tensor(X[i:i + batch]).to(device))
        out.append(r.cpu().numpy())
    return np.concatenate(out)


def tiles(sci, valid, size, stride, min_cov):
    ny, nx = sci.shape
    for y0 in range(0, ny - size + 1, stride):
        for x0 in range(0, nx - size + 1, stride):
            cov = valid[y0:y0 + size, x0:x0 + size].mean()
            yield x0, y0, cov


def run(ctx: Ctx):
    cfg = ctx.cfg["frame_qc"]
    out = ctx.stage_dir("1b")
    pts = load_json(ctx.require("1a", "pointings.json"))["pointings"]
    files = sorted({f for p in pts.values() for f in p["bands"].values()})
    size, rng = cfg["tile"], np.random.default_rng(ctx.seed)
    device = ctx.device()

    # ---- 1. training sample of well-covered tiles from every file ----
    train = []
    for f in files:
        sci, wht, _ = load_image(f)
        val = valid_mask(sci, wht)
        sig = frame_sigma(sci, val)
        sci = np.where(val, sci - np.median(sci[val]), 0.0)
        for x0, y0, cov in tiles(sci, val, size, cfg["train_stride"], cfg["min_tile_coverage"]):
            if cov >= cfg["min_tile_coverage"]:
                train.append(qc_normalise(sci[y0:y0 + size, x0:x0 + size], sig))
    if len(train) > cfg["max_train_tiles"]:
        train = [train[i] for i in rng.choice(len(train), cfg["max_train_tiles"], replace=False)]
    X = np.stack(train)[:, None]
    # 20 % of the tiles are HELD OUT: the threshold is set on tiles the CAE never trained on. A threshold
    # from its own training tiles is biased low (they are reconstructed best) and flags ordinary sky.
    perm = rng.permutation(len(X))
    n_cal = max(int(round(0.2 * len(X))), min(20, len(X) // 2))
    Xcal, Xfit = X[perm[:n_cal]], X[perm[n_cal:]]
    ctx.log.info(f"[Stage 1b] training QC-CAE on {len(Xfit)} tiles from {len(files)} file(s) ({device}); "
                 f"{len(Xcal)} held-out tiles set the threshold")
    model = ConvAutoencoder(in_ch=1, latent_dim=cfg["latent_dim"], size=size)
    steps_per_epoch = int(np.ceil(len(Xfit) / cfg["batch_size"]))
    epochs = max(int(cfg["epochs"]), int(np.ceil(cfg.get("min_steps", 0) / steps_per_epoch)))
    ctx.log.info(f"[Stage 1b] {epochs} epochs x {steps_per_epoch} steps = {epochs * steps_per_epoch} optimisation steps")
    train_autoencoder(model, Xfit, epochs, cfg["batch_size"], device, log=ctx.log.info, tag="QC-CAE")
    train_err = source_masked_mse(Xfit, reconstruct(model, Xfit, device), size)
    cal_err = source_masked_mse(Xcal, reconstruct(model, Xcal, device), size)
    thr = float(np.percentile(cal_err, cfg["threshold_percentile"]))
    torch.save(model.state_dict(), out / "qc_cae.pt")
    save_json({"threshold_mse": thr, "percentile": cfg["threshold_percentile"], "n_train_tiles": len(Xfit),
               "n_heldout_tiles": len(Xcal), "median_mse_train": float(np.median(train_err)),
               "median_mse_heldout": float(np.median(cal_err))},
              out / "qc_threshold.json")

    # ---- 2. score every tile (overlapping) of every file ----
    rows = []
    for f in files:
        sci, wht, _ = load_image(f)
        val = valid_mask(sci, wht)
        sig = frame_sigma(sci, val)
        sci = np.where(val, sci - np.median(sci[val]), 0.0)
        pos, batch = [], []
        for x0, y0, cov in tiles(sci, val, size, cfg["score_stride"], 0):
            if cov < 0.05:            # empty border: nothing to judge
                continue
            pos.append((x0, y0, cov))
            batch.append(qc_normalise(sci[y0:y0 + size, x0:x0 + size], sig))
        Xs = np.stack(batch)[:, None]
        err = source_masked_mse(Xs, reconstruct(model, Xs, device), size)
        for (x0, y0, cov), e in zip(pos, err):
            rows.append({"file": f, "x0": x0, "y0": y0, "size": size, "coverage": round(cov, 3),
                         "mse": float(e), "flagged": bool(e > thr)})
        _flag_map(f, sci, val, pos, err, thr, out)
    df = pd.DataFrame(rows)
    df.to_csv(out / "tile_qc.csv", index=False)
    _hist(cal_err, df["mse"].to_numpy(), thr, out / "qc_error_hist.png")
    frac = df.groupby("file")["flagged"].mean().round(4).to_dict()
    ctx.log.info(f"[Stage 1b] threshold MSE = {thr:.5f}; flagged tile fraction per file: "
                 f"{ {Path(k).name: v for k, v in frac.items()} }")
    return {"threshold": thr, "flagged_fraction": float(df["flagged"].mean())}


def _hist(train_err, all_err, thr, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5.5, 3.2))
    ax.hist(all_err, bins=60, alpha=0.6, label="all scored tiles")
    ax.hist(train_err, bins=60, alpha=0.6, label="held-out tiles (set the threshold)")
    ax.axvline(thr, color="r", ls="--", label=f"threshold {thr:.4f}")
    ax.set_xlabel("reconstruction MSE")
    ax.set_yscale("log")
    ax.legend(fontsize=8)
    ax.set_title("Stage 1b: frame QC")
    fig.tight_layout()
    fig.savefig(path, dpi=110)
    plt.close(fig)


def _flag_map(f, sci, val, pos, err, thr, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    step = max(1, max(sci.shape) // 1024)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(asinh_normalise(sci[::step, ::step]), origin="lower", cmap="gray")
    for (x0, y0, _), e in zip(pos, err):
        if e > thr:
            ax.add_patch(Rectangle((x0 / step, y0 / step), 128 / step, 128 / step, fill=False, ec="red", lw=0.8))
    ax.set_title(f"{Path(f).name}: flagged tiles (red)", fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.tight_layout()
    fig.savefig(out / (Path(f).name.replace(".gz", "").replace(".fits", "") + "_qc.png"), dpi=100)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage1c_detection.py
"""Stage 1c - Detection, deblending, band registration, cosmic-ray checks and aligned cutouts.

Purpose : locate every source and register all filters into one cutout.
Method  : per pointing, each band is background-subtracted (Background2D on
          covered pixels only). Sources are detected on the smoothed detection
          band (nsigma x rms, npixels) and deblended.
          Band registration: bright extended sources of every band, detected on
          a median-filtered image (cosmic rays erased), are matched to those of
          the detection band; the offset is removed from that band's WCS (bands
          from different HST programmes can disagree by ~1"). An offset that
          does not stand out from random pairs stops the stage; a known offset can
          be given in detection.band_offsets_arcsec.
          The stage FAILS if a residual offset exceeds
          detection.max_band_offset_arcsec (multiband.py).
          Cosmic rays (single exposures cannot reject them): every detection
          segment is measured in every other band; a source that no other
          covering band confirms at S/N >= detection.confirm_snr is a cosmic-ray
          candidate (cr_candidate). The source cap is applied to confirmed sources
          and to cosmic-ray candidates separately, so cosmic rays cannot crowd
          real sources out.
          For every source a north-up 128x128 stamp is resampled from EACH band
          at the same RA/Dec with a local affine WCS mapping, so bands with
          different pixel scales or orientations line up exactly. Values are
          scaled by the pixel-area ratio (e-/s per output pixel). Cosmic-ray
          pixels inside a stamp (much brighter in one band than the other bands
          predict from the source's colour) are replaced by that prediction and
          marked 3 in the segmentation stamp, so morphology masks them.
          A segmentation stamp records target (1) / other sources (2) / cosmic-ray
          pixels (3): the image models see the UNMASKED stamp (merger companions
          kept), while Stage 1e measures morphology with neighbours and cosmic-ray
          pixels masked. Truncation is flagged from the WHT coverage (edge_flag),
          frame-QC flags come from Stage 1b (any band), exposures per source from
          the drizzle context (CTX), and each source gets its Stage-0 split
          ('buffer' inside a collar).
Output  : sources.csv, cutouts_raw.npy (N, bands, 128, 128, float16),
          segstamps.npy (N, 128, 128, uint8), registration.json, preview PNGs.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from photutils.utils import calc_total_error
from scipy import ndimage

from ..compat import SourceCatalog, n_labels, values
from ..context import Ctx, load_json, save_json
from ..detection import detect_and_deblend, subtract_background
from ..fitsio import load_ctx, load_image, read_meta, valid_mask
from ..multiband import (cr_pixel_mask, measure_offset, median_image, popcount, registration_catalog,
                         segment_photometry, shift_wcs, tangent_arcsec)
from .stage0_partition import SkyRegionPartitioner


def affine_for(wcs, ra0, dec0, size, scale_arcsec):
    """Matrix/offset for ndimage.affine_transform mapping a north-up cutout
    (row, col) onto the band's pixel grid (row, col) around (ra0, dec0)."""
    c = (size - 1) / 2.0
    cosd = np.cos(np.deg2rad(dec0))
    # cutout pixel (u, v) -> sky: east is left (RA grows to -u), north is up
    def sky(du, dv):
        return ra0 - du * scale_arcsec / 3600.0 / cosd, dec0 + dv * scale_arcsec / 3600.0
    pts = [sky(0, 0), sky(1, 0), sky(0, 1)]
    xy = np.array([wcs.world_to_pixel_values(r, d) for r, d in pts], float)
    x0, y0 = xy[0]
    J = np.array([[xy[1, 0] - x0, xy[2, 0] - x0],          # dx/du dx/dv
                  [xy[1, 1] - y0, xy[2, 1] - y0]])         # dy/du dy/dv
    M = np.array([[J[1, 1], J[1, 0]],                      # (row, col) order
                  [J[0, 1], J[0, 0]]])
    offset = np.array([y0, x0]) - M @ np.array([c, c])
    return M, offset, abs(np.linalg.det(J))


def make_stamp(img, M, offset, size, order=1):
    return ndimage.affine_transform(img, M, offset=offset, output_shape=(size, size),
                                    order=order, mode="constant", cval=0.0, prefilter=False)


def flagged_tiles(ctx: Ctx):
    """Flagged tiles as their CENTRAL stride x stride cell. The QC tiles overlap (128 px every 64 px), so
    each pixel lies in the centre cell of exactly one tile per band; judging a source by every tile that
    touches it would give it up to 4 chances per band and inflate the flag rate several-fold."""
    p = ctx.path("1b", "tile_qc.csv")
    if not p.exists():
        return {}
    t = pd.read_csv(p)
    t = t[t["flagged"].astype(str).str.lower() == "true"]
    stride = int(ctx.cfg["frame_qc"].get("score_stride", 64))
    m = (t["size"] - stride) / 2.0
    t = t.assign(cx0=t.x0 + m, cy0=t.y0 + m, cs=np.minimum(stride, t["size"]))
    return {f: g[["cx0", "cy0", "cs"]].to_numpy(float) for f, g in t.groupby("file")}


def in_flagged(x, y, boxes):
    if boxes is None or len(boxes) == 0:
        return np.zeros(len(x), bool)
    x, y = np.asarray(x)[:, None], np.asarray(y)[:, None]
    x0, y0, s = boxes[:, 0][None], boxes[:, 1][None], boxes[:, 2][None]
    return np.any((x >= x0) & (x < x0 + s) & (y >= y0) & (y < y0 + s), axis=1)


def saturated_cores(sub, rms, x, y, half=4, tol=0.02, min_peak_sigma=50.0):
    """Flat-topped cores on the NATIVE pixels of the detection image (a resampled cutout rounds a flat top
    off): the 5 brightest pixels within +-4 px of the centroid agree to 2 % and the peak is >= 50 sigma."""
    h, w = sub.shape
    out = np.zeros(len(x), bool)
    for k, (xc, yc) in enumerate(zip(np.round(x).astype(int), np.round(y).astype(int))):
        y0, y1, x0, x1 = max(0, yc - half), min(h, yc + half + 1), max(0, xc - half), min(w, xc + half + 1)
        box = sub[y0:y1, x0:x1]
        if box.size < 5:
            continue
        top = np.sort(box.ravel())[-5:]
        out[k] = top[-1] > min_peak_sigma * rms[min(h - 1, max(0, yc)), min(w - 1, max(0, xc))] and \
            (top[-1] - top[0]) / top[-1] < tol
    return out


def register_bands(ctx, pid, imgs, det_b, dcfg):
    """Measure (and remove) the sky offset of every band against the detection band. Returns a report.
    Both lists come from median-filtered images (registration_catalog): in a single exposure the brightest
    'sources' are cosmic rays, which pair at random between bands."""
    det = imgs[det_b]
    ra0, dec0 = det["wcs"].wcs.crval
    limit = float(dcfg.get("max_band_offset_arcsec", 0.05))
    search = float(dcfg.get("register_search_arcsec", 3.0))
    min_sig = float(dcfg.get("register_min_significance", 5.0))
    manual = {str(k).upper(): v for k, v in (dcfg.get("band_offsets_arcsec") or {}).items()}
    rr, rd = registration_catalog(det["sub"], det["rms"], det["val"], det["wcs"], dcfg)
    ref_xy = np.column_stack(tangent_arcsec(rr, rd, ra0, dec0))
    report = {}
    for b, im in imgs.items():
        if b == det_b:
            continue
        br, bd = registration_catalog(im["sub"], im["rms"], im["val"], im["wcs"], dcfg)
        bpx = im["wcs"].world_to_pixel_values(br, bd)        # pixel positions: fixed, only the WCS changes

        def offset(wcs):
            r, d = wcs.pixel_to_world_values(*bpx)
            return measure_offset(ref_xy, np.column_stack(tangent_arcsec(r, d, ra0, dec0)), search)

        dx, dy, n, scat, sig = offset(im["wcs"])
        reliable = n >= 20 and sig >= min_sig
        rep = {"n_reference": len(ref_xy), "n_band": len(br), "n_pairs": n, "significance": round(sig, 1),
               "offset_measured_arcsec": [round(dx, 4), round(dy, 4)] if n else None,
               "scatter_arcsec": round(scat, 4) if n else None, "reliable": bool(reliable), "applied": None}
        if b in manual:                                  # offset given in config.yaml
            ox, oy = (float(v) for v in manual[b])
            rep["applied"] = {"source": "config detection.band_offsets_arcsec", "offset_arcsec": [ox, oy]}
        elif reliable and dcfg.get("register_bands", True) and np.hypot(dx, dy) > 0.01:
            ox, oy = dx, dy
            rep["applied"] = {"source": "measured", "offset_arcsec": [round(ox, 4), round(oy, 4)]}
        if rep["applied"]:
            im["wcs"] = shift_wcs(im["wcs"], ox, oy)
            dx, dy, n2, _, sig2 = offset(im["wcs"])
            reliable = n2 >= 20 and sig2 >= min_sig
            rep.update(n_pairs_after=n2, significance_after=round(sig2, 1))
        rep["offset_after_arcsec"] = [round(dx, 4), round(dy, 4)] if reliable else None
        rep["residual_arcsec"] = round(float(np.hypot(dx, dy)), 4) if reliable else None
        report[b] = rep
        ctx.log.info(f"[Stage 1c] {pid}: {b} vs {det_b}: {rep['n_pairs']} matched sources (peak {rep['significance']}x the "
                     f"random-pair level), offset {rep['offset_measured_arcsec']}\" -> "
                     f"{'corrected (' + rep['applied']['source'] + ')' if rep['applied'] else 'no correction'}, "
                     f"residual {rep['offset_after_arcsec']}\" (dRA cos(dec), dDec)")
        if not reliable:
            raise RuntimeError(
                f"[Stage 1c] {pid}: the offset of {b} against {det_b} cannot be measured reliably ({rep['n_pairs']} "
                f"pairs, {rep['significance']}x the random level; need >= 20 pairs and >= {min_sig}x). Colours and "
                f"two-band tests would use unchecked positions. If you know the offset, give it in config.yaml, e.g. "
                f"detection.band_offsets_arcsec: {{{b}: [dRA_cos_dec, dDec]}} (arcsec, band minus {det_b}).")
        if np.hypot(dx, dy) > limit:
            raise RuntimeError(
                f"[Stage 1c] {pid}: {b} is still offset by {np.hypot(dx, dy):.3f}\" from {det_b} after registration "
                f"(limit detection.max_band_offset_arcsec = {limit}\"). "
                + ("Turn on detection.register_bands." if not dcfg.get("register_bands", True) else
                   "Check detection.band_offsets_arcsec, or use a file whose WCS is aligned to a star catalogue."))
    return report


def process_pointing(ctx, pid, p, bands, rules, qc_boxes, pidx):
    dcfg = ctx.cfg["detection"]
    S = dcfg["cutout_size"]
    corr = dcfg.get("correlated_noise_factor", 1.0)
    imgs = {}
    for b, f in p["bands"].items():
        sci, wht, w = load_image(f)
        val = valid_mask(sci, wht)
        sub, rms = subtract_background(sci, val, dcfg["background_box"], dcfg["background_filter"])
        imgs[b] = {"sub": sub, "rms": rms, "val": val, "valf": val.astype(np.float32), "wcs": w, "file": f,
                   "scale": p["pixscale"][b], "pixarea": p["pixscale"][b] ** 2, "zp": p["zp_ab"][b],
                   "exptime": max(float(read_meta(Path(f))["exptime"]), 1.0)}
        del sci, wht
    det_b = p["detection_band"]
    det = imgs[det_b]
    segm, conv = detect_and_deblend(det["sub"], det["rms"], det["val"], dcfg)
    if n_labels(segm) == 0:
        ctx.log.info(f"[Stage 1c] {pid}: no sources detected")
        return None, None, None, None
    err = calc_total_error(det["sub"], det["rms"] * corr, effective_gain=max(p["exptime_detection"], 1.0))
    cat = SourceCatalog(det["sub"], segm, convolved_data=conv, error=err)
    x, y = values(cat, "x_centroid"), values(cat, "y_centroid")
    kron = values(cat, "kron_flux")
    kerr = values(cat, "kron_flux_err")
    segf = values(cat, "segment_flux")
    flux = np.where(np.isfinite(kron) & (kron > 0), kron, segf)
    ferr = np.where(np.isfinite(kerr) & (kerr > 0), kerr, values(cat, "segment_flux_err"))
    ra, dec = det["wcs"].pixel_to_world_values(x, y)
    tab = pd.DataFrame({
        "label": np.asarray(cat.labels, int), "x": x, "y": y, "ra": ra, "dec": dec,
        "area_px": values(cat, "area"), "semimajor_px": values(cat, "semimajor_axis"),
        "semiminor_px": values(cat, "semiminor_axis"), "ellipticity": values(cat, "ellipticity"),
        "orientation_deg": values(cat, "orientation"), "fwhm_px": values(cat, "fwhm"),
        "flux": flux, "flux_err": ferr, "snr": flux / np.where(ferr > 0, ferr, np.nan)})
    tab = tab[np.isfinite(tab.x) & np.isfinite(tab.ra) & (tab.flux > 0)]
    tab["saturated"] = saturated_cores(det["sub"], det["rms"], tab.x.to_numpy(), tab.y.to_numpy())

    # ---- 1. register every band to the detection band (fails on a residual offset) ----
    registration = register_bands(ctx, pid, imgs, det_b, dcfg)

    # ---- 2. every segment measured in every band -> cosmic-ray candidates (no second band confirms) ----
    lab_img = segm.data
    max_lab = int(lab_img.max())
    li = tab.label.to_numpy()
    confirm = float(dcfg.get("confirm_snr", 3.0))
    # allowed centroid shift: 1.5 px, or a fraction of the source's equivalent radius for large galaxies
    # (colour gradients move a big galaxy's centroid between bands; cosmic-ray segments are small)
    max_shift = np.maximum(float(dcfg.get("confirm_max_shift_px", 1.5)),
                           float(dcfg.get("confirm_max_shift_frac", 0.5)) * np.sqrt(tab.area_px.to_numpy() / np.pi))
    best_other = np.full(len(tab), np.nan)
    n_cover_other = np.zeros(len(tab), int)
    for b, im in imgs.items():                       # cosmic-ray-free copies for the centroid test
        im["med"], im["med_noise"] = median_image(im["sub"], im["val"], 3)
    phot = {b: segment_photometry(lab_img, det["wcs"], imgs.get(b), det["pixarea"], max_lab, corr) for b in bands}
    dcx, dcy = phot[det_b][3][li], phot[det_b][4][li]
    for b in bands:
        fl, er, cov, cx, cy = phot[b]
        tab[f"segflux_{b}"] = fl[li]
        tab[f"segsnr_{b}"] = (fl / er)[li]
        tab[f"segcov_{b}"] = cov[li]
        tab[f"segshift_{b}"] = np.hypot(cx[li] - dcx, cy[li] - dcy)       # centroid shift vs detection band [px]
        if b != det_b and b in imgs:
            covered = cov[li] >= 0.9
            n_cover_other += covered
            # a band confirms only with light at the SAME place: S/N counts only if the centroid agrees
            snr_here = np.where(tab[f"segshift_{b}"].to_numpy() <= max_shift, (fl / er)[li], np.nan)
            best_other = np.fmax(best_other, np.where(covered, snr_here, np.nan))
    tab["confirm_snr_best"] = best_other
    # True = no other covering band sees it; False = confirmed; None = no other band covers it (unknown)
    use_cr = str(dcfg.get("cr_rejection", "two_band")).lower() != "none"
    tab["cr_candidate"] = [None if (not use_cr or n == 0) else bool(not (np.isfinite(s) and s >= confirm))
                           for n, s in zip(n_cover_other, best_other)]
    is_cr = tab.cr_candidate.map(lambda v: v is True).astype(bool)
    ctx.log.info(f"[Stage 1c] {pid}: {len(tab)} detections | confirmed by another band {int((~is_cr & (n_cover_other > 0)).sum())} "
                 f"| cosmic-ray candidates {int(is_cr.sum())} | no other band covers {int((n_cover_other == 0).sum())}")

    # ---- 3. caps: real (confirmed / unknown) sources and cosmic-ray candidates separately ----
    cap, cap_cr = int(dcfg["max_sources_per_pointing"]), int(dcfg.get("max_cr_candidates_per_pointing", 1000))
    real, crs = tab[~is_cr], tab[is_cr]
    if len(real) > cap:
        ctx.log.info(f"[Stage 1c] {pid}: {len(real)} confirmed sources, keeping the brightest {cap} "
                     "(detection.max_sources_per_pointing)")
        real = real.nlargest(cap, "flux")
    if len(crs) > cap_cr:
        ctx.log.info(f"[Stage 1c] {pid}: {len(crs)} cosmic-ray candidates, keeping the brightest {cap_cr} "
                     "(detection.max_cr_candidates_per_pointing; they stay in the catalogue as Noise)")
        crs = crs.nlargest(cap_cr, "flux")
    tab = pd.concat([real, crs]).sort_values("label").reset_index(drop=True)

    # ---- 4. aligned stamps; cosmic-ray pixels replaced and marked 3 ----
    seg_f = lab_img.astype(np.float32)          # converted once, sampled per source
    scale = det["scale"] if dcfg["cutout_pixscale_arcsec"] == "auto" else float(dcfg["cutout_pixscale_arcsec"])
    cubes = np.zeros((len(tab), len(bands), S, S), np.float16)
    segs = np.zeros((len(tab), S, S), np.uint8)
    yy, xx = np.mgrid[:S, :S]
    core = np.hypot(yy - (S - 1) / 2, xx - (S - 1) / 2) <= dcfg["core_radius_px"]
    grow = int(dcfg.get("cr_mask_grow_px", 1))
    mag_snr = float(dcfg.get("mag_min_snr", 2.0))
    rows = []
    for i, r in enumerate(tab.itertuples()):
        vstamps, n_nb, valid_det = {}, 0, None
        raw = np.zeros((len(bands), S, S), np.float32)
        sig = np.full((len(bands), S, S), np.inf, np.float32)
        val = np.zeros((len(bands), S, S), bool)
        for bi, b in enumerate(bands):
            if b not in imgs:
                continue
            im = imgs[b]
            M, off, area_ratio = affine_for(im["wcs"], r.ra, r.dec, S, scale)
            raw[bi] = make_stamp(im["sub"], M, off, S) * area_ratio
            vstamps[b] = val[bi] = make_stamp(im["valf"], M, off, S, order=0) > 0.5
            bx, by = im["wcs"].world_to_pixel_values(r.ra, r.dec)
            yi = int(np.clip(round(float(by)), 0, im["rms"].shape[0] - 1))
            xi = int(np.clip(round(float(bx)), 0, im["rms"].shape[1] - 1))
            rms_px = float(im["rms"][yi, xi]) * area_ratio * corr
            sig[bi] = np.sqrt(rms_px ** 2 + np.clip(raw[bi], 0, None) * area_ratio / im["exptime"])
            if b == det_b:
                lab = make_stamp(seg_f, M, off, S, order=0).astype(np.int64)
                segs[i][(lab > 0) & (lab != r.label)] = 2
                segs[i][lab == r.label] = 1
                n_nb = len(set(np.unique(lab).tolist()) - {0, r.label})
                valid_det = vstamps[b]
        target = target0 = segs[i] == 1           # the full target footprint, before the cosmic-ray mask
        sflux = np.array([getattr(r, f"segflux_{b}") for b in bands], float)
        ssnr = np.array([getattr(r, f"segsnr_{b}") for b in bands], float)
        n_cr_target = 0
        if use_cr and len(imgs) > 1:
            crm, repl = cr_pixel_mask(raw, val, segs[i], sig, sflux, ssnr, float(dcfg.get("cr_pixel_nsigma", 5.0)),
                                      float(dcfg.get("cr_pixel_ratio", 4.0)))
            raw = np.where(crm, repl, raw)
            anyc = crm.any(axis=0)
            if anyc.any():
                if grow > 0:                      # PSF matching (Stage 1d) spreads a residual a little
                    anyc = ndimage.binary_dilation(anyc, iterations=grow)
                anyc &= segs[i] != 2
                n_cr_target = int((anyc & target).sum())
                segs[i][anyc] = 3
        cubes[i] = raw.astype(np.float16)
        target = segs[i] == 1                     # target pixels left after the cosmic-ray mask
        row = {"valid_frac": float(valid_det.mean()), "core_ok": bool(valid_det[core].all()), "n_neighbours": n_nb,
               "cr_pixels_masked": n_cr_target}
        for bi, b in enumerate(bands):
            covered = b in imgs and bool(vstamps[b][core].all())       # this band has data on the source core
            row[f"covered_{b}"] = covered
            fl = float(raw[bi][target0 & vstamps[b]].astype(np.float64).sum()) \
                if covered else np.nan                    # whole footprint, cosmic-ray pixels replaced
            row[f"flux_{b}"] = fl
            ok = b in imgs and fl > 0 and np.isfinite(imgs[b]["zp"])
            ok = ok and np.nan_to_num(getattr(r, f"segsnr_{b}"), nan=0.0) >= mag_snr   # not detected: no magnitude
            row[f"mag_{b}"] = -2.5 * np.log10(fl) + imgs[b]["zp"] if ok else np.nan
            e = getattr(r, f"segflux_{b}") / getattr(r, f"segsnr_{b}") if np.isfinite(getattr(r, f"segsnr_{b}")) else np.nan
            row[f"magerr_{b}"] = 1.0857 * e / fl if ok and np.isfinite(e) else np.nan
        rows.append(row)
    tab = pd.concat([tab, pd.DataFrame(rows)], axis=1)
    tab["object_id"] = (pidx + 1) * 100000 + tab["label"]
    tab["pointing"] = pid
    tab["role"] = p["role"]
    tab["detection_band"] = det_b
    tab["pixscale_out"] = scale
    tab["exptime"] = p["exptime_detection"]
    tab["mag"] = -2.5 * np.log10(tab["flux"]) + det["zp"] if np.isfinite(det["zp"]) else -2.5 * np.log10(tab["flux"])
    # statistical centroid error, sigma = source size / SNR per axis (the absolute WCS error is not included)
    pos_err = tab.fwhm_px / 2.3548 / tab.snr * det["scale"]
    tab["ra_err"] = pos_err.round(5)
    tab["dec_err"] = pos_err.round(5)
    ctx_img = load_ctx(det["file"])
    if ctx_img is not None:
        xi = np.clip(np.round(tab.x).astype(int), 0, ctx_img.shape[1] - 1)
        yi = np.clip(np.round(tab.y).astype(int), 0, ctx_img.shape[0] - 1)
        tab["n_exposures"] = popcount(ctx_img[yi, xi])
    else:
        tab["n_exposures"] = np.nan
    tab["edge_flag"] = (tab["valid_frac"] < dcfg["min_valid_fraction"]) | (~tab["core_ok"])
    bad = np.zeros(len(tab), bool)
    bad_bands = [[] for _ in range(len(tab))]
    for b, im in imgs.items():                 # a source is QC-flagged if ANY band's tile is flagged
        bx, by = im["wcs"].world_to_pixel_values(tab.ra.to_numpy(), tab.dec.to_numpy())
        hit = in_flagged(bx, by, qc_boxes.get(im["file"]))
        bad |= hit
        for j in np.where(hit)[0]:
            bad_bands[j].append(b)
    tab["frame_quality_flag"] = (~bad).astype(int)          # 1 = clean, 0 = in a flagged tile
    tab["qc_flagged_bands"] = [",".join(v) for v in bad_bands]
    tab["bands_missing"] = ",".join(p["missing_bands"])
    split = rules.assign_many(tab.ra, tab.dec)
    if p["role"] == "external":
        split = ["external" if s == "external" else None for s in split]
    tab["split_tag"] = [s if s is not None else "buffer" for s in split]     # buffer = inside a collar
    ctx.log.info(f"[Stage 1c] {pid}: cosmic-ray pixels masked in {int((tab.cr_pixels_masked > 0).sum())} of "
                 f"{len(tab)} target footprints (median {tab.cr_pixels_masked[tab.cr_pixels_masked > 0].median() if (tab.cr_pixels_masked > 0).any() else 0} px)")
    _preview(ctx, pid, det, tab, cubes, bands)
    return tab, cubes, segs, registration


def run(ctx: Ctx):
    out = ctx.stage_dir("1c")
    info = load_json(ctx.require("1a", "pointings.json"))
    bands, pts = info["bands"], info["pointings"]
    rules = SkyRegionPartitioner.from_file(ctx.require("0", "split_rule.json"))
    qc = flagged_tiles(ctx)
    parts, registration = [], {}
    for pidx, (pid, p) in enumerate(pts.items()):
        ctx.log.info(f"[Stage 1c] {pid}: detecting on {p['detection_band']} ...")
        tab, cubes, segs, reg = process_pointing(ctx, pid, p, bands, rules, qc, pidx)
        if tab is None:
            continue
        registration[pid] = reg
        tmp = out / f"_tmp_{pid}"
        np.save(f"{tmp}_cut.npy", cubes)
        np.save(f"{tmp}_seg.npy", segs)
        parts.append((tab, tmp))
        ctx.log.info(f"[Stage 1c] {pid}: {len(tab)} sources | median SNR {tab.snr.median():.1f} | "
                     f"edge-flagged {int(tab.edge_flag.sum())} | QC-flagged {int((tab.frame_quality_flag == 0).sum())} | "
                     f"splits {tab.split_tag.value_counts(dropna=False).to_dict()}")
    if not parts:
        raise RuntimeError("No sources detected in any pointing.")

    # ---- merge pointings into single arrays ----
    n = sum(len(t) for t, _ in parts)
    S = ctx.cfg["detection"]["cutout_size"]
    cut = np.lib.format.open_memmap(out / "cutouts_raw.npy", mode="w+", dtype=np.float16, shape=(n, len(bands), S, S))
    seg = np.lib.format.open_memmap(out / "segstamps.npy", mode="w+", dtype=np.uint8, shape=(n, S, S))
    k, tabs = 0, []
    for tab, tmp in parts:
        c, s = np.load(f"{tmp}_cut.npy"), np.load(f"{tmp}_seg.npy")
        cut[k:k + len(tab)], seg[k:k + len(tab)] = c, s
        tab["cutout_index"] = np.arange(k, k + len(tab))
        tabs.append(tab)
        k += len(tab)
        Path(f"{tmp}_cut.npy").unlink()
        Path(f"{tmp}_seg.npy").unlink()
    cut.flush()
    seg.flush()
    sources = pd.concat(tabs, ignore_index=True)
    bl = list(bands)
    sources["colour_1"] = sources[f"mag_{bl[0]}"] - sources[f"mag_{bl[1]}"] if len(bl) > 1 else np.nan
    sources["colour_2"] = sources[f"mag_{bl[1]}"] - sources[f"mag_{bl[2]}"] if len(bl) > 2 else np.nan
    # band consistency: weakest other-band flux relative to the detection band. A real source
    # shows up in every filter; a cosmic ray or hot pixel in a single exposure stack does not.
    det_flux = sources.apply(lambda r: r[f"flux_{r.detection_band}"], axis=1)
    # only bands that COVER the source count: outside the overlap of the bands' footprints a missing band
    # is missing data, not evidence of a single-band artefact
    ratios = [(sources[f"flux_{b}"] / det_flux).where((sources.detection_band != b) & sources[f"covered_{b}"])
              for b in bl]
    sources["band_consistency"] = np.clip(pd.concat(ratios, axis=1).min(axis=1), 0, 2) if len(bl) > 1 else np.nan
    front = ["object_id", "pointing", "role", "split_tag", "ra", "dec", "x", "y", "mag", "snr", "flux", "flux_err"]
    sources = sources[front + [c for c in sources.columns if c not in front]]
    sources.to_csv(out / "sources.csv", index=False)
    save_json(registration, out / "registration.json")
    save_json({"bands": bands, "cutout_size": S, "n_sources": n,
               "split_counts": sources["split_tag"].value_counts(dropna=False).to_dict()}, out / "summary.json")
    ctx.log.info(f"[Stage 1c] {n} sources total -> sources.csv, cutouts_raw.npy {cut.shape}")
    return {"n_sources": n}


def _preview(ctx, pid, det, tab, cubes, bands):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from ..utils import asinh_normalise
    out = ctx.stage_dir("1c")
    step = max(1, max(det["sub"].shape) // 1024)
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(asinh_normalise(det["sub"][::step, ::step]), origin="lower", cmap="gray")
    sc = ax.scatter(tab.x / step, tab.y / step, s=8, c=np.log10(tab.snr.clip(1)), cmap="viridis",
                    edgecolors="none")
    fig.colorbar(sc, ax=ax, fraction=0.046, label="log10 SNR")
    ax.set_title(f"Stage 1c {pid}: {len(tab)} detections on {tab.detection_band.iloc[0]}")
    ax.set_xticks([])
    ax.set_yticks([])
    fig.tight_layout()
    fig.savefig(out / f"{pid}_detections.png", dpi=100)
    plt.close(fig)
    # 16 brightest stamps as colour composites (red = reddest band)
    pick = tab.sort_values("flux", ascending=False).index[:16]
    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    for ax, i in zip(axes.ravel(), pick):
        cube = cubes[i].astype(np.float32)
        rgb = np.stack([asinh_normalise(cube[min(j, len(bands) - 1)]) for j in (2, 1, 0)], -1)
        ax.imshow(rgb, origin="lower")
        ax.set_title(f"id {tab.loc[i, 'label']}  SNR {tab.loc[i, 'snr']:.0f}", fontsize=7)
        ax.axis("off")
    for ax in axes.ravel()[len(pick):]:
        ax.axis("off")
    fig.suptitle(f"{pid}: brightest cutouts (aligned, {len(bands)} bands)")
    fig.tight_layout()
    fig.savefig(out / f"{pid}_cutouts.png", dpi=100)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage1d_psf_match.py
"""Stage 1d - Cross-filter PSF homogenisation.

Purpose : make every band of a cutout share one resolution, so chromatic PSF
          differences are not mistaken for galaxy structure.
Method  : per pointing, bright, isolated, unsaturated, compact sources
          (SNR >= 50, ellipticity < 0.2, FWHM near the stellar minimum) are
          recentred and median-stacked into a quick PSF per band. The band's PSF
          FWHM is the MEDIAN of the stars' individual 2-D Gaussian fits: stacking
          undersampled single-exposure stars (sub-pixel shifts) blurs the stack by
          ~0.5 px. With fewer
          than `min_stars`, a Gaussian of the configured instrument FWHM is
          used instead (flagged in psf_match.json); with NO star at all the stage
          stops (psf_match.require_stars), because a configured width is not a
          measurement. Cosmic-ray candidates (Stage 1c) are never PSF stars.
          The broadest band is the
          target; every sharper band is convolved with a matching kernel K
          (PSF_sharp * K ~ PSF_broad): by default a Gaussian with
          sigma_K^2 = sigma_broad^2 - sigma_sharp^2 from 2-D Gaussian fits
          (robust for noisy stacks); 'photutils' uses create_matching_kernel.
          The Pass-1 placeholder PSF (used by Stage 1e) is a Gaussian with the
          target FWHM - the high-quality empirical PSF comes later (Stage 5a).
Output  : cutouts.npy (PSF-matched, used by all later stages),
          psf_match.json, psf_<pointing>_<band>.npy, placeholder_psf_<pointing>.npy
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import ndimage
from scipy.signal import fftconvolve

from ..context import Ctx, load_json, save_json
from ..utils import as_bool, fwhm_fit, gaussian_psf

try:
    from photutils.psf.matching import TopHatWindow, create_matching_kernel
    _HAVE_MATCH = True
except Exception:
    _HAVE_MATCH = False


def crop_center(img, n):
    c = img.shape[0] // 2
    h = n // 2
    return img[c - h:c + h + 1, c - h:c + h + 1]


def core_centroid(stamp, half=3):
    """Flux centroid of the central (2*half+1)^2 pixels only: a centroid over the whole stamp is pulled
    by neighbours and cosmic-ray remnants (by up to 5 px in a single exposure)."""
    s = np.clip(np.asarray(stamp, float), 0, None)
    c = s.shape[0] // 2
    box = s[c - half:c + half + 1, c - half:c + half + 1]
    if box.sum() <= 0:
        return float(c), float(c)
    cy, cx = ndimage.center_of_mass(box)
    return float(cy + c - half), float(cx + c - half)


def stack_stars(stamps, centers=None, masks=None):
    """Sub-pixel recentre each star (cubic shift) on its core centroid, normalise, median.
    centers : (cy, cx) per star - pass the DETECTION-band centroids so every band is shifted identically
    masks   : True = pixel of another source or a cosmic ray, left out of the median."""
    n = stamps[0].shape[0]
    c = (n - 1) / 2.0
    out = []
    for k, s in enumerate(stamps):
        s = np.asarray(s, float)
        cy, cx = centers[k] if centers is not None else core_centroid(s)
        bad = np.asarray(masks[k], bool) if masks is not None else np.zeros_like(s, bool)
        s = ndimage.shift(np.where(bad, 0.0, s), (c - cy, c - cx), order=3, mode="constant")
        bad = ndimage.shift(bad.astype(float), (c - cy, c - cx), order=0, mode="constant", cval=1.0) > 0.5
        tot = s[~bad].sum()
        if tot > 0:
            out.append(np.where(bad, np.nan, s / tot))
    psf = np.nan_to_num(np.clip(np.nanmedian(out, axis=0), 0, None))
    return psf / psf.sum()


def star_masks(segs, idx, n):
    """Pixels of OTHER sources (2) in each star stamp, cropped like the stamps. Cosmic-ray pixels (3) are not
    masked here: Stage 1c already replaced them, in the band they hit, by what the other bands predict; the
    mask value 3 is the union over all bands and would punch holes into the star in every band."""
    return [crop_center(np.asarray(segs[i]), n) == 2 for i in idx]


def flat_top(stamp, tol=0.02):
    top = np.sort(np.asarray(stamp, float).ravel())[-5:]
    return (top[-1] - top[0]) / max(abs(top[-1]), 1e-12) < tol


def matching_kernel(psf_sharp, psf_broad, beta, method="gaussian", fwhm_sharp=None, fwhm_broad=None):
    """gaussian : K = Gaussian with sigma_K^2 = sigma_broad^2 - sigma_sharp^2 (robust for
                  noisy, near-Gaussian stacked PSFs - the default)
    photutils  : Fourier-ratio kernel with a TopHat window (exact for clean PSFs)."""
    if method == "gaussian":
        fs = fwhm_sharp if fwhm_sharp is not None else fwhm_fit(psf_sharp)
        fb = fwhm_broad if fwhm_broad is not None else fwhm_fit(psf_broad)
        fk = np.sqrt(max(fb ** 2 - fs ** 2, 0.0))
        if fk < 0.3:
            k = np.zeros_like(psf_sharp)
            k[k.shape[0] // 2, k.shape[1] // 2] = 1.0
            return k
        return gaussian_psf(fk, psf_sharp.shape[0])
    if _HAVE_MATCH:
        return create_matching_kernel(psf_sharp, psf_broad, window=TopHatWindow(beta))
    S = np.fft.fft2(np.fft.ifftshift(psf_sharp))
    B = np.fft.fft2(np.fft.ifftshift(psf_broad))
    k = np.fft.fftshift(np.real(np.fft.ifft2(B * np.conj(S) / (np.abs(S) ** 2 + 1e-3))))
    return k / k.sum()


def convolve_stamp(stamp, K):
    pad = K.shape[0] // 2
    padded = np.pad(stamp, pad, mode="reflect")
    return fftconvolve(padded, K, mode="same")[pad:-pad, pad:-pad]


def gaia_flags(ctx, src):
    """True for sources within 0.5" of a Gaia DR3 star (the catalogue in data/catalogs), else False."""
    try:
        from ..catalogs import load_gaia
        from ..utils import sky_crossmatch
        g = load_gaia(ctx, list(load_json(ctx.require("1a", "manifest.json")).values()))
        if g is None or g.empty:
            return np.zeros(len(src), bool)
        return sky_crossmatch(src.ra, src.dec, g.ra, g.dec, ctx.cfg["semantic_audit"]["gaia_radius_arcsec"])[2]
    except Exception:
        return np.zeros(len(src), bool)


def select_stars(src: pd.DataFrame, segs, raw, bands, present, cfg, expected_px, gaia=None):
    """Bright, round, isolated, compact sources detected in every band that COVERS them and in at least
    one band besides the detection band (a cosmic ray is as compact as a star but appears in one band only)."""
    bright = src[(src.snr >= cfg["star_min_snr"]) & (~src.edge_flag.astype(bool))
                 & (src.ellipticity < cfg.get("star_max_ellipticity", 0.3))]
    if "cr_candidate" in bright:                # never a cosmic ray (no second band confirms it)
        bright = bright[~as_bool(bright.cr_candidate)]
    if "saturated" in bright:
        bright = bright[~as_bool(bright.saturated)]
    det = src.detection_band.iloc[0] if len(src) else None
    others = [b for b in present if b != det]
    seen_in_other = np.zeros(len(bright), bool)
    for b in others:
        cov = bright[f"covered_{b}"].astype(bool) if f"covered_{b}" in bright else pd.Series(True, index=bright.index)
        ok_b = bright[f"flux_{b}"] > 0.2 * bright[f"flux_{det}"]
        keep = ~cov | ok_b                      # detected in every band that COVERS it ...
        seen_in_other = seen_in_other[keep.to_numpy()] | ok_b[keep].to_numpy()
        bright = bright[keep]
    if others:
        bright = bright[seen_in_other]          # ... and in at least one other band (cosmic rays are single-band)
    if len(bright) == 0:
        return bright, np.nan
    # size from a 2-D Gaussian fit of the detection-band stamp (neighbours / cosmic-ray pixels removed): the
    # segment-moment FWHM depends on the isophote and lets compact galaxies in
    det_i = bands.index(det)
    size = []
    for i in bright.cutout_index:
        img = np.where(np.asarray(segs[i]) == 2, 0.0, np.asarray(raw[i, det_i], np.float32))
        size.append(fwhm_fit(crop_center(img, cfg["stamp"])))
    bright = bright.assign(fit_fwhm_px=size)
    # the stellar locus: the median size of the Gaia stars among them when there are >= 2, otherwise the
    # PEAK of the size distribution inside a window around the expected PSF width (instrument fallback FWHM).
    # Chance two-band cosmic-ray coincidences are sharper than the PSF and compact galaxies broader, so a
    # 'most compact 30 %' rule is pulled off the stars by either.
    g = bright.index.isin(src.index[gaia]) if gaia is not None else np.zeros(len(bright), bool)
    if g.sum() >= 2:
        locus = float(np.median(bright.fit_fwhm_px[g]))
    else:
        lo, hi = 0.85 * expected_px, 1.5 * expected_px
        inwin = bright.fit_fwhm_px.between(lo, hi)
        if inwin.sum() < 3:
            return bright.iloc[:0], np.nan
        h, e = np.histogram(bright.fit_fwhm_px[inwin], bins=np.arange(lo, hi + 0.1, 0.1))
        h = np.convolve(h, np.ones(3) / 3, mode="same")
        locus = float(0.5 * (e[np.argmax(h)] + e[np.argmax(h) + 1]))
    cand = bright[np.abs(bright.fit_fwhm_px - locus) <= 0.15 * locus]
    S = segs.shape[1]
    yy, xx = np.mgrid[:S, :S]
    # only the CORE must be free of other segments: neighbour / cosmic-ray pixels further out are left out of
    # the stack (star_masks); a 12-px rule rejected most stars in a cosmic-ray-dense single exposure
    near = np.hypot(yy - (S - 1) / 2, xx - (S - 1) / 2) <= cfg.get("star_isolation_px", 5)
    iso = [not np.any(segs[i][near] == 2) for i in cand.cutout_index]
    cand = cand[np.asarray(iso, bool)]
    return cand.nlargest(cfg["max_stars"], "snr"), float(locus)


def run(ctx: Ctx):
    cfg = ctx.cfg["psf_match"]
    out = ctx.stage_dir("1d")
    info = load_json(ctx.require("1a", "pointings.json"))
    bands, pts = info["bands"], info["pointings"]
    src = pd.read_csv(ctx.require("1c", "sources.csv"))
    raw = np.load(ctx.require("1c", "cutouts_raw.npy"), mmap_mode="r")
    segs = np.load(ctx.require("1c", "segstamps.npy"), mmap_mode="r")
    matched = np.lib.format.open_memmap(out / "cutouts.npy", mode="w+", dtype=np.float16, shape=raw.shape)
    report = {}
    for pid, p in pts.items():
        rows = src[src.pointing == pid]
        if rows.empty:
            continue
        fb = cfg["fallback_fwhm_arcsec"]
        expected = fb.get(p["instrument"], fb.get(p["instrument"].split("_")[0], fb["default"])) / float(rows.pixscale_out.iloc[0])
        stars, locus = select_stars(rows, segs, raw, bands, list(p["bands"]), cfg, expected, gaia_flags(ctx, rows))
        det_i = bands.index(p["detection_band"])
        stars = stars[np.array([not flat_top(raw[i, det_i]) for i in stars.cutout_index], dtype=bool)]   # rows, also when empty
        if len(stars):
            locus = float(np.median(stars.fwhm_px))
        elif cfg.get("require_stars", True):
            raise RuntimeError(
                f"[Stage 1d] {pid}: no PSF star passed the cuts (SNR >= {cfg['star_min_snr']}, round, isolated, "
                "unsaturated, confirmed in a second band). The PSFs would be the configured Gaussian, not a "
                "measurement. Check the Stage 1c registration / cosmic-ray log lines; to continue with the "
                "Gaussian fallback anyway set psf_match.require_stars: false (the catalogue then stores no "
                "measured matched FWHM).")
        scale = float(rows.pixscale_out.iloc[0])
        psfs, method, star_fwhm = {}, {}, {}
        for bi, b in enumerate(bands):
            if b not in p["bands"]:
                continue
            in_b = stars[stars[f"covered_{b}"].astype(bool)] if f"covered_{b}" in stars else stars
            if f"segsnr_{b}" in in_b:              # a red star is too faint in a blue band to shape its PSF
                in_b = in_b[in_b[f"segsnr_{b}"].fillna(0) >= cfg.get("star_min_snr_per_band", 30)]
            if len(in_b) >= cfg["min_stars"]:
                idx = in_b.cutout_index.tolist()
                cen = [core_centroid(crop_center(raw[i, det_i].astype(np.float32), cfg["stamp"])) for i in idx]
                masks = star_masks(segs, idx, cfg["stamp"])
                crops = [crop_center(raw[i, bi].astype(np.float32), cfg["stamp"]) for i in idx]
                psfs[b] = stack_stars(crops, cen, masks)
                star_fwhm[b] = float(np.median([fwhm_fit(np.where(m, 0.0, c)) for c, m in zip(crops, masks)]))
                method[b] = f"stacked {len(in_b)} stars"
            else:
                fb = cfg["fallback_fwhm_arcsec"]
                fw = fb.get(p["instrument"], fb.get(p["instrument"].split("_")[0], fb["default"]))
                psfs[b] = gaussian_psf(fw / scale, cfg["stamp"])
                method[b] = f"Gaussian fallback ({fw}\" FWHM; only {len(in_b)} stars cover {b})"
            np.save(out / f"psf_{pid}_{b}.npy", psfs[b])
        fwhm = {b: star_fwhm.get(b, fwhm_fit(v)) for b, v in psfs.items()}
        med = float(np.median(list(fwhm.values())))
        for b in [b for b, f in fwhm.items() if f > 1.6 * med and method[b].startswith("stacked")]:
            ctx.log.info(f"[Stage 1d] {pid}: WARNING - the {b} PSF stack ({fwhm[b]:.2f} px) is > 1.6 x the median of the "
                         f"bands ({med:.2f} px): too few clean stars in {b}; the Gaussian fallback is used for it.")
            fb = cfg["fallback_fwhm_arcsec"]
            fw = fb.get(p["instrument"], fb.get(p["instrument"].split("_")[0], fb["default"]))
            psfs[b] = gaussian_psf(fw / scale, cfg["stamp"])
            np.save(out / f"psf_{pid}_{b}.npy", psfs[b])
            method[b] = f"Gaussian fallback ({fw}\" FWHM; the {b} stack was too broad)"
            fwhm[b] = fwhm_fit(psfs[b])
        target = max(fwhm, key=fwhm.get)
        kernels = {b: matching_kernel(psfs[b], psfs[target], cfg["window"], cfg.get("kernel", "gaussian"),
                                      fwhm[b], fwhm[target]) for b in psfs if b != target}
        # FWHM after matching. Gaussian kernel: widths add in quadrature, sqrt(f_b^2 + f_K^2) with
        # f_K = sqrt(f_target^2 - f_b^2) (no kernel if f_K < 0.3 px); 'photutils' kernel: fit the convolved stack
        def after_matching(b, K):
            if cfg.get("kernel", "gaussian") != "gaussian":
                return fwhm_fit(fftconvolve(psfs[b], K, mode="same"))
            fk = np.sqrt(max(fwhm[target] ** 2 - fwhm[b] ** 2, 0.0))
            return float(np.hypot(fwhm[b], fk)) if fk >= 0.3 else fwhm[b]
        check = {b: round(after_matching(b, K), 3) for b, K in kernels.items()}
        for i in rows.cutout_index:
            cube = raw[i].astype(np.float32)
            for bi, b in enumerate(bands):
                if b in kernels:
                    cube[bi] = convolve_stamp(cube[bi], kernels[b])
            matched[i] = cube.astype(np.float16)
        np.save(out / f"placeholder_psf_{pid}.npy", gaussian_psf(fwhm[target], cfg["stamp"]))
        if not np.isfinite(locus):          # no bright compact sources at all: use the PSF model width
            locus = fwhm[target]
        measured = method[target].startswith("stacked")
        if not measured:
            ctx.log.info(f"[Stage 1d] {pid}: WARNING - the target band {target} uses the Gaussian fallback; "
                         "no measured matched FWHM is stored.")
        report[pid] = {"n_stars": int(len(stars)), "star_ids": stars.object_id.tolist(), "method": method,
                       "target_psf_measured": bool(measured),
                       "matched_fwhm_arcsec": round(fwhm[target] * scale, 4) if measured else None,
                       "stellar_locus_fwhm_px": round(float(locus), 3),
                       "fwhm_px": {b: round(v, 3) for b, v in fwhm.items()}, "target_band": target,
                       "matched_fwhm_px": check, "pixscale_arcsec": scale}
        ctx.log.info(f"[Stage 1d] {pid}: stars={len(stars)} FWHM(px)={report[pid]['fwhm_px']} -> target {target}; "
                     f"after matching {check}")
    matched.flush()
    save_json(report, out / "psf_match.json")
    _plot(report, out)
    return {"pointings": len(report)}


def _plot(report, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    for pid in report:
        files = sorted(out.glob(f"psf_{pid}_*.npy"))
        if not files:
            continue
        fig, axes = plt.subplots(1, len(files), figsize=(3 * len(files), 3))
        for ax, f in zip(np.atleast_1d(axes), files):
            ax.imshow(np.log10(np.load(f) + 1e-5), origin="lower", cmap="magma")
            b = f.stem.split("_")[-1]
            ax.set_title(f"{b}: FWHM {report[pid]['fwhm_px'][b]:.2f} px", fontsize=9)
            ax.axis("off")
        fig.suptitle(f"Stage 1d {pid}: per-band PSFs (log)")
        fig.tight_layout()
        fig.savefig(out / f"{pid}_psfs.png", dpi=100)
        plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage1e_pass1_morphology.py
"""Stage 1e - Pass-1 photometry and morphology.

Purpose : a provisional table of physical parameters for every source.
Method  : statmorph on the sum of the PSF-matched bands, with the target
          segment from Stage 1c and every neighbouring segment masked. The
          Sersic fit is convolved with a generic PLACEHOLDER PSF (Gaussian with
          the Stage-1d target FWHM); Stage 5b re-runs the galaxy track with the
          empirical PSF. Sources below morphology.min_snr are not measured
          (NaN, flag 4). Gain = detection-band EXPTIME (images are in e-/s).
          fwhm_over_psf = source FWHM / PSF FWHM (1 = point-like), both from a
          2-D Gaussian fit to the detection-band cutout (neighbours removed,
          cosmic-ray pixels already replaced by Stage 1c) and to the Stage-1d PSF of that band. A fit
          does not shrink with S/N the way a segment-moment FWHM does (faint
          galaxies only show their core above the threshold and looked
          point-like or even sharper than the PSF).
Output  : features_pass1.csv  (psf_pass = 1)
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..context import Ctx, load_json
from ..morphology import FEATURES, _HAVE_STATMORPH, run_parallel
from ..utils import fwhm_fit
from .stage1d_psf_match import crop_center


def fit_fwhm_ratio(ctx, src):
    """Gaussian-fit FWHM of every source / of its pointing's detection-band PSF (Stage 1d)."""
    raw = np.load(ctx.require("1c", "cutouts_raw.npy"), mmap_mode="r")
    seg = np.load(ctx.require("1c", "segstamps.npy"), mmap_mode="r")
    bands = load_json(ctx.require("1a", "pointings.json"))["bands"]
    stamp = ctx.cfg["psf_match"]["stamp"]
    locus = {}
    rep = load_json(ctx.require("1d", "psf_match.json"))
    for pid, det in src.groupby("pointing").detection_band.first().items():
        locus[pid] = rep[pid]["fwhm_px"].get(det) or fwhm_fit(np.load(ctx.require("1d", f"psf_{pid}_{det}.npy")))
    out = np.full(len(src), np.nan)
    for k, (i, p, det) in enumerate(zip(src.cutout_index, src.pointing, src.detection_band)):
        img = np.asarray(raw[int(i), bands.index(det)], np.float32)
        img = np.where(np.asarray(seg[int(i)]) == 2, 0.0, img)    # neighbours (cosmic-ray pixels: already replaced)
        out[k] = fwhm_fit(crop_center(img, stamp)) / locus[p]
    return out, locus


def run(ctx: Ctx):
    out = ctx.stage_dir("1e")
    src = pd.read_csv(ctx.require("1c", "sources.csv"))
    cut_path = ctx.require("1d", "cutouts.npy")
    seg_path = ctx.require("1c", "segstamps.npy")
    min_snr = ctx.cfg["morphology"]["min_snr"]
    todo = src[src.snr >= min_snr]
    psf = {pid: np.load(ctx.require("1d", f"placeholder_psf_{pid}.npy")) for pid in todo.pointing.unique()}
    ctx.log.info(f"[Stage 1e] measuring {len(todo)} of {len(src)} sources (SNR >= {min_snr}) with "
                 f"{'statmorph' if _HAVE_STATMORPH else 'NumPy fallback'} on {ctx.n_jobs} worker(s) ...")
    res = run_parallel(ctx, cut_path, seg_path, todo.cutout_index.tolist(), todo.exptime.tolist(),
                       [psf[p] for p in todo.pointing], mode="psf_fit")
    feats = pd.DataFrame(res, index=todo.index)
    table = pd.DataFrame(index=src.index, columns=FEATURES, dtype=float)
    table.loc[feats.index, FEATURES] = feats[FEATURES].to_numpy()
    table["flag"] = table["flag"].fillna(4)
    table.insert(0, "object_id", src.object_id)
    table["ellipticity"] = src["ellipticity"]           # from the Stage-1c segment moments
    # size relative to the PSF, measured with the SAME estimator (2-D Gaussian fit) on source and PSF
    ratio, locus = fit_fwhm_ratio(ctx, src)
    table["fwhm_over_psf"] = np.round(ratio, 4)
    ctx.log.info(f"[Stage 1e] PSF FWHM (Gaussian fit, detection band) per pointing: "
                 f"{ {p: round(v, 3) for p, v in locus.items()} } px | fwhm_over_psf median {np.nanmedian(ratio):.2f}, "
                 f"share point-like (0.8-1.2) {np.mean(np.abs(ratio - 1) <= 0.2):.1%}")
    table["psf_pass"] = 1
    table["morph_engine"] = "statmorph" if _HAVE_STATMORPH else "numpy_fallback"
    table.to_csv(out / "features_pass1.csv", index=False)
    good = table[table.flag <= 1]
    ctx.log.info(f"[Stage 1e] flags: {table.flag.value_counts().sort_index().to_dict()} | "
                 f"median Gini {good.gini.median():.3f}, C {good.concentration.median():.2f}, "
                 f"A {good.asymmetry.median():.3f}, n {good.sersic_n.median():.2f}")
    _plot(table, out)
    return {"measured": int(len(todo)), "good_flag": int(len(good))}


def _plot(t, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    g = t[t.flag <= 1]
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].scatter(g.m20, g.gini, s=4, alpha=0.5)
    xs = np.array([-3.0, 0.0])
    ax[0].plot(xs, -0.14 * xs + 0.33, "k--", lw=1, label="merger line (Lotz+08)")
    ax[0].plot(xs, 0.14 * xs + 0.80, "k:", lw=1, label="E/S0 vs Sb line")
    ax[0].invert_xaxis()
    ax[0].set_xlabel("M20")
    ax[0].set_ylabel("Gini")
    ax[0].legend(fontsize=7)
    ax[1].scatter(g.asymmetry, g.concentration, s=4, alpha=0.5)
    ax[1].set_xlabel("asymmetry A")
    ax[1].set_ylabel("concentration C")
    fig.suptitle("Stage 1e: Pass-1 morphology (flag <= 1)")
    fig.tight_layout()
    fig.savefig(out / "pass1_morphology.png", dpi=110)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage2a_representation.py
"""Stage 2a - Representation learning and feature fusion.

Purpose : one feature space that captures both pixel structure and physics.
Method  : a convolutional autoencoder (one input channel per band) compresses
          each asinh-scaled 128x128 cutout to a 64-d latent; PCA keeps the
          leading components (up to n_pca, dropping any below pca_min_variance;
          scaled as one block, not whitened). The classical Pass-1 features are median-imputed,
          signed-log transformed and RobustScaled. The two blocks are weighted
          (spatial_weight / 1 - spatial_weight) and concatenated.
          CAE, PCA, imputer and scaler are fitted on the TRAIN split only and
          applied with transform() to everything else (no leakage).
Output  : rep_cae.pt, latent.npy, hybrid_features.npy (row = sources.csv row), pca_scree.png,
          fusion.joblib, recon_error.npy
"""
from __future__ import annotations

import joblib
import numpy as np
import torch
from sklearn.decomposition import PCA

from ..context import Ctx, load_objects, save_json
from ..features import TabularPrep, load_cutouts
from ..nets import ConvAutoencoder, ae_errors_and_latents, train_autoencoder
from ..utils import asinh_stack


def encode_all(model, cutouts, device, batch=256):
    errs, zs = [], []
    for i in range(0, len(cutouts), batch):
        X = asinh_stack(np.asarray(cutouts[i:i + batch], np.float32))
        e, z = ae_errors_and_latents(model, X, device)
        errs.append(e)
        zs.append(z)
    return np.concatenate(errs), np.concatenate(zs)


def run(ctx: Ctx):
    cfg = ctx.cfg["representation"]
    out = ctx.stage_dir("2a")
    df = load_objects(ctx, ["sources", "pass1"])
    cut = load_cutouts(ctx)
    train = df[df.split_tag == "train"]
    if len(train) < 20:
        raise RuntimeError(f"Only {len(train)} train-split sources - check Stage 0 regions / collar size.")
    device = ctx.device()
    X = asinh_stack(np.asarray(cut[train.cutout_index.to_numpy()], np.float32))
    ctx.log.info(f"[Stage 2a] training representation CAE ({cut.shape[1]} bands -> {cfg['latent_dim']}-d) "
                 f"on {len(X)} train-split cutouts ({device})")
    model = ConvAutoencoder(in_ch=cut.shape[1], latent_dim=cfg["latent_dim"], size=cut.shape[-1])
    # train for a minimum number of optimisation STEPS, not just epochs: with few training
    # cutouts, 15 epochs is only a few dozen steps and the latent collapses to one direction
    steps_per_epoch = int(np.ceil(len(X) / cfg["batch_size"]))
    epochs = max(int(cfg["epochs"]), int(np.ceil(cfg.get("min_steps", 0) / steps_per_epoch)))
    ctx.log.info(f"[Stage 2a] {epochs} epochs x {steps_per_epoch} steps = {epochs * steps_per_epoch} optimisation steps")
    train_autoencoder(model, X, epochs, cfg["batch_size"], device, log=ctx.log.info, tag="Rep-CAE")
    torch.save(model.state_dict(), out / "rep_cae.pt")

    errs, latent = encode_all(model, cut, device)
    rows = df.cutout_index.to_numpy()
    errs, latent = errs[rows], latent[rows]
    is_train = (df.split_tag == "train").to_numpy()
    # scree on the TRAIN latents; keep the leading components up to n_pca, but drop any that
    # explain less than pca_min_variance (a ~0 % component only adds noise to the clustering)
    scree = PCA(random_state=ctx.seed).fit(latent[is_train]).explained_variance_ratio_
    if scree[0] > 0.9:
        ctx.log.info(f"[Stage 2a] WARNING: PC1 explains {scree[0]:.1%} of the latent variance - the CAE latent has "
                     "collapsed to one direction (under-trained). Raise representation.min_steps.")
    min_var = float(cfg.get("pca_min_variance", 0.0))
    k = int(min(cfg["n_pca"], latent.shape[1]))
    k_used = max(2, min(k, int(np.sum(scree[:k] >= min_var))))
    if k_used < k:
        ctx.log.info(f"[Stage 2a] PC{k_used + 1}..PC{k} explain < {min_var:.0%} each "
                     f"({np.round(scree[k_used:k] * 100, 2).tolist()} %) -> keeping {k_used} components")
    pca = PCA(n_components=k_used, random_state=ctx.seed).fit(latent[is_train])
    Z = pca.transform(latent)
    _scree_plot(scree, k_used, out)
    z_scale = float(Z[is_train].std()) or 1.0          # one scale for the whole image block (not whitened:
    Z = Z / z_scale                                    # whitening gives noise components equal weight)
    prep = TabularPrep(cfg["tabular_features"]).fit(df[is_train])
    w = cfg["spatial_weight"]
    hybrid = np.hstack([Z * w, prep.transform(df) * (1 - w)]).astype(np.float32)

    np.save(out / "latent.npy", latent)
    np.save(out / "recon_error.npy", errs)
    np.save(out / "hybrid_features.npy", hybrid)
    np.save(out / "object_id.npy", df.object_id.to_numpy())
    joblib.dump({"pca": pca, "z_scale": z_scale, "prep": prep, "spatial_weight": w}, out / "fusion.joblib")
    save_json({"n_objects": len(df), "n_train": int(is_train.sum()), "hybrid_dim": hybrid.shape[1],
               "tabular_kept": prep.kept, "n_pca_used": k_used,
               "pca_explained_variance": float(pca.explained_variance_ratio_.sum()),
               "scree_first10_percent": np.round(scree[:10] * 100, 2).tolist()},
              out / "summary.json")
    ctx.log.info(f"[Stage 2a] hybrid matrix {hybrid.shape} | tabular kept {prep.kept} | "
                 f"PCA keeps {pca.explained_variance_ratio_.sum():.1%} of latent variance")
    return {"hybrid_dim": int(hybrid.shape[1])}


def _scree_plot(ratio, k_used, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    n = min(len(ratio), 15)
    x = np.arange(1, n + 1)
    fig, ax = plt.subplots(figsize=(5.5, 3.4))
    ax.bar(x, ratio[:n] * 100, color=["#2E5496" if i < k_used else "#BBBBBB" for i in range(n)])
    ax.plot(x, np.cumsum(ratio[:n]) * 100, "o-", color="#E69F00", ms=3, label="cumulative")
    ax.set_xticks(x)
    ax.set_xlabel("principal component of the CAE latent")
    ax.set_ylabel("explained variance [%]")
    ax.set_title(f"Stage 2a scree (train split) - {k_used} components kept")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out / "pca_scree.png", dpi=110)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage2b_clustering.py
"""Stage 2b - Unsupervised clustering (GMM + HDBSCAN).

Purpose : group sources into populations without any label.
Method  : a Gaussian Mixture Model (diagonal covariance by default: far fewer
          parameters, so BIC can afford more, purer components) with a BIC sweep over k,
          and HDBSCAN (density-based, marks low-density points as noise -1).
          HDBSCAN is tried with a small grid of min_cluster_size values and the
          fit with the best density-based validity (DBCV) among those with >= 2
          clusters is kept; if none finds 2 clusters HDBSCAN is reported as
          FAILED and cluster_id_hdbscan is left empty (not -1, which would claim
          every object is density noise).
          Both are fitted on the TRAIN split; every other object is assigned
          with GMM.predict / hdbscan.approximate_predict - never by refitting.
Output  : clusters.csv (gmm label + soft posteriors, hdbscan label + strength),
          gmm.joblib, hdbscan.joblib, bic.png
"""
from __future__ import annotations

import joblib
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture

from ..context import Ctx, save_json

try:
    import hdbscan as _hdb
    _HDB = "contrib"
except Exception:
    _HDB = None


def run(ctx: Ctx):
    cfg = ctx.cfg["clustering"]
    out = ctx.stage_dir("2b")
    X = np.load(ctx.require("2a", "hybrid_features.npy"))
    oid = np.load(ctx.require("2a", "object_id.npy"))
    src = pd.read_csv(ctx.require("1c", "sources.csv")).set_index("object_id").loc[oid]
    tr = (src.split_tag == "train").to_numpy()
    Xtr = X[tr]

    # float64: in float32 the variance of a tight component can round to <= 0 ("ill-defined covariance")
    X, Xtr = X.astype(np.float64), Xtr.astype(np.float64)
    p1 = ctx.path("1e", "features_pass1.csv")
    if p1.exists():
        f = pd.read_csv(p1, usecols=["object_id", "flag"]).set_index("object_id").reindex(oid)["flag"].to_numpy()
        n_faint = int((f[tr] == 4).sum())
        ctx.log.info(f"[Stage 2b] {n_faint} of {int(tr.sum())} train sources are too faint for morphology "
                     "(their shape features are median-filled, so they sit on top of each other)")
    reg0 = float(cfg.get("reg_covar", 1e-4))
    bics, models = {}, {}
    for k in range(cfg["k_min"], cfg["k_max"] + 1):
        g, reg = None, reg0
        while g is None and reg <= 1e-2:
            try:
                g = GaussianMixture(n_components=k, covariance_type=cfg.get("covariance_type", "diag"),
                                    random_state=ctx.seed, n_init=3, reg_covar=reg).fit(Xtr)
            except ValueError:                     # a component collapsed onto (near-)identical points
                reg *= 10
        if g is None:
            ctx.log.info(f"  [GMM] k={k}  failed (collapsed component even with reg_covar 1e-2) - skipped")
            continue
        bics[k], models[k] = g.bic(Xtr), g
        ctx.log.info(f"  [GMM] k={k}  BIC={bics[k]:.1f}" + (f"  (reg_covar raised to {reg:g})" if reg > reg0 else ""))
    if not bics:
        raise RuntimeError("Every GMM fit collapsed. Lower clustering.k_max or raise clustering.reg_covar.")
    best_k = min(bics, key=bics.get)
    if best_k == cfg["k_max"]:
        ctx.log.info(f"[Stage 2b] note: BIC is lowest at the largest k tried ({best_k}); raising clustering.k_max "
                     "may split the populations further (cluster naming is many-to-one, so this is safe).")
    gmm = models[best_k]
    probs = gmm.predict_proba(X)
    res = pd.DataFrame({"object_id": oid, "cluster_id_gmm": probs.argmax(1)})
    for j in range(best_k):
        res[f"gmm_p{j}"] = probs[:, j].round(5)
    joblib.dump(gmm, out / "gmm.joblib")

    mcs = cfg["hdbscan_min_cluster_size"]
    mcs = max(10, len(Xtr) // 50) if mcs == "auto" else int(mcs)
    hdb_status, hdb_trials = "not run", []
    if _HDB:
        grid = sorted({mcs, max(5, mcs // 2), max(5, mcs // 4), 10}, reverse=True)
        best = None
        for m in grid:
            h = _hdb.HDBSCAN(min_cluster_size=m, min_samples=cfg.get("hdbscan_min_samples", 5),
                             prediction_data=True, gen_min_span_tree=True).fit(Xtr)
            n_c = len(set(h.labels_) - {-1})
            try:
                dbcv = float(h.relative_validity_) if n_c >= 2 else np.nan
            except Exception:
                dbcv = np.nan
            hdb_trials.append({"min_cluster_size": m, "clusters": n_c, "noise_fraction": round(float((h.labels_ == -1).mean()), 4),
                               "dbcv": None if not np.isfinite(dbcv) else round(dbcv, 4)})
            ctx.log.info(f"  [HDBSCAN] min_cluster_size={m}: {n_c} clusters, noise {hdb_trials[-1]['noise_fraction']:.1%}, "
                         f"DBCV {hdb_trials[-1]['dbcv']}")
            if n_c >= 2 and (best is None or np.nan_to_num(dbcv, nan=-1) > best[2]):
                best = (h, m, np.nan_to_num(dbcv, nan=-1))
        if best is not None:
            h, mcs = best[0], best[1]
            lab, strength = _hdb.approximate_predict(h, X)
            lab[tr] = h.labels_                       # training points keep their fitted labels
            res["cluster_id_hdbscan"] = lab
            res["hdbscan_strength"] = np.round(strength, 4)
            joblib.dump(h, out / "hdbscan.joblib")
            n_h = len(set(h.labels_) - {-1})
            hdb_status = "ok"
        else:
            res["cluster_id_hdbscan"] = np.nan
            n_h = 0
            hdb_status = "failed"
            ctx.log.info(f"[Stage 2b] HDBSCAN FAILED: no min_cluster_size in {grid} gives 2 clusters on this feature "
                         "space. cluster_id_hdbscan is left empty and Stage 2e gating runs without it - report "
                         "HDBSCAN as failed.")
    else:
        res["cluster_id_hdbscan"] = np.nan
        n_h = 0
        ctx.log.info("[Stage 2b] hdbscan package missing (pip install hdbscan) - HDBSCAN skipped")
    res.to_csv(out / "clusters.csv", index=False)
    save_json({"bic": bics, "best_k": best_k, "hdbscan_status": hdb_status, "hdbscan_trials": hdb_trials,
               "hdbscan_min_cluster_size": mcs, "hdbscan_clusters": n_h,
               "gmm_sizes": res.cluster_id_gmm.value_counts().to_dict()}, out / "summary.json")
    _plot(bics, best_k, out)
    ctx.log.info(f"[Stage 2b] BIC-selected k = {best_k}; GMM sizes {res.cluster_id_gmm.value_counts().to_dict()}; "
                 f"HDBSCAN clusters = {n_h} (min_cluster_size {mcs}), noise fraction "
                 f"{(res.cluster_id_hdbscan == -1).mean():.1%}")
    return {"best_k": best_k, "hdbscan_clusters": n_h}


def _plot(bics, best, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5, 3.2))
    ks = sorted(bics)
    ax.plot(ks, [bics[k] for k in ks], "o-")
    ax.axvline(best, color="r", ls="--", label=f"best k = {best}")
    ax.set_xlabel("number of GMM components k")
    ax.set_ylabel("BIC (lower is better)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out / "bic.png", dpi=110)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage2c_baseline.py
"""Stage 2c - Baseline segregation (+ the direct Gaia DR3 cross-match).

Purpose : a simple, transparent Star / Galaxy / Noise separator - the yardstick
          the learned segregation (Stages 2a-2d) must beat (Objective 2).
Method  : (1) Gaia DR3 match within 0.5" after propagating Gaia positions to the
          HST epoch with proper motions -> Star;
          (2) noise: SNR < 3, a profile SHARPER than the PSF (FWHM < 0.7 x the
              stellar locus - no real source can be sharper; cosmic rays can),
              or a single-band detection (band_consistency < 0.05);
          (3) stellar locus: FWHM within 20% of the stellar locus and a
              CLASS_STAR-like stellarity > 0.5 -> Star;
          (4) everything else -> Galaxy.
          The stellarity is a stand-in for SExtractor's CLASS_STAR built from
          how close the source FWHM is to the PSF FWHM.
Output  : baseline.csv, gaia_match.csv
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..catalogs import load_gaia, propagate_gaia
from ..context import Ctx, load_json, load_objects
from ..utils import noise_evidence, sky_crossmatch


def gaia_crossmatch(ctx: Ctx, df: pd.DataFrame, pts: dict, metas):
    cfg = ctx.cfg["semantic_audit"]
    gaia = load_gaia(ctx, metas)
    out = pd.DataFrame({"object_id": df.object_id, "gaia_hit": False})
    if gaia is None or gaia.empty:
        ctx.log.info("[Stage 2c] no Gaia table (data/catalogs/*gaia*.csv) - Gaia leg disabled")
        return out, False
    ctx.log.info(f"[Stage 2c] Gaia: {len(gaia)} sources in the footprint")
    cols = {k: np.full(len(df), np.nan) for k in ("gaia_sep_arcsec", "parallax", "pmra", "pmdec", "gaia_g", "gaia_teff")}
    sid = np.full(len(df), -1, dtype=np.int64)
    hit_all = np.zeros(len(df), bool)
    for pid, p in pts.items():
        m = (df.pointing == pid).to_numpy()
        if not m.any():
            continue
        g = propagate_gaia(gaia, p["epoch"], ctx.cfg["catalogs"]["gaia"].get("epoch", 2016.0))
        idx, sep, hit = sky_crossmatch(df.ra[m], df.dec[m], g.ra, g.dec, cfg["gaia_radius_arcsec"])
        rows = np.where(m)[0][hit]
        j = idx[hit]
        hit_all[rows] = True
        sid[rows] = pd.to_numeric(g["source_id"], errors="coerce").fillna(-1).astype(np.int64).to_numpy()[j]
        cols["gaia_sep_arcsec"][rows] = sep[hit]
        for c, gc in (("parallax", "parallax"), ("pmra", "pmra"), ("pmdec", "pmdec"), ("gaia_g", "g_mag"), ("gaia_teff", "teff")):
            cols[c][rows] = g[gc].to_numpy(float)[j]
    out["gaia_hit"] = hit_all
    out["gaia_source_id"] = np.where(hit_all, sid, -1)
    for c, v in cols.items():
        out[c] = v
    return out, True


def run(ctx: Ctx):
    cfg = ctx.cfg["baseline"]
    out = ctx.stage_dir("2c")
    df = load_objects(ctx, ["sources", "pass1"])
    pts = load_json(ctx.require("1a", "pointings.json"))["pointings"]
    metas = list(load_json(ctx.require("1a", "manifest.json")).values())
    gm, have_gaia = gaia_crossmatch(ctx, df, pts, metas)
    gm.to_csv(out / "gaia_match.csv", index=False)

    ratio = df.fwhm_over_psf.to_numpy(float)                       # 1 = as compact as a star (Stage 1e)
    stellarity = np.exp(-((ratio - 1.0) ** 2) / (2 * 0.15 ** 2))   # CLASS_STAR-like, 1 = PSF-like
    stellarity = np.where(np.isfinite(stellarity), stellarity, 0.0)
    noise = noise_evidence(df, cfg)
    locus = np.abs(ratio - 1.0) <= cfg["point_source_tolerance"]
    star = locus & (stellarity > cfg["class_star_threshold"]) & ~noise
    label = np.where(noise, "Noise", np.where(star, "Star", "Galaxy")).astype(object)
    label[gm.gaia_hit.to_numpy()] = "Star"
    res = pd.DataFrame({"object_id": df.object_id, "class_star_proxy": stellarity.round(4),
                        "fwhm_over_psf": ratio.round(4), "baseline_semantic": label,
                        "baseline_semantic_nogaia": np.where(noise, "Noise", np.where(star, "Star", "Galaxy"))})
    res.to_csv(out / "baseline.csv", index=False)
    ctx.log.info(f"[Stage 2c] baseline: {pd.Series(label).value_counts().to_dict()} | Gaia matches "
                 f"{int(gm.gaia_hit.sum())} (Gaia leg {'on' if have_gaia else 'off'})")
    return {"baseline_counts": pd.Series(label).value_counts().to_dict(), "gaia_matches": int(gm.gaia_hit.sum())}

In [ ]:
%%writefile src/p2p/stages/stage2d_semantic_audit.py
"""Stage 2d - Semantic audit: naming the clusters from INDEPENDENT evidence.

Purpose : turn abstract GMM cluster IDs into Star / Galaxy / Noise, and
          measure how good the learned segregation is against Gaia.
Method  : three evidence signals per object, none of them used to cluster:
            Star   - Gaia DR3 match (Stage 2c), OR on the stellar locus of the
                   size-magnitude plane: point-like (|FWHM / stellar locus - 1|
                   <= baseline.point_source_tolerance) with SNR >=
                   baseline.star_locus_min_snr, or saturated, AND confirmed in a
                   second band (a cosmic ray is as compact as a star but
                   single-band). Below that S/N the point-like band fills up with
                   compact distant galaxies that size cannot separate from stars
                   (check size_magnitude.png). Deep fields hold only a handful of
                   Gaia stars, too few to name a cluster alone;
            Galaxy - position matched to the Galaxy Zoo: Hubble catalogue, OR
                     resolved (FWHM / PSF > 1 + point_source_tolerance), SNR >= 5
                     and confirmed in a second band: GZH stops near mag 23.5, and
                     without this a cluster of fainter galaxies would carry no
                     galaxy evidence and be named Noise;
            Noise  - SNR < 3, a profile sharper than the PSF, or a single-band
                     detection (utils.noise_evidence).
          Each signal's per-cluster rate - measured on TRAIN-split members
          only - is divided by its maximum over the clusters (so a rare signal
          like Gaia is not swamped) and the strongest one names the cluster;
          several clusters may share a name.
          A cluster with no evidence takes the majority of the baseline rule.
          Gaia confirms stars but cannot reject them (it stops at G~21), so
          precision/recall are reported for STARS ONLY, inside Gaia's magnitude
          range, on HELD-OUT (non-train) objects, so naming and scoring never
          use the same Gaia matches; the baseline is scored WITHOUT its own Gaia leg.
          25-object montages per cluster support a visual check.
Output  : evidence.csv, cluster_names.json, semantic_audit.json, montage_cluster_*.png,
          size_magnitude.png (check the stellar locus before trusting the names)
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..catalogs import load_gzh
from ..context import Ctx, load_json, load_objects, save_json
from ..features import load_cutouts
from ..utils import as_bool, asinh_normalise, match_with_offset, noise_evidence

NAMES = ("Star", "Galaxy", "Noise")


def star_metrics(pred, truth, in_range):
    p = (np.asarray(pred) == "Star") & in_range
    t = np.asarray(truth, bool) & in_range
    tp, fp, fn = int((p & t).sum()), int((p & ~t).sum()), int((~p & t).sum())
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {"precision": round(prec, 3), "recall": round(rec, 3),
            "f1": round(2 * prec * rec / max(prec + rec, 1e-9), 3), "n_gaia_stars": int(t.sum())}


def run(ctx: Ctx):
    cfg = ctx.cfg["semantic_audit"]
    out = ctx.stage_dir("2d")
    df = load_objects(ctx, ["sources", "pass1", "clusters", "baseline", "gaia"])
    metas = list(load_json(ctx.require("1a", "manifest.json")).values())

    gzh = load_gzh(ctx, metas, required=False)
    if gzh is not None and len(gzh):
        _, _, hit, off = match_with_offset(df.ra, df.dec, gzh.ra, gzh.dec, cfg["gzh_radius_arcsec"],
                                           estimate=ctx.cfg["catalogs"]["gzh"].get("estimate_offset", True))
        ctx.log.info(f"[Stage 2d] GZH positions: {int(hit.sum())} matches (offset removed: "
                     f"dRA {off[0]:+.3f}\", dDec {off[1]:+.3f}\")")
    else:
        hit = np.zeros(len(df), bool)
        ctx.log.info("[Stage 2d] no GZH table - galaxy evidence falls back to the baseline rule")
    gaia = df.gaia_hit.astype(str).str.lower().eq("true").to_numpy()
    noise = noise_evidence(df, ctx.cfg["baseline"])
    locus_star = stellar_locus(df, ctx.cfg["baseline"])
    star_ev = gaia | locus_star
    resolved_real = resolved_two_band(df, ctx.cfg["baseline"])
    gal_ev = hit | resolved_real
    ctx.log.info(f"[Stage 2d] galaxy evidence: {int(hit.sum())} GZH + {int((resolved_real & ~hit).sum())} resolved and "
                 "confirmed in a second band")
    ctx.log.info(f"[Stage 2d] star evidence: {int(gaia.sum())} Gaia + {int((locus_star & ~gaia).sum())} on the stellar "
                 f"locus (point-like or saturated, confirmed in a second band)")
    ev = pd.DataFrame({"cluster": df.cluster_id_gmm, "Star": star_ev, "Galaxy": gal_ev, "Noise": noise})
    # Clusters are NAMED from TRAIN-split evidence only and SCORED on held-out objects: naming and
    # scoring on the same Gaia matches would be circular, and test objects must not shape the labels.
    tr = (df.split_tag == "train").to_numpy()
    rate = ev[tr].groupby("cluster")[list(NAMES)].mean()
    rate["n_train"] = ev[tr].groupby("cluster").size()
    no_train = sorted(set(ev.cluster) - set(rate.index))
    if no_train:                                   # a cluster with no train member: use all its members
        extra = ev[ev.cluster.isin(no_train)].groupby("cluster")[list(NAMES)].mean()
        extra["n_train"] = 0
        rate = pd.concat([rate, extra])
    rate["n"] = ev.groupby("cluster").size()
    score = rate[list(NAMES)] / rate[list(NAMES)].max().replace(0, 1)
    names = {}
    for c, row in score.iterrows():
        if row.max() > 0:
            names[int(c)] = NAMES[int(np.argmax(row.values))]
        else:
            names[int(c)] = df.loc[df.cluster_id_gmm == c, "baseline_semantic_nogaia"].mode().iloc[0]
    rate["semantic"] = pd.Series(names)
    semantic = df.cluster_id_gmm.map(names)

    in_range = (df.mag < cfg["gaia_mag_limit"]).to_numpy()
    held = in_range & ~tr
    metrics = {"learned_star_vs_gaia": star_metrics(semantic, gaia, held),
               "baseline_star_vs_gaia": star_metrics(df.baseline_semantic_nogaia, gaia, held),
               "scored_on": "held-out objects (not train), detection-band mag < gaia_mag_limit",
               "learned_star_vs_gaia_all_objects": star_metrics(semantic, gaia, in_range),
               "baseline_star_vs_gaia_all_objects": star_metrics(df.baseline_semantic_nogaia, gaia, in_range),
               "learned_galaxy_recall_vs_gzh": round(float((semantic[hit & ~tr] == "Galaxy").mean()), 3)
               if (hit & ~tr).any() else None,
               "baseline_galaxy_recall_vs_gzh": round(float((df.baseline_semantic_nogaia[hit & ~tr] == "Galaxy").mean()), 3)
               if (hit & ~tr).any() else None,
               "cluster_names": {str(k): v for k, v in names.items()},
               "cluster_evidence": rate.round(3).reset_index().to_dict(orient="records")}
    l, b = metrics["learned_star_vs_gaia"], metrics["baseline_star_vs_gaia"]
    metrics["objective2_minimum_met"] = bool(l["n_gaia_stars"] > 0 and l["f1"] > b["f1"])
    save_json(metrics, out / "semantic_audit.json")
    save_json({str(k): v for k, v in names.items()}, out / "cluster_names.json")
    pd.DataFrame({"object_id": df.object_id, "gzh_position_hit": hit, "noise_evidence": noise,
                  "stellar_locus": locus_star, "cluster_semantic_2d": semantic}).to_csv(out / "evidence.csv", index=False)
    _montages(ctx, df, names, cfg["montage_n"], out)
    _size_magnitude(df, gaia, locus_star, noise, semantic, ctx.cfg["baseline"], out)
    ctx.log.info("[Stage 2d] cluster evidence (share of members with each signal):\n" + rate.round(3).to_string())
    ctx.log.info(f"[Stage 2d] stars vs Gaia (held-out, in range, n={l['n_gaia_stars']}): learned P={l['precision']} "
                 f"R={l['recall']} F1={l['f1']} | baseline P={b['precision']} R={b['recall']} F1={b['f1']} | "
                 f"Objective-2 minimum met: {metrics['objective2_minimum_met']}")
    return {"learned_f1": l["f1"], "baseline_f1": b["f1"]}


def two_band_confirmed(df):
    return df.cr_candidate.map(lambda v: str(v).strip().lower() in ("false", "0", "0.0")).to_numpy() \
        if "cr_candidate" in df else np.ones(len(df), bool)


def resolved_two_band(df, cfg_b, min_snr=5.0):
    """A real extended source: broader than a point source, SNR >= 5, confirmed in a second band."""
    ratio = np.nan_to_num(df.fwhm_over_psf.to_numpy(float), nan=0)
    return (ratio > 1 + cfg_b["point_source_tolerance"]) & (df.snr.to_numpy(float) >= min_snr) & two_band_confirmed(df)


def stellar_locus(df, cfg_b):
    """Point-like (or saturated) AND confirmed in a second band AND bright enough (SNR >= star_locus_min_snr)
    that size separates a star from a compact galaxy: the stellar sequence of the size-magnitude plane."""
    min_snr = float(cfg_b.get("star_locus_min_snr", 150))
    ratio = df.fwhm_over_psf.to_numpy(float)
    two_band = df.cr_candidate.map(lambda v: str(v).strip().lower() in ("false", "0", "0.0")).to_numpy() \
        if "cr_candidate" in df else np.ones(len(df), bool)
    bright = df.snr.to_numpy(float) >= min_snr
    point = np.abs(ratio - 1.0) <= cfg_b["point_source_tolerance"]
    sat = as_bool(df.saturated).to_numpy() if "saturated" in df else np.zeros(len(df), bool)
    return ((point & bright) | sat) & two_band


def _size_magnitude(df, gaia, locus, noise, semantic, cfg_b, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
    y = df.fwhm_over_psf.to_numpy(float)
    m = df.mag.to_numpy(float)
    for a, title, groups in (
            (ax[0], "evidence", [(noise, "noise evidence", "0.75", 3), (~noise & ~locus & ~gaia, "other", "tab:blue", 4),
                                 (locus & ~gaia, "stellar locus", "tab:orange", 10), (gaia, "Gaia", "red", 30)]),
            (ax[1], "cluster names (Stage 2d)", [(np.asarray(semantic) == n, n, c, 4) for n, c in
                                                 (("Noise", "0.75"), ("Galaxy", "tab:blue"), ("Star", "tab:orange"))])):
        for mask, lab, col, s in groups:
            mask = np.asarray(mask, bool)
            a.scatter(m[mask], y[mask], s=s, c=col, label=f"{lab} ({int(mask.sum())})", edgecolors="none", alpha=0.7)
        a.axhspan(1 - cfg_b["point_source_tolerance"], 1 + cfg_b["point_source_tolerance"], color="k", alpha=0.08)
        lim = np.nanmedian(m[np.abs(df.snr.to_numpy(float) / cfg_b.get("star_locus_min_snr", 150) - 1) < 0.1])
        if np.isfinite(lim):
            a.axvline(lim, color="k", ls=":", lw=1)
            a.text(lim, a.get_ylim()[1] if a.get_yscale() == "linear" else 8, f" SNR {cfg_b.get('star_locus_min_snr', 150):g}",
                   fontsize=7, va="top")
        a.set_yscale("log")
        a.set_xlabel("detection-band AB magnitude")
        a.set_title(title)
        a.legend(fontsize=7, markerscale=1.5)
    ax[0].set_ylabel("FWHM / stellar-locus FWHM")
    fig.suptitle("Stage 2d: size-magnitude plane (grey band = point-like); stars sit on a horizontal sequence at 1")
    fig.tight_layout()
    fig.savefig(out / "size_magnitude.png", dpi=110)
    plt.close(fig)


def _montages(ctx, df, names, n, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    cut = load_cutouts(ctx)
    rng = np.random.default_rng(ctx.seed)
    for c in sorted(df.cluster_id_gmm.unique()):
        members = df[df.cluster_id_gmm == c]
        pick = members.sample(min(n, len(members)), random_state=int(rng.integers(1e6)))
        fig, axes = plt.subplots(5, 5, figsize=(6, 6))
        for ax in axes.ravel():
            ax.axis("off")
        for ax, i in zip(axes.ravel(), pick.cutout_index):
            ax.imshow(asinh_normalise(np.asarray(cut[i], np.float32).sum(0)), origin="lower", cmap="gray")
        fig.suptitle(f"GMM cluster {c} -> {names[int(c)]}  (n = {len(members)})")
        fig.tight_layout()
        fig.savefig(out / f"montage_cluster_{c}.png", dpi=80)
        plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage2e_continuum_gating.py
"""Stage 2e - Continuum handling and outlier gating.

Purpose : do not throw away real galaxies that sit between populations, and do
          not silently drop true anomalies either.
Method  : GMM components are summed into Star / Galaxy / Noise posteriors using
          the Stage-2d names. An object that HDBSCAN calls noise (-1) but whose
          galaxy posterior is >= 0.80 is KEPT as a galaxy and tagged
          is_transitional. An HDBSCAN-noise object with no posterior above 0.50
          AND a CAE reconstruction error above the 99th percentile of the train
          split is an extreme outlier -> anomaly_log.csv (anomaly_flag = 1).
          Physical consistency gate (gating.physical_gate): a GMM cluster is
          named as a whole, but one cluster can mix stars with compact galaxies.
          Each object is then checked against evidence the cluster name must
          not contradict:
            cosmic-ray candidate (no second band, Stage 1c)  -> Noise
            Gaia star                                         -> Star
            in a Star cluster without star evidence (Gaia, or the bright stellar
              locus of Stage 2d): sharper than the PSF -> Noise, otherwise -> Galaxy
              (faint point-like sources in a deep field are mostly compact galaxies)
            in a Galaxy cluster but on the stellar locus (Stage 2d)                    -> Star
            in a Noise cluster but confirmed in two bands, SNR >= 5, not sharper than
              the PSF (a real source): on the stellar locus -> Star, otherwise -> Galaxy
          The reason is kept in classification_override (empty = cluster name
          kept); cluster_semantic_2d keeps the pure cluster name, which is what
          Stage 2d scores for Objective 2.
          If HDBSCAN failed (Stage 2b), the transitional rule cannot be applied:
          is_transitional is left EMPTY (unknown, not False), and the outlier
          rule runs without the HDBSCAN condition.
Output  : segregation.csv (final cluster_semantic for every object), anomaly_log.csv
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..context import Ctx, load_json, load_objects
from ..utils import as_bool

NAMES = ("Star", "Galaxy", "Noise")


def physical_gate(df, sem, override, cfg_b):
    """Object-level checks a cluster name must not contradict (see the module docstring)."""
    tol = cfg_b["point_source_tolerance"]
    ratio = df.fwhm_over_psf.to_numpy(float)
    cr = as_bool(df.cr_candidate).to_numpy() if "cr_candidate" in df else np.zeros(len(df), bool)
    gaia = as_bool(df.gaia_hit).to_numpy() if "gaia_hit" in df else np.zeros(len(df), bool)
    locus = as_bool(df.stellar_locus).to_numpy() if "stellar_locus" in df else np.zeros(len(df), bool)
    two_band = df.cr_candidate.map(lambda v: str(v).strip().lower() in ("false", "0", "0.0")).to_numpy() \
        if "cr_candidate" in df else np.ones(len(df), bool)
    resolved = np.nan_to_num(ratio, nan=0) > 1 + 2 * tol
    real_noise_cluster = two_band & (df.snr.to_numpy(float) >= 5) & (np.nan_to_num(ratio, nan=0) >= cfg_b["sub_psf_factor"])
    sem, override = sem.copy(), override.copy()

    def move(mask, to, why):
        mask = mask & (sem != to)
        sem[mask] = to
        override[mask] = why

    move((sem == "Noise") & real_noise_cluster & locus, "Star", "Noise cluster, but confirmed in 2 bands on the stellar locus")
    move((sem == "Noise") & real_noise_cluster & ~locus, "Galaxy",
         "Noise cluster, but a real source (confirmed in 2 bands, SNR >= 5, not sharper than the PSF)")
    move((sem == "Galaxy") & locus & ~resolved, "Star", "Galaxy cluster, but on the stellar locus")
    broader = np.nan_to_num(ratio, nan=0) > 1 + tol
    sharper = np.nan_to_num(ratio, nan=1) < cfg_b["sub_psf_factor"]
    no_star_evidence = (sem == "Star") & ~locus & ~gaia
    move(no_star_evidence & broader, "Galaxy", "Star cluster, but broader than a point source")
    move(no_star_evidence & sharper, "Noise", "Star cluster, but sharper than the PSF")
    move(no_star_evidence & ~broader & ~sharper, "Galaxy",
         "Star cluster, but too faint for size to separate a star from a compact galaxy")
    move(gaia, "Star", "Gaia star")
    move(cr & ~gaia, "Noise", "cosmic-ray candidate (no second band confirms it)")
    return sem, override


def run(ctx: Ctx):
    cfg = ctx.cfg["gating"]
    out = ctx.stage_dir("2e")
    df = load_objects(ctx, ["sources", "pass1", "clusters", "gaia", "evidence"])
    names = {int(k): v for k, v in load_json(ctx.require("2d", "cluster_names.json")).items()}
    err = pd.Series(np.load(ctx.require("2a", "recon_error.npy")),
                    index=np.load(ctx.require("2a", "object_id.npy"))).reindex(df.object_id).to_numpy()

    post = {n: np.zeros(len(df)) for n in NAMES}
    for k, n in names.items():
        col = f"gmm_p{k}"
        if col in df and n in post:
            post[n] += df[col].to_numpy(float)
    hdb = df.cluster_id_hdbscan.to_numpy(float)
    hdb_ok = np.isfinite(hdb).any() and (hdb >= 0).any()
    if not hdb_ok:
        ctx.log.info("[Stage 2e] WARNING: HDBSCAN failed (Stage 2b) - is_transitional left empty; the outlier "
                     "rule uses GMM posterior + reconstruction error only.")
    hnoise = (hdb == -1) if hdb_ok else np.ones(len(df), bool)
    P = np.column_stack([post[n] for n in NAMES])
    thr = float(np.nanpercentile(err[(df.split_tag == "train").to_numpy()], cfg["recon_percentile"]))
    transitional = hnoise & (post["Galaxy"] >= cfg["transitional_p"]) if hdb_ok else np.zeros(len(df), bool)
    anomaly = hnoise & (P.max(1) < cfg["outlier_max_p"]) & (np.nan_to_num(err, nan=-1) > thr)

    sem = df.cluster_semantic_2d.to_numpy(object).copy()
    sem[transitional] = "Galaxy"
    override = np.full(len(df), None, dtype=object)
    override[transitional] = "transitional (HDBSCAN noise, GMM galaxy posterior >= transitional_p)"
    if cfg.get("physical_gate", True):
        sem, override = physical_gate(df, sem, override, ctx.cfg["baseline"])
        ctx.log.info(f"[Stage 2e] physical gate: {pd.Series(override).value_counts().to_dict()}")
    res = pd.DataFrame({"object_id": df.object_id, "cluster_semantic": sem,
                        "gmm_p_star": post["Star"].round(4), "gmm_p_galaxy": post["Galaxy"].round(4),
                        "gmm_p_noise": post["Noise"].round(4), "reconstruction_error": err,
                        "is_transitional": transitional if hdb_ok else None, "anomaly_flag": anomaly.astype(int),
                        "classification_override": override})
    res.to_csv(out / "segregation.csv", index=False)
    log = res[res.anomaly_flag == 1].merge(df[["object_id", "ra", "dec", "pointing", "cluster_id_gmm",
                                               "cluster_id_hdbscan"]], on="object_id")
    log["reason"] = "HDBSCAN noise + no GMM population > 0.5 + reconstruction error above train 99th pct"
    log.to_csv(out / "anomaly_log.csv", index=False)
    ctx.log.info(f"[Stage 2e] populations {res.cluster_semantic.value_counts().to_dict()} | transitional kept "
                 f"{int(transitional.sum())} | anomalies logged {int(anomaly.sum())} (recon threshold {thr:.5f})")
    return {"populations": res.cluster_semantic.value_counts().to_dict(), "anomalies": int(anomaly.sum())}

In [ ]:
%%writefile src/p2p/stages/stage3a_gzh_labels.py
"""Stage 3a - Galaxies: Galaxy Zoo: Hubble label anchoring.

Purpose : attach crowd-sourced morphology labels to the galaxy track.
Method  : galaxy positions are matched one-to-one to GZH within 0.5" after the
          median catalogue-to-image offset is removed. GZH vote fractions are
          conditional on the earlier answers of the decision tree, so:
            Smooth        f_smooth >= 0.6
            featured      f_features >= 0.5 (and not Smooth), then
              EdgeOn        f_edgeon >= 0.5
              FaceOnSpiral  not edge-on and f_spiral >= 0.5
              Irregular     not edge-on, not spiral, and (f_clumpy >= 0.5 or
                            [f_odd >= 0.4 and max(irregular, disturbed) >= 0.5])
            Ambiguous     everything else
          merger flag: f_odd >= 0.4 and f_merger >= 0.5.
          Human agreement = the WEAKEST vote fraction along the path to the
          assigned class; matched galaxies with agreement < 0.5 form the
          Ambiguous Boundary set used by Stage 6b. Unmatched galaxies are kept
          for inference only (is_pseudo_label = True).
Output  : labels.csv, label_summary.json
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..catalogs import load_gzh
from ..context import Ctx, load_json, load_objects, save_json
from ..nets import MORPH_CLASSES
from ..utils import match_with_offset


def map_labels(g: pd.DataFrame, t: dict) -> pd.DataFrame:
    f = lambda c: g[c].fillna(0).to_numpy(float)
    smooth, feat, edge, spiral = f("smooth"), f("features"), f("edgeon"), f("spiral")
    odd, clumpy = f("odd"), f("clumpy")
    irr_odd = np.maximum(f("odd_irregular"), f("odd_disturbed"))
    irregular_votes = np.maximum(clumpy, np.where(odd >= t["odd"], irr_odd, 0.0))
    lab = np.full(len(g), 4)
    featured = (smooth < t["smooth"]) & (feat >= t["features"])
    lab[smooth >= t["smooth"]] = 0
    lab[featured & (edge >= t["edgeon"])] = 2
    lab[featured & (edge < t["edgeon"]) & (spiral >= t["spiral"])] = 1
    lab[featured & (edge < t["edgeon"]) & (spiral < t["spiral"]) & (irregular_votes >= t["irregular"])] = 3
    # agreement = weakest link along the decision path
    A = np.column_stack([smooth,
                         np.minimum.reduce([feat, 1 - edge, spiral]),
                         np.minimum(feat, edge),
                         np.minimum.reduce([feat, 1 - edge, 1 - spiral, irregular_votes])])
    A = np.column_stack([A, 1.0 - A.max(1)])
    agree = A[np.arange(len(lab)), lab]
    return pd.DataFrame({"gz_label_id": lab, "gz_label": np.array(MORPH_CLASSES)[lab],
                         "gz_vote_frac": agree.round(4), "human_disagreement": (1 - agree).round(4),
                         "gz_merger_flag": ((odd >= t["odd"]) & (f("odd_merger") >= t["merger"])).astype(int)})


def run(ctx: Ctx):
    cfg = ctx.cfg["catalogs"]["gzh"]
    out = ctx.stage_dir("3a")
    df = load_objects(ctx, ["sources", "segregation"])
    metas = list(load_json(ctx.require("1a", "manifest.json")).values())
    gzh = load_gzh(ctx, metas, required=True)
    gal = df[(df.cluster_semantic == "Galaxy") & (df.anomaly_flag == 0)].copy()
    idx, sep, hit, off = match_with_offset(gal.ra, gal.dec, gzh.ra, gzh.dec, cfg["match_radius_arcsec"],
                                           estimate=cfg.get("estimate_offset", True))
    lab = map_labels(gzh, cfg["thresholds"])
    res = pd.DataFrame({"object_id": gal.object_id.to_numpy()})
    for c in lab.columns:
        res[c] = np.where(hit, lab[c].to_numpy()[np.clip(idx, 0, None)], np.nan)
    res["gz_label"] = np.where(hit, lab["gz_label"].to_numpy()[np.clip(idx, 0, None)], None)
    res["gzh_sep_arcsec"] = np.where(hit, sep, np.nan).round(4)
    res["gzh_row"] = np.where(hit, gzh.gzh_row.to_numpy()[np.clip(idx, 0, None)], -1)
    res["is_pseudo_label"] = ~hit
    res["in_boundary_set"] = hit & (res.gz_vote_frac.fillna(1).to_numpy() < cfg["boundary_agreement"])
    res.to_csv(out / "labels.csv", index=False)

    lab_rows = res[~res.is_pseudo_label].merge(gal[["object_id", "split_tag"]], on="object_id")
    table = pd.crosstab(lab_rows.split_tag.fillna("collar"), lab_rows.gz_label)
    summary = {"galaxies": len(gal), "matched": int(hit.sum()), "unmatched": int((~hit).sum()),
               "offset_arcsec": {"dra": off[0], "ddec": off[1]},
               "per_split_class": table.to_dict(), "boundary_set": int(res.in_boundary_set.sum()),
               "merger_prevalence_train": float(lab_rows[lab_rows.split_tag == "train"].gz_merger_flag.mean())
               if (lab_rows.split_tag == "train").any() else None}
    save_json(summary, out / "label_summary.json")
    ctx.log.info(f"[Stage 3a] galaxy track {len(gal)} | GZH-matched {int(hit.sum())} | unmatched {int((~hit).sum())} "
                 f"| offset removed dRA {off[0]:+.3f}\" dDec {off[1]:+.3f}\"")
    ctx.log.info("[Stage 3a] labelled galaxies per split / class:\n" + table.to_string())
    return {"matched": int(hit.sum())}

In [ ]:
%%writefile src/p2p/stages/stage3b_star_routing.py
"""Stage 3b - Stars: calibration routing.

Purpose : point sources skip morphology classification and become PSF tracers.
Method  : candidates are the star track PLUS every Gaia-matched object (a Gaia
          star mis-assigned to another track is still a star). A candidate is
          routed to the empirical-PSF builder if it is a Gaia star or on the
          bright stellar locus of Stage 2d (point-like, SNR >=
          baseline.star_locus_min_snr, confirmed in a second band), and it has SNR >= 20, is not saturated (no flat-topped
          core in the detection band), has no other segment within
          psf_match.star_isolation_px (5 px) of its centre (Stage 5a leaves
          neighbour and cosmic-ray pixels out of the stack) and is not at an edge. Only Gaia stars are used
          whenever at least 5 of them in its pointing pass those cuts (in deep
          fields most Gaia stars saturate).
Output  : star_routing.csv (one row per star with each test), routing_summary.json
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..context import Ctx, load_json, load_objects, save_json
from ..utils import as_bool
from .stage1d_psf_match import flat_top


def run(ctx: Ctx, min_snr=20.0, min_epsf_stars=5):
    iso_radius = ctx.cfg["psf_match"].get("star_isolation_px", 5)   # Stage 5a leaves neighbour pixels out
    out = ctx.stage_dir("3b")
    df = load_objects(ctx, ["sources", "pass1", "gaia", "evidence", "segregation"])
    tol = ctx.cfg["baseline"]["point_source_tolerance"]
    bands = load_json(ctx.require("1a", "pointings.json"))["bands"]
    raw = np.load(ctx.require("1c", "cutouts_raw.npy"), mmap_mode="r")
    segs = np.load(ctx.require("1c", "segstamps.npy"), mmap_mode="r")
    stars = df[((df.cluster_semantic == "Star") | as_bool(df.gaia_hit)) & (df.anomaly_flag == 0)].copy()
    S = segs.shape[1]
    yy, xx = np.mgrid[:S, :S]
    near = np.hypot(yy - (S - 1) / 2, xx - (S - 1) / 2) <= iso_radius
    det_i = stars.detection_band.map(bands.index).to_numpy() if len(stars) else []
    stars["epsf_snr_ok"] = stars.snr >= min_snr
    stars["epsf_unsaturated"] = [not flat_top(raw[i, d]) for i, d in zip(stars.cutout_index, det_i)]
    stars["epsf_isolated"] = [not np.any(segs[i][near] == 2) for i in stars.cutout_index]
    stars["epsf_not_edge"] = ~as_bool(stars.edge_flag)
    stars["epsf_unsaturated"] &= ~as_bool(stars.saturated) if "saturated" in stars else True
    stars["epsf_point_like"] = (stars.fwhm_over_psf.astype(float) - 1.0).abs() <= tol
    stars["epsf_two_band"] = stars.cr_candidate.map(lambda v: str(v).strip().lower() in ("false", "0", "0.0")) \
        if "cr_candidate" in stars else True
    gaia = as_bool(stars.gaia_hit)
    locus = as_bool(stars.stellar_locus) if "stellar_locus" in stars else (stars.epsf_point_like & stars.epsf_two_band)
    stars["epsf_star_evidence"] = gaia | (locus & stars.epsf_point_like)
    ok = (stars.epsf_star_evidence & stars.epsf_snr_ok & stars.epsf_unsaturated & stars.epsf_isolated
          & stars.epsf_not_edge)
    # Gaia confirmation is required only when enough confirmed stars survive the cuts: in a deep field
    # most Gaia stars (G < 21) saturate a long ACS exposure, and Stage 5a needs >= 5 stars per pointing
    gaia_required = {}
    for pid, idx in stars.groupby("pointing").groups.items():
        with_gaia = ok.loc[idx] & gaia.loc[idx]
        gaia_required[pid] = bool(with_gaia.sum() >= min_epsf_stars)
        if gaia_required[pid]:
            ok.loc[idx] = with_gaia
        elif gaia.loc[idx].any():
            ctx.log.info(f"[Stage 3b] {pid}: only {int(with_gaia.sum())} Gaia-confirmed clean stars - using all "
                         f"{int(ok.loc[idx].sum())} clean stars of the star track for the ePSF")
    stars["used_in_epsf"] = ok
    cols = ["object_id", "pointing", "snr", "gaia_hit", "cluster_semantic", "epsf_point_like", "epsf_two_band",
            "epsf_star_evidence", "epsf_snr_ok", "epsf_unsaturated", "epsf_isolated", "epsf_not_edge", "used_in_epsf"]
    stars[cols].to_csv(out / "star_routing.csv", index=False)
    per = stars.groupby("pointing").used_in_epsf.sum().astype(int).to_dict()
    save_json({"stars": len(stars), "routed": int(ok.sum()), "per_pointing": per,
               "gaia_required_per_pointing": gaia_required}, out / "routing_summary.json")
    ctx.log.info(f"[Stage 3b] candidates {len(stars)} (star track + Gaia) | Gaia {int(gaia.sum())} | point-like "
                 f"{int(stars.epsf_point_like.sum())} | confirmed in 2 bands {int(stars.epsf_two_band.sum())} | "
                 f"SNR ok {int(stars.epsf_snr_ok.sum())} | unsaturated "
                 f"{int(stars.epsf_unsaturated.sum())} | isolated {int(stars.epsf_isolated.sum())} | routed to ePSF "
                 f"{int(ok.sum())} {per}")
    return {"routed": int(ok.sum())}

In [ ]:
%%writefile src/p2p/stages/stage3c_noise_model.py
"""Stage 3c - Noise: false-positive modelling.

Purpose : characterise spurious detections - the Noise population has no
          external catalogue, so this model is also its validation.
Method  : two kinds of spurious detection are modelled separately:
          (1) sky-noise peaks - the NEGATIVE-IMAGE test on each detection frame.
              Sky noise is symmetric, so the number of 'sources' found on -SCI
              at nu x sigma estimates how many detections on +SCI at the same
              threshold are spurious. nu is each source's PEAK significance in
              the smoothed detection image (the quantity the detection threshold
              is applied to), not its total SNR. The grid of thresholds extends
              to the brightest source, so bright sources are not clamped to the
              last grid value. The FPR cannot be resolved below 1 / n_pos(nu)
              (one negative detection); that floor is reported, and an FPR of 0
              means 'below the floor'.
          (2) cosmic rays - positive-only, so the negative-image test cannot see
              them (single exposures are full of them, at HIGH S/N). Their term
              is p_cr = sigmoid((confirm_snr - best other-band S/N) / 0.5) from
              the Stage-1c two-band test (0 when no other band covers the source).
          (3) a profile sharper than the PSF: p_sub = sigmoid((sub_psf_factor - FWHM/locus) / 0.05).
          noise_score = 1 - (1 - FPR(nu)) (1 - p_sub) (1 - p_cr).
Output  : noise_model.json (FPR curve per pointing, operating threshold, resolution floor),
          noise_scores.csv, fpr_curve.png
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..compat import detect, n_labels
from ..context import Ctx, load_json, load_objects, save_json
from ..detection import smooth, subtract_background
from ..fitsio import load_image, valid_mask


def threshold_grid(nu_max, lo=1.5, fine_to=6.0, step=0.25, n_log=20):
    """Fine linear steps to 6 sigma, then logarithmic steps up to the brightest source."""
    fine = np.arange(lo, fine_to + 1e-9, step)
    if not np.isfinite(nu_max) or nu_max <= fine_to:
        return fine
    return np.unique(np.concatenate([fine, np.geomspace(fine_to, nu_max * 1.05, n_log)]))


def run(ctx: Ctx, target_fpr=0.01):
    out = ctx.stage_dir("3c")
    dcfg = ctx.cfg["detection"]
    pts = load_json(ctx.require("1a", "pointings.json"))["pointings"]
    df = load_objects(ctx, ["sources", "baseline", "segregation"])
    nu = np.full(len(df), np.nan)
    curves = {}
    for pid, p in pts.items():
        sci, wht, _ = load_image(p["bands"][p["detection_band"]])
        val = valid_mask(sci, wht)
        sub, rms = subtract_background(sci, val, dcfg["background_box"], dcfg["background_filter"])
        conv = smooth(sub, dcfg["kernel_fwhm_px"])                 # same filter as Stage 1c, done once
        base = rms * dcfg.get("correlated_noise_factor", 1.0)
        m = (df.pointing == pid).to_numpy()
        xi = np.clip(np.round(df.x[m].to_numpy()).astype(int), 0, conv.shape[1] - 1)
        yi = np.clip(np.round(df.y[m].to_numpy()).astype(int), 0, conv.shape[0] - 1)
        nu[m] = conv[yi, xi] / base[yi, xi]                        # peak significance at the centroid
        grid = threshold_grid(np.nanmax(nu[m]) if m.any() else np.nan)
        rows = []
        for k in grid:
            n_pos = n_labels(detect(conv, k * base, dcfg["npixels"], mask=~val))
            n_neg = n_labels(detect(-conv, k * base, dcfg["npixels"], mask=~val))
            rows.append({"nsigma": float(k), "n_pos": n_pos, "n_neg": n_neg,
                         "fpr": float(min(n_neg / max(n_pos, 1), 1.0)),
                         "fpr_floor": float(1.0 / max(n_pos, 1))})
            if n_pos == 0:
                break
        cur = pd.DataFrame(rows)
        ok = cur[cur.fpr <= target_fpr]
        last_neg = cur[cur.n_neg > 0]
        floor_nu = float(last_neg.nsigma.max()) if len(last_neg) else float(cur.nsigma.min())
        curves[pid] = {"curve": cur.to_dict(orient="list"),
                       "operating_nsigma": float(ok.nsigma.iloc[0]) if len(ok) else float(cur.nsigma.iloc[-1]),
                       "fpr_at_configured_nsigma": float(np.interp(dcfg["nsigma"], cur.nsigma, cur.fpr)),
                       "last_threshold_with_a_negative_detection": floor_nu,
                       "resolution_note": "FPR = n_neg / n_pos; above the last threshold with a negative detection "
                                          "the FPR is only known to be below 1 / n_pos at that threshold"}
        ctx.log.info(f"[Stage 3c] {pid}: FPR <= {target_fpr} from {curves[pid]['operating_nsigma']:.2f} sigma; "
                     f"FPR at the detection threshold ({dcfg['nsigma']} sigma) = "
                     f"{curves[pid]['fpr_at_configured_nsigma']:.3f}; no negative detection above "
                     f"{floor_nu:.1f} sigma (FPR resolution there: 1/{int(cur.n_pos[cur.nsigma >= floor_nu].iloc[0])})")

    fpr = np.full(len(df), np.nan)
    for pid, c in curves.items():
        m = (df.pointing == pid).to_numpy()
        fpr[m] = np.interp(nu[m], c["curve"]["nsigma"], c["curve"]["fpr"], right=0.0)
    sub_psf = 1.0 / (1.0 + np.exp((df.fwhm_over_psf.to_numpy(float) - ctx.cfg["baseline"]["sub_psf_factor"]) / 0.05))
    confirm = float(dcfg.get("confirm_snr", 3.0))
    best = pd.to_numeric(df.get("confirm_snr_best", pd.Series(np.nan, index=df.index)), errors="coerce").to_numpy(float)
    p_cr = np.where(np.isfinite(best), 1.0 / (1.0 + np.exp((best - confirm) / 0.5)), 0.0)
    score = 1 - (1 - np.nan_to_num(fpr)) * (1 - np.nan_to_num(sub_psf)) * (1 - p_cr)
    res = pd.DataFrame({"object_id": df.object_id, "peak_significance": np.round(nu, 3),
                        "fpr_at_threshold": np.round(fpr, 5), "p_cosmic_ray": np.round(p_cr, 4),
                        "noise_score": np.round(score, 4)})
    res.to_csv(out / "noise_scores.csv", index=False)
    validation = res.groupby(df.cluster_semantic.to_numpy())["noise_score"].mean().round(3).to_dict()
    snr_by_track = df.groupby("cluster_semantic").snr.median().round(1).to_dict()
    save_json({"target_fpr": target_fpr, "pointings": curves, "validation_mean_noise_score_by_track": validation,
               "median_snr_by_track": snr_by_track,
               "terms": "noise_score = 1 - (1 - FPR(peak significance)) (1 - p_sub_psf) (1 - p_cosmic_ray)"},
              out / "noise_model.json")
    _plot(curves, out)
    ctx.log.info(f"[Stage 3c] mean noise_score by track (validation): {validation} | median SNR by track "
                 f"{snr_by_track} (cosmic rays are high-S/N: the p_cosmic_ray term, not the FPR, scores them)")
    return {"validation": validation}


def _plot(curves, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5.5, 3.5))
    for pid, c in curves.items():
        ax.plot(c["curve"]["nsigma"], c["curve"]["fpr"], "o-", ms=3, label=f"{pid} FPR")
        ax.plot(c["curve"]["nsigma"], c["curve"]["fpr_floor"], ":", lw=1, label=f"{pid} resolution 1/n_pos")
    ax.axhline(0.01, color="k", ls=":", lw=1)
    ax.set_xscale("log")
    ax.set_yscale("symlog", linthresh=1e-4)
    ax.set_xlabel("peak significance threshold [sigma]")
    ax.set_ylabel("false-positive rate (negative image)")
    ax.legend(fontsize=7)
    ax.set_title("Stage 3c: false-positive curve")
    fig.tight_layout()
    fig.savefig(out / "fpr_curve.png", dpi=110)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage4a_dual_branch.py
"""Stage 4a - Dual-branch multi-task training.

Purpose : the core supervised model of the proposal.
Method  : ResNet-18 image branch (512-d, one input channel per band) and a
          Dense(n_features -> 64) projection of the Pass-2 table, each
          LayerNorm-ed so the larger image vector cannot drown the tabular one,
          concatenated into Head A (5-class softmax) and Head B (merger sigmoid).
          Loss = class-weighted focal loss (gamma 2) + 0.5 x positive-weighted BCE.
          AdamW + cosine schedule, 90-degree rotations and flips as label-safe
          augmentation; fixed epoch budget, best dev Macro-F1 checkpoint kept.
          The tabular pre-processing (impute, signed log, RobustScaler) is
          fitted on the train split only.
Output  : dual_branch.pt, tab_prep.joblib, model_meta.json, dataset_index.csv, dev_metrics.json
"""
from __future__ import annotations

import pandas as pd
import torch

from ..context import Ctx, save_json
from ..features import TabularPrep, load_cutouts, make_tensors, supervised_splits
from ..nets import MORPH_CLASSES, DualBranchNet, fit_classifier, morph_metrics, predict_model


def run(ctx: Ctx):
    cfg = ctx.cfg["supervised"]
    out = ctx.stage_dir("4a")
    splits = supervised_splits(ctx)
    cut = load_cutouts(ctx)
    prep = TabularPrep(ctx.cfg["representation"]["tabular_features"]).fit(splits["train"])
    prep.save(out / "tab_prep.joblib")
    data = {s: make_tensors(v, cut, prep) for s, v in splits.items()}
    pd.concat([v.assign(split=s)[["object_id", "split", "gz_label", "gz_label_id", "gz_merger_flag"]]
               for s, v in splits.items()]).to_csv(out / "dataset_index.csv", index=False)
    n_tab, in_ch = data["train"]["tab"].shape[1], cut.shape[1]
    save_json({"n_tab": n_tab, "in_ch": in_ch, "tab_features": prep.kept, "classes": list(MORPH_CLASSES),
               "sizes": {s: len(v) for s, v in splits.items()}}, out / "model_meta.json")
    allrows = pd.concat([v.assign(split=s) for s, v in splits.items()], ignore_index=True)
    counts = pd.crosstab(allrows.split, allrows.gz_label)
    ctx.log.info(f"[Stage 4a] device {ctx.device()} | tabular features {prep.kept}\n" + counts.to_string())

    model = fit_classifier(DualBranchNet(n_tab, in_ch=in_ch, mode="hybrid", pretrained=cfg["pretrained"]),
                           data["train"], data["dev"], ctx.device(), epochs=cfg["epochs"],
                           batch_size=cfg["batch_size"], lr=cfg["lr"], weight_decay=cfg["weight_decay"],
                           gamma=cfg["focal_gamma"], merger_weight=cfg["merger_weight"], seed=ctx.seed,
                           log=ctx.log.info, tag="dual-branch")
    torch.save(model.state_dict(), out / "dual_branch.pt")
    p = predict_model(model, data["dev"], ctx.device())
    m = morph_metrics(data["dev"]["y"].numpy(), p["morph"], data["dev"]["ym"].numpy(), p["merger"])
    save_json(m, out / "dev_metrics.json")
    ctx.log.info(f"[Stage 4a] dual-branch (dev): {m}")
    return m

In [ ]:
%%writefile src/p2p/stages/stage4b_ensemble.py
"""Stage 4b - Ensemble consensus.

Purpose : combine models that make different errors.
Method  : a standalone EfficientNet-B0 (pixels only) and an XGBoost model
          (Pass-2 table only; balanced sample weights, early stopping on dev)
          are trained and soft-voted with the dual-branch network
          (supervised.ensemble_weights). XGBoost runs in its own process (see
          p2p/xgb_worker.py: on macOS it can crash next to PyTorch). The merger probability comes from the
          dual-branch model alone - the other members have no merger head.
Output  : efficientnet.pt, tabular_member.joblib, dev_metrics.json
"""
from __future__ import annotations

import joblib
import torch

from ..context import Ctx, save_json
from ..features import TabularPrep, load_cutouts, make_tensors, supervised_splits
from ..inference import TabularMember, ensemble_predict, load_models
from ..nets import EfficientNetMember, fit_classifier, morph_metrics


def run(ctx: Ctx):
    cfg = ctx.cfg["supervised"]
    out = ctx.stage_dir("4b")
    splits = supervised_splits(ctx)
    cut = load_cutouts(ctx)
    prep = TabularPrep.load(ctx.require("4a", "tab_prep.joblib"))
    data = {s: make_tensors(v, cut, prep) for s, v in splits.items() if s != "test"}

    eff = fit_classifier(EfficientNetMember(in_ch=cut.shape[1], pretrained=cfg["pretrained"]), data["train"],
                         data["dev"], ctx.device(), epochs=cfg["epochs"], batch_size=cfg["batch_size"], lr=cfg["lr"],
                         weight_decay=cfg["weight_decay"], gamma=cfg["focal_gamma"], seed=ctx.seed,
                         log=ctx.log.info, tag="efficientnet")
    torch.save(eff.state_dict(), out / "efficientnet.pt")
    tab = TabularMember(seed=ctx.seed, log=ctx.log.info).fit(data["train"]["tab"].numpy(), data["train"]["y"].numpy(),
                                           data["dev"]["tab"].numpy(), data["dev"]["y"].numpy())
    joblib.dump(tab, out / "tabular_member.joblib")

    models = load_models(ctx)
    ens = ensemble_predict(ctx, data["dev"], models)
    y, ym = data["dev"]["y"].numpy(), data["dev"]["ym"].numpy()
    res = {k: morph_metrics(y, P) for k, P in ens["members"].items()}
    res["ensemble"] = morph_metrics(y, ens["morph"], ym, ens["merger"])
    res["tabular_member_kind"] = tab.kind
    save_json(res, out / "dev_metrics.json")
    for k in ("dual", "effnet", "tabular", "ensemble"):
        ctx.log.info(f"[Stage 4b] {k:9s} dev Macro-F1 = {res[k]['macro_f1']:.3f}")
    return {"ensemble_dev_macro_f1": res["ensemble"]["macro_f1"]}

In [ ]:
%%writefile src/p2p/stages/stage4c_calibration.py
"""Stage 4c - Probability calibration (temperature scaling).

Purpose : make predicted confidence match empirical accuracy.
Method  : one temperature T for the soft-voted morphology log-probabilities
          (T < 1 sharpens - an averaged ensemble is usually under-confident) and
          Platt scaling p = sigmoid(a z + b) for the merger logit (the positive-class
          weight used in training shifts the logits; a temperature cannot remove a
          shift). Both are fitted on the DEV split by minimising the NLL. Reported with the Expected
          Calibration Error (ECE) before and after.
Output  : temperature.json, reliability_dev.png
"""
from __future__ import annotations

import numpy as np
from scipy.optimize import minimize_scalar

from ..context import Ctx, save_json
from ..features import TabularPrep, load_cutouts, make_tensors, supervised_splits
from ..inference import ensemble_predict, load_models
from ..nets import logit, softmax_np
from ..utils import expected_calibration_error


def fit_platt(z, y):
    """p = sigmoid(a z + b): a re-scales AND shifts the logit. The merger head is
    trained with a positive-class weight, which shifts every logit up; a single
    temperature cannot undo a shift, Platt scaling can."""
    from scipy.optimize import minimize
    y = np.asarray(y, float)

    def nll(ab):
        p = 1 / (1 + np.exp(-(ab[0] * z + ab[1])))
        return -np.mean(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12))
    r = minimize(nll, x0=[1.0, 0.0], method="Nelder-Mead")
    return float(r.x[0]), float(r.x[1])


def fit_temperature(logits, y):
    y = np.asarray(y, int)
    nll = lambda lt: -np.mean(np.log(softmax_np(logits / np.exp(lt))[np.arange(len(y)), y] + 1e-12))
    return float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method="bounded").x))


def run(ctx: Ctx):
    out = ctx.stage_dir("4c")
    splits = supervised_splits(ctx)
    prep = TabularPrep.load(ctx.require("4a", "tab_prep.joblib"))
    dev = make_tensors(splits["dev"], load_cutouts(ctx), prep)
    ens = ensemble_predict(ctx, dev, load_models(ctx))
    y, ym = dev["y"].numpy(), dev["ym"].numpy().astype(int)
    L = np.log(np.clip(ens["morph"], 1e-12, 1))
    T_m = fit_temperature(L, y)
    zm = logit(ens["merger"])
    a, b = fit_platt(zm, ym) if 0 < ym.sum() < len(ym) else (1.0, 0.0)
    Pc = softmax_np(L / T_m)
    two = lambda p: np.column_stack([1 - p, p])
    pm_c = 1 / (1 + np.exp(-(a * zm + b)))
    rep = {"T_morph": round(T_m, 4), "merger_platt_a": round(a, 4), "merger_platt_b": round(b, 4),
           "dev_ece_morph_before": round(expected_calibration_error(ens["morph"], y), 4),
           "dev_ece_morph_after": round(expected_calibration_error(Pc, y), 4),
           "dev_ece_merger_before": round(expected_calibration_error(two(ens["merger"]), ym), 4),
           "dev_ece_merger_after": round(expected_calibration_error(two(pm_c), ym), 4), "n_dev": int(len(y)),
           "dev_merger_rate_true": round(float(ym.mean()), 4),
           "dev_merger_rate_predicted_before": round(float((ens["merger"] >= 0.5).mean()), 4),
           "dev_merger_rate_predicted_after": round(float((pm_c >= 0.5).mean()), 4)}
    # decision threshold for the binary merger FLAG: flag the same fraction of dev galaxies as are
    # true mergers (prevalence matching). The probability p_merger itself stays calibrated.
    if 0 < ym.sum() < len(ym):
        thr = float(np.quantile(pm_c, 1 - ym.mean()))
        flag = pm_c >= thr
        rep["merger_threshold"] = round(thr, 4)
        rep["dev_merger_precision_at_threshold"] = round(float((flag & (ym == 1)).sum() / max(flag.sum(), 1)), 4)
        rep["dev_merger_recall_at_threshold"] = round(float((flag & (ym == 1)).sum() / ym.sum()), 4)
    else:
        rep["merger_threshold"] = 0.5
    save_json(rep, out / "temperature.json")
    _plot(ens["morph"], Pc, y, out)
    ctx.log.info(f"[Stage 4c] {rep}")
    return rep


def _plot(P0, P1, y, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(4.5, 4))
    for P, lab in ((P0, "before"), (P1, "after")):
        conf, acc = P.max(1), (P.argmax(1) == y)
        idx = np.digitize(conf, np.linspace(0, 1, 11)) - 1
        xs = [conf[idx == b].mean() for b in range(10) if (idx == b).any()]
        ys = [acc[idx == b].mean() for b in range(10) if (idx == b).any()]
        ax.plot(xs, ys, "o-", label=f"{lab} T-scaling")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("confidence")
    ax.set_ylabel("accuracy")
    ax.set_title("Stage 4c: dev reliability")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out / "reliability_dev.png", dpi=110)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage4d_ablation.py
"""Stage 4d - Controlled ablation.

Purpose : measure the value the hybrid design adds (Objective 3).
Method  : CNN-only and classical-only models are the SAME DualBranchNet with
          one branch removed, trained with the same recipe and epoch budget
          (the tiny tabular-only net gets more epochs: that can only strengthen
          the baseline). All rows are scored on DEV - the test split is used
          once, in Stage 6a. The segregation rows compare the Stage-2c baseline
          with the learned CAE + clustering (Stage 2d).
Output  : ablation_table.csv, ablation.png
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..context import Ctx, load_json, save_json
from ..features import TabularPrep, load_cutouts, make_tensors, supervised_splits
from ..inference import calibrated_predict, load_models
from ..nets import DualBranchNet, fit_classifier, morph_metrics, predict_model
from ..utils import expected_calibration_error


def run(ctx: Ctx):
    cfg = ctx.cfg["supervised"]
    out = ctx.stage_dir("4d")
    splits = supervised_splits(ctx)
    cut = load_cutouts(ctx)
    prep = TabularPrep.load(ctx.require("4a", "tab_prep.joblib"))
    data = {s: make_tensors(v, cut, prep) for s, v in splits.items() if s != "test"}
    meta = load_json(ctx.require("4a", "model_meta.json"))
    common = dict(batch_size=cfg["batch_size"], weight_decay=cfg["weight_decay"], gamma=cfg["focal_gamma"],
                  merger_weight=cfg["merger_weight"], seed=ctx.seed, log=ctx.log.info)
    img_only = fit_classifier(DualBranchNet(meta["n_tab"], in_ch=meta["in_ch"], mode="image", pretrained=cfg["pretrained"]),
                              data["train"], data["dev"], ctx.device(), epochs=cfg["epochs"], lr=cfg["lr"],
                              tag="cnn-only", **common)
    tab_only = fit_classifier(DualBranchNet(meta["n_tab"], in_ch=meta["in_ch"], mode="tabular"),
                              data["train"], data["dev"], ctx.device(), epochs=cfg["tabular_only_epochs"], lr=3e-3,
                              tag="classical-only", **common)
    models = load_models(ctx)
    dev = data["dev"]
    y, ym = dev["y"].numpy(), dev["ym"].numpy()
    rows = []

    def add(name, rep, P, pm=None):
        m = morph_metrics(y, P, ym, pm)
        rows.append({"task": "morphology (dev)", "configuration": name, "representation": rep,
                     "macro_f1": m["macro_f1"], "balanced_acc": m["balanced_acc"],
                     "merger_pr_auc": m.get("merger_pr_auc", np.nan),
                     "ece": round(expected_calibration_error(P, y), 4)})

    for name, rep, mdl in (("Classical-only (Dense branch)", "tabular", tab_only),
                           ("CNN-only (ResNet-18 branch)", "image", img_only),
                           ("Hybrid fusion (ResNet-18 + Dense)", "image+tabular", models["dual"])):
        p = predict_model(mdl, dev, ctx.device())
        add(name, rep, p["morph"], p.get("merger"))
    add("XGBoost member (tabular)", "tabular", models["tabular"].predict_proba(dev["tab"].numpy()))
    add("EfficientNet member (image)", "image", predict_model(models["effnet"], dev, ctx.device())["morph"])
    cal = calibrated_predict(ctx, dev, models)
    add("Full ensemble (soft vote + T-scaling)", "ensemble", cal["morph"], cal["merger"])
    audit = load_json(ctx.require("2d", "semantic_audit.json"))
    for name, key in (("Baseline segregator (Stage 2c, no Gaia leg)", "baseline_star_vs_gaia"),
                      ("CAE + GMM/HDBSCAN (Stage 2d)", "learned_star_vs_gaia")):
        m = audit[key]
        rows.append({"task": "star segregation vs Gaia", "configuration": name, "star_precision": m["precision"],
                     "star_recall": m["recall"], "star_f1": m["f1"]})
    tab = pd.DataFrame(rows)
    tab.to_csv(out / "ablation_table.csv", index=False)
    f = tab.set_index("configuration").macro_f1
    delta = float(f["Hybrid fusion (ResNet-18 + Dense)"] - max(f["Classical-only (Dense branch)"],
                                                              f["CNN-only (ResNet-18 branch)"]))
    tier = "Stretch" if delta >= 0.05 else "Target" if delta >= 0.03 else "Minimum" if delta >= 0 else "Below minimum"
    save_json({"hybrid_delta_macro_f1": round(delta, 4), "tier": tier}, out / "summary.json")
    _plot(tab[tab.task == "morphology (dev)"], out)
    ctx.log.info("[Stage 4d]\n" + tab.to_string(index=False))
    ctx.log.info(f"[Stage 4d] hybrid minus best single branch = {delta:+.4f} -> {tier}")
    return {"hybrid_delta": round(delta, 4), "tier": tier}


def _plot(t, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    t = t.sort_values("macro_f1")
    col = ["#2E5496" if "Hybrid" in c else "#9AA3B5" if "branch" in c else "#D3D9E5" for c in t.configuration]
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.barh(t.configuration, t.macro_f1, color=col)
    for i, v in enumerate(t.macro_f1):
        ax.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=8)
    ax.set_xlim(0, 1.1)
    ax.set_xlabel("dev Macro-F1")
    ax.set_title("Stage 4d: controlled ablation (blue = hybrid, dark grey = its single branches)")
    fig.tight_layout()
    fig.savefig(out / "ablation.png", dpi=110)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage5a_empirical_psf.py
"""Stage 5a - Empirical PSF construction (photutils EPSFBuilder).

Purpose : a localised, high-quality PSF built from the pipeline's own stars.
Method  : the stars routed by Stage 3b are cut to 31x31 stamps from the image
          that is actually measured (the sum of the PSF-matched bands) and
          stacked with photutils.psf.EPSFBuilder (oversampling 2, sub-pixel
          recentring, sigma-clipped). Pointings with too few stars pool the
          stars of all pointings with the same instrument; if that is still too
          few, the Pass-1 placeholder PSF is kept (and the report says so).
Output  : epsf_<pointing>.npy (+ .fits), epsf_report.json, epsf.png
"""
from __future__ import annotations

import warnings

import numpy as np
import pandas as pd
from astropy.io import fits

from ..context import Ctx, load_json, save_json
from ..features import load_cutouts
from ..morphology import morph_image
from ..utils import as_bool, fwhm_fit
from .stage1d_psf_match import core_centroid, crop_center, stack_stars, star_masks

try:
    from photutils.psf import EPSFBuilder, EPSFStar, EPSFStars
    _HAVE_EPSF = True
except Exception:
    _HAVE_EPSF = False


def build_epsf(stamps, masks=None, oversampling=2, maxiters=5):
    """masks: True = neighbour / cosmic-ray pixel (weight 0). Stars are centred on their core centroid."""
    masks = masks if masks is not None else [np.zeros_like(s, bool) for s in stamps]
    if _HAVE_EPSF:
        try:
            stars = EPSFStars([EPSFStar(np.where(m, 0.0, s).astype(float), weights=(~m).astype(float),
                                        cutout_center=core_centroid(np.where(m, 0.0, s))[::-1])
                               for s, m in zip(stamps, masks)])
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                res = EPSFBuilder(oversampling=oversampling, maxiters=maxiters, progress_bar=False)(stars)
            epsf = res.epsf if hasattr(res, "epsf") else res[0]
            n = stamps[0].shape[0]
            yy, xx = np.mgrid[:n, :n]
            psf = np.clip(epsf.evaluate(xx, yy, flux=1.0, x_0=(n - 1) / 2, y_0=(n - 1) / 2), 0, None)
            return psf / psf.sum(), "photutils.EPSFBuilder"
        except Exception as e:
            print(f"[Stage 5a] EPSFBuilder failed ({e}); median stack used")
    return stack_stars(stamps, masks=masks), "median stack"


def run(ctx: Ctx, size=31, min_stars=5):
    out = ctx.stage_dir("5a")
    pts = load_json(ctx.require("1a", "pointings.json"))["pointings"]
    routing = pd.read_csv(ctx.require("3b", "star_routing.csv"))
    src = pd.read_csv(ctx.require("1c", "sources.csv"))[["object_id", "cutout_index", "pointing"]]
    routed = routing[as_bool(routing.used_in_epsf)].merge(src, on=["object_id", "pointing"])
    cut = load_cutouts(ctx)
    segs = np.load(ctx.require("1c", "segstamps.npy"), mmap_mode="r")
    match = load_json(ctx.require("1d", "psf_match.json"))
    stamps = {r.object_id: crop_center(morph_image(cut[r.cutout_index]), size) for r in routed.itertuples()}
    report = {}
    for pid, p in pts.items():
        own = routed[routed.pointing == pid].object_id.tolist()
        use, how = own, "own stars"
        if len(own) < min_stars:
            same = [q for q, v in pts.items() if v["instrument"] == p["instrument"]]
            use = routed[routed.pointing.isin(same)].object_id.tolist()
            how = f"pooled over {len(same)} pointing(s) of {p['instrument']}"
        if len(use) >= min_stars:
            idx = routed.set_index("object_id").loc[use, "cutout_index"].tolist()
            psf, method = build_epsf([stamps[o] for o in use], star_masks(segs, idx, size))
        else:
            ph = np.load(ctx.require("1d", f"placeholder_psf_{pid}.npy"))
            psf = np.zeros((size, size))
            h = (size - ph.shape[0]) // 2
            psf[h:h + ph.shape[0], h:h + ph.shape[1]] = ph
            method, how = "Pass-1 placeholder kept", f"only {len(use)} star(s) available"
        np.save(out / f"epsf_{pid}.npy", psf.astype(np.float32))
        fits.writeto(out / f"epsf_{pid}.fits", psf.astype(np.float32), overwrite=True)
        report[pid] = {"n_stars": len(use), "source": how, "method": method, "fwhm_px": round(fwhm_fit(psf), 3)}
        # the ePSF is built on PSF-MATCHED cutouts, so it cannot be sharper than the matched PSF, and a much
        # broader one means galaxies or cosmic rays got into the stack
        m = match.get(pid, {})
        ref = m.get("fwhm_px", {}).get(m.get("target_band")) if m.get("target_psf_measured") else None
        if ref:
            ratio = report[pid]["fwhm_px"] / ref
            report[pid]["fwhm_over_matched"] = round(ratio, 3)
            if ratio > 1.5 or ratio < 0.85:
                ctx.log.info(f"[Stage 5a] {pid}: WARNING - ePSF FWHM {report[pid]['fwhm_px']} px is {ratio:.2f} x the "
                             f"Stage-1d matched FWHM {ref} px (expected ~1): check the routed stars "
                             "(outputs/stage3b_star_routing/star_routing.csv).")
        ctx.log.info(f"[Stage 5a] {pid}: {method} from {len(use)} stars ({how}); FWHM {report[pid]['fwhm_px']} px")
    save_json(report, out / "epsf_report.json")
    _plot(report, out)
    return {"pointings": len(report)}


def _plot(report, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    pids = list(report)
    fig, axes = plt.subplots(1, len(pids), figsize=(2.6 * len(pids), 2.8))
    for ax, pid in zip(np.atleast_1d(axes), pids):
        ax.imshow(np.log10(np.load(out / f"epsf_{pid}.npy") + 1e-5), origin="lower", cmap="magma")
        ax.set_title(f"{pid}: {report[pid]['n_stars']} stars\nFWHM {report[pid]['fwhm_px']} px", fontsize=8)
        ax.axis("off")
    fig.suptitle("Stage 5a: empirical PSFs (log scale)")
    fig.tight_layout()
    fig.savefig(out / "epsf.png", dpi=100)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage5b_pass2_refinement.py
"""Stage 5b - Deterministic Pass-2 refinement.

Purpose : physics-calibrated morphology for the galaxy track.
Method  : the empirical PSF from Stage 5a is fed back into the SAME statmorph
          measurement (same image, same segmentation, same gain) and the galaxy
          track is re-measured exactly once:
            psf_fit    (default) the Sersic model is convolved with the ePSF, so
                       n and r_eff are PSF-corrected; non-parametric indices are
                       re-measured on the same image;
            deconvolve Richardson-Lucy deconvolution (10 fixed iterations)
                       inside the dilated footprint, then statmorph.
          Split tags and cluster IDs are hashed before and after: Pass 2 is not
          allowed to change them.
Output  : features_pass2.csv (psf_pass = 2, galaxy rows only - load_objects()
          lets these overwrite the Pass-1 values), morph_delta_pass.csv
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from ..context import Ctx, load_objects, save_json
from ..morphology import FEATURES, run_parallel

FROZEN = ["object_id", "split_tag", "cluster_semantic", "cluster_id_gmm", "is_transitional", "anomaly_flag"]


def _hash(df):
    return int(pd.util.hash_pandas_object(df[FROZEN].astype(str), index=False).sum())


def run(ctx: Ctx):
    out = ctx.stage_dir("5b")
    mode = ctx.cfg["morphology"]["pass2_mode"]
    df = load_objects(ctx, ["sources", "pass1", "clusters", "segregation"])
    before = _hash(df)
    gal = df[(df.cluster_semantic == "Galaxy") & (df.anomaly_flag == 0) & (df.snr >= ctx.cfg["morphology"]["min_snr"])]
    psf = {p: np.load(ctx.require("5a", f"epsf_{p}.npy")) for p in gal.pointing.unique()}
    ctx.log.info(f"[Stage 5b] Pass-2 ({mode}) on {len(gal)} galaxies with {ctx.n_jobs} worker(s) ...")
    res = run_parallel(ctx, ctx.require("1d", "cutouts.npy"), ctx.require("1c", "segstamps.npy"),
                       gal.cutout_index.tolist(), gal.exptime.tolist(), [psf[p] for p in gal.pointing], mode=mode)
    p2 = pd.DataFrame(res)[FEATURES]
    p2.insert(0, "object_id", gal.object_id.to_numpy())
    p2["psf_pass"] = 2
    p2["pass2_mode"] = mode
    p2.to_csv(out / "features_pass2.csv", index=False)
    assert _hash(df) == before, "Pass 2 changed a frozen column (split / cluster)!"

    p1 = gal.set_index("object_id")[FEATURES]
    d = (p2.set_index("object_id")[FEATURES] - p1).add_suffix("_delta")
    d.reset_index().to_csv(out / "morph_delta_pass.csv", index=False)
    med = d[[c for c in d.columns if c.split("_delta")[0] in ("gini", "m20", "concentration", "asymmetry",
                                                               "sersic_n", "sersic_rhalf", "r_half")]].median()
    save_json({"mode": mode, "galaxies": len(gal), "median_delta": med.round(4).to_dict(),
               "frozen_columns_unchanged": True}, out / "summary.json")
    ctx.log.info("[Stage 5b] median Pass-2 minus Pass-1:\n" + med.round(4).to_string())
    return {"galaxies": len(gal)}

In [ ]:
%%writefile src/p2p/stages/stage6a_test_eval.py
"""Stage 6a - In-domain test (run once).

Purpose : the final, honest in-domain score.
Method  : the frozen, calibrated ensemble is run on the untouched GZH test
          split: Macro-F1, balanced accuracy, per-class precision/recall,
          one-vs-rest PR-AUC, merger PR-AUC and ECE, each with a 95% bootstrap
          interval (small test sets need error bars), scored against the tiered
          success criteria. Every run is appended to a ledger, so the report can
          state how many times the test split was looked at.
Output  : test_metrics.json, test_predictions.csv, confusion_matrix_test.png, test_split_ledger.json
"""
from __future__ import annotations

import datetime

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix, f1_score

from ..context import Ctx, load_json, save_json
from ..features import TabularPrep, load_cutouts, make_tensors, supervised_splits
from ..inference import calibrated_predict, load_models
from ..nets import MORPH_CLASSES, N_CLASSES, morph_metrics
from ..utils import bootstrap_ci, expected_calibration_error


def tier(v, minimum, target, stretch, lower_better=False):
    if v is None or not np.isfinite(v):
        return "n/a"
    if lower_better:
        v, minimum, target, stretch = -v, -minimum, -target, -stretch
    return "Stretch" if v >= stretch else "Target" if v >= target else "Minimum" if v > minimum else "Below minimum"


def _ci(c):
    return f"{c[0]:.3f}-{c[1]:.3f}" if c and np.isfinite(c[0]) else "n/a"


def run(ctx: Ctx):
    out = ctx.stage_dir("6a")
    ledger_p = out / "test_split_ledger.json"
    ledger = load_json(ledger_p) if ledger_p.exists() else {"evaluations": []}
    if ledger["evaluations"]:
        ctx.log.info(f"[Stage 6a] WARNING: the test split was already evaluated {len(ledger['evaluations'])} time(s). "
                     "Report the FIRST result and do not tune on these numbers.")
    splits = supervised_splits(ctx)
    prep = TabularPrep.load(ctx.require("4a", "tab_prep.joblib"))
    test = make_tensors(splits["test"], load_cutouts(ctx), prep)
    res = calibrated_predict(ctx, test, load_models(ctx))
    y, ym, P = test["y"].numpy(), test["ym"].numpy().astype(int), res["morph"]
    pred = P.argmax(1)
    maj = np.bincount(splits["train"].gz_label_id, minlength=N_CLASSES).argmax()
    maj_f1 = morph_metrics(y, np.eye(N_CLASSES)[np.full(len(y), maj)])["macro_f1"]
    m = morph_metrics(y, P, ym, res["merger"])
    f1 = lambda yy, PP: f1_score(yy, PP.argmax(1), average="macro",
                                 labels=sorted(set(yy.tolist()) | set(PP.argmax(1).tolist())), zero_division=0)
    m["macro_f1_ci95"] = bootstrap_ci(f1, y, P, seed=ctx.seed)
    present = np.eye(N_CLASSES)[y].sum(0) > 0
    m["morph_pr_auc_ovr"] = round(float(average_precision_score(np.eye(N_CLASSES)[y][:, present], P[:, present],
                                                                 average="macro")), 4)
    m["ece"] = round(expected_calibration_error(P, y), 4)
    m["ece_ci95"] = bootstrap_ci(lambda yy, PP: expected_calibration_error(PP, yy), y, P, seed=ctx.seed)
    if m.get("merger_pr_auc") is not None and 0 < ym.sum() < len(ym):
        ap = lambda a, p: average_precision_score(a, p) if 0 < a.sum() < len(a) else float("nan")
        lo, hi = bootstrap_ci(ap, ym, np.asarray(res["merger"], float), seed=ctx.seed)
        m["merger_pr_auc_ci95"] = (lo, hi)
        m["n_merger_test"] = int(ym.sum())
    m["majority_class_macro_f1"] = maj_f1
    m["per_class"] = classification_report(y, pred, labels=list(range(N_CLASSES)), target_names=list(MORPH_CLASSES),
                                           output_dict=True, zero_division=0)
    m["tiers"] = {"Morphology Macro-F1": tier(m["macro_f1"], maj_f1, 0.65, 0.75),
                  "Merger PR-AUC": tier(m.get("merger_pr_auc"), m.get("merger_prevalence", 0), 0.50, 0.65),
                  "Calibration ECE": tier(m["ece"], 0.10, 0.05, 0.03, lower_better=True)}
    save_json(m, out / "test_metrics.json")
    ledger["evaluations"].append({"utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
                                  "macro_f1": m["macro_f1"], "n_test": int(len(y))})
    save_json(ledger, ledger_p)
    pr = pd.DataFrame({"object_id": test["object_id"], "y_true": y, "y_pred": pred, "p_merger": res["merger"]})
    for k, c in enumerate(MORPH_CLASSES):
        pr[f"p_{c}"] = P[:, k]
    pr.to_csv(out / "test_predictions.csv", index=False)
    _confusion(y, pred, out)
    ctx.log.info(f"[Stage 6a] test n={len(y)}: Macro-F1 {m['macro_f1']} (95% CI {m['macro_f1_ci95'][0]:.3f}-"
                 f"{m['macro_f1_ci95'][1]:.3f}) | balanced acc {m['balanced_acc']} | merger PR-AUC "
                 f"{m.get('merger_pr_auc')} (95% CI {_ci(m.get('merger_pr_auc_ci95'))}, "
                 f"{m.get('n_merger_test', 0)} mergers) | ECE {m['ece']} (95% CI {_ci(m['ece_ci95'])}) | "
                 f"majority-class F1 {maj_f1}")
    ctx.log.info(f"[Stage 6a] tiers: {m['tiers']}")
    return {"macro_f1": m["macro_f1"], "tiers": m["tiers"]}


def _confusion(y, pred, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    cm = confusion_matrix(y, pred, labels=list(range(N_CLASSES)))
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(N_CLASSES))
    ax.set_xticklabels(MORPH_CLASSES, rotation=35, ha="right")
    ax.set_yticks(range(N_CLASSES))
    ax.set_yticklabels(MORPH_CLASSES)
    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            ax.text(j, i, f"{cmn[i, j]:.2f}\n({cm[i, j]})", ha="center", va="center", fontsize=7,
                    color="white" if cmn[i, j] > 0.5 else "black")
    ax.set_xlabel("predicted")
    ax.set_ylabel("GZH label")
    ax.set_title("Stage 6a: in-domain test (row-normalised, counts)")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(out / "confusion_matrix_test.png", dpi=110)
    plt.close(fig)

In [ ]:
%%writefile src/p2p/stages/stage6b_uncertainty.py
"""Stage 6b - Uncertainty audit.

Purpose : does the model hesitate where humans disagree?
Method  : normalised predictive entropy of each test galaxy is compared with
          human disagreement (1 - the weakest GZH vote fraction along the path
          to its label): Spearman correlation, plus a one-sided Mann-Whitney U
          test that the Ambiguous Boundary set (agreement < 0.5) has higher
          entropy than confidently labelled galaxies.
Output  : uncertainty_audit.json, uncertainty_audit.png
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, spearmanr

from ..context import Ctx, save_json
from ..features import supervised_splits
from ..nets import MORPH_CLASSES
from ..utils import as_bool, predictive_entropy


def run(ctx: Ctx):
    out = ctx.stage_dir("6b")
    pred = pd.read_csv(ctx.require("6a", "test_predictions.csv"))
    test = supervised_splits(ctx)["test"][["object_id", "human_disagreement", "in_boundary_set"]]
    d = pred.merge(test, on="object_id")
    P = d[[f"p_{c}" for c in MORPH_CLASSES]].to_numpy()
    e = predictive_entropy(P)
    h = d.human_disagreement.to_numpy(float)
    b = as_bool(d.in_boundary_set).to_numpy()
    rho, p = spearmanr(e, h) if len(d) > 2 else (np.nan, np.nan)
    acc = (d.y_true == d.y_pred).to_numpy()
    res = {"n_test": int(len(d)), "spearman_entropy_vs_disagreement": round(float(rho), 4),
           "spearman_p": float(p), "n_boundary": int(b.sum()), "n_confident": int((~b).sum()),
           "mean_entropy_boundary": float(e[b].mean()) if b.any() else None,
           "mean_entropy_confident": float(e[~b].mean()) if (~b).any() else None,
           "accuracy_boundary": float(acc[b].mean()) if b.any() else None,
           "accuracy_confident": float(acc[~b].mean()) if (~b).any() else None}
    if b.sum() >= 3 and (~b).sum() >= 3:
        res["mannwhitney_p_boundary_gt_confident"] = float(mannwhitneyu(e[b], e[~b], alternative="greater").pvalue)
    else:
        res["note"] = "fewer than 3 boundary-set galaxies in the test split - the U-test is not run"
    save_json(res, out / "uncertainty_audit.json")
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(4.8, 4))
    ax.scatter(h[~b], e[~b], s=12, alpha=0.6, label="confident (humans agree)")
    ax.scatter(h[b], e[b], s=18, marker="x", label="Ambiguous Boundary set")
    ax.set_xlabel("human disagreement (1 - GZH path agreement)")
    ax.set_ylabel("model predictive entropy")
    ax.set_title(f"Stage 6b: Spearman rho = {rho:.2f}")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(out / "uncertainty_audit.png", dpi=110)
    plt.close(fig)
    ctx.log.info(f"[Stage 6b] {res}")
    return res

In [ ]:
%%writefile src/p2p/stages/stage6c_xai.py
"""Stage 6c - Explainable AI: Grad-CAM and SHAP.

Purpose : show what drives each prediction.
Method  : Grad-CAM on the last convolutional block (layer4) of the dual-branch
          ResNet-18, for the most confident correct test galaxy of each class;
          exact TreeSHAP attributions from XGBoost's built-in pred_contribs
          (mean |SHAP| per class and feature).
Output  : gradcam_examples.png, shap_summary.csv, shap_summary.png
"""
from __future__ import annotations

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from ..context import Ctx, save_json
from ..features import TabularPrep, load_cutouts, make_tensors, supervised_splits
from ..inference import load_models
from ..nets import MORPH_CLASSES, N_CLASSES


class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        self.h = [layer.register_forward_hook(lambda m, i, o: setattr(self, "acts", o)),
                  layer.register_full_backward_hook(lambda m, gi, go: setattr(self, "grads", go[0]))]

    def __call__(self, x_img, x_tab, device):
        self.model.to(device).eval()
        x_img, x_tab = x_img.to(device), x_tab.to(device)
        with torch.enable_grad():
            logits = self.model(x_img, x_tab)["morph"]
            k = int(logits.argmax(1))
            self.model.zero_grad()
            logits[0, k].backward()
        cam = F.relu((self.grads.mean(dim=(2, 3), keepdim=True) * self.acts).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x_img.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = cam.detach().cpu().numpy()
        return cam / (cam.max() + 1e-12), k

    def remove(self):
        for h in self.h:
            h.remove()


def run(ctx: Ctx):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    out = ctx.stage_dir("6c")
    models = load_models(ctx)
    pred = pd.read_csv(ctx.require("6a", "test_predictions.csv"))
    test_df = supervised_splits(ctx)["test"]
    data = make_tensors(test_df, load_cutouts(ctx), TabularPrep.load(ctx.require("4a", "tab_prep.joblib")))
    P = pred[[f"p_{c}" for c in MORPH_CLASSES]].to_numpy()
    picks = []
    for k in range(N_CLASSES):
        ok = np.where((pred.y_pred == k) & (pred.y_true == k))[0]
        if ok.size:
            picks.append(ok[np.argmax(P[ok, k])])
    cam = GradCAM(models["dual"], models["dual"].backbone.layer4)
    if picks:
        fig, axes = plt.subplots(2, len(picks), figsize=(2.4 * len(picks), 5), squeeze=False)
        for j, i in enumerate(picks):
            heat, k = cam(data["img"][i:i + 1], data["tab"][i:i + 1], ctx.device())
            base = data["img"][i].numpy().mean(0)
            axes[0, j].imshow(base, origin="lower", cmap="gray")
            axes[0, j].set_title(MORPH_CLASSES[k], fontsize=9)
            axes[1, j].imshow(base, origin="lower", cmap="gray")
            axes[1, j].imshow(heat, origin="lower", cmap="jet", alpha=0.45)
            for a in axes[:, j]:
                a.axis("off")
        fig.suptitle("Stage 6c: Grad-CAM (dual-branch ResNet-18, layer4)")
        fig.tight_layout()
        fig.savefig(out / "gradcam_examples.png", dpi=100)
        plt.close(fig)
    cam.remove()

    tab = models["tabular"]
    names = models["prep"].kept
    X = data["tab"].numpy()
    if tab.kind == "xgboost":
        c = tab.contribs(X)                   # (n, classes, features), computed in XGBoost's own process
        idx = [MORPH_CLASSES[k] for k in tab.classes_] if c.shape[1] == len(tab.classes_) else ["all classes"]
        shap = pd.DataFrame(np.abs(c).mean(0), index=idx, columns=names)
        method = "XGBoost TreeSHAP (pred_contribs)"
    elif tab.kind == "sklearn-hgb":
        from sklearn.inspection import permutation_importance
        remap = {c: i for i, c in enumerate(tab.classes_)}
        yt = test_df.gz_label_id.map(remap)
        ok = yt.notna().to_numpy()
        pi = permutation_importance(tab.model, X[ok], yt[ok].astype(int), n_repeats=10, random_state=ctx.seed)
        shap = pd.DataFrame([pi.importances_mean], index=["all classes"], columns=names)
        method = "permutation importance"
    else:                                     # a constant member has nothing to explain
        shap = pd.DataFrame([np.zeros(len(names))], index=["all classes"], columns=names)
        method = "none (tabular member is constant: one class in the training data)"
    shap.to_csv(out / "shap_summary.csv")
    ax = shap.T.plot.barh(figsize=(6.5, 0.35 * len(names) + 1.5), width=0.8)
    ax.set_xlabel("mean |SHAP| (log-odds)")
    ax.set_title("Stage 6c: tabular attributions per class")
    plt.tight_layout()
    plt.savefig(out / "shap_summary.png", dpi=110)
    plt.close()
    save_json({"gradcam_classes": [MORPH_CLASSES[int(pred.y_pred[i])] for i in picks], "attribution": method},
              out / "summary.json")
    ctx.log.info(f"[Stage 6c] Grad-CAM for {len(picks)} classes; {method}:\n" + shap.round(3).to_string())
    return {"attribution": method}

In [ ]:
%%writefile src/p2p/stages/stage7a_taxonomy.py
"""Stage 7a - Taxonomy alignment and the human-agreement ceiling.

Purpose : separate label-scheme differences from model failure.
Method  : galaxies in the EXTERNAL (CANDELS) pointings - or, with a single field,
          in the held-out TEST split (partition.phase7_region: auto) - are matched to the
          Kartaltepe et al. (2015) expert catalogue. Both schemes are mapped to
          a shared coarse taxonomy (Spheroid, Disk, Irregular, Unclassifiable;
          face-on and edge-on discs both become Disk; expert point sources are
          dropped). On galaxies labelled by BOTH GZH and the experts, the
          alignment matrix and the Macro-F1 of GZH vs expert labels give the
          Taxonomy Ceiling - the best any GZH-trained model could score.
          Leakage guards: no evaluated object may lie within 1" of an object
          on the training side (train/dev, plus test when external fields exist).
Output  : candels_objects.csv, taxonomy_alignment_matrix.csv, taxonomy_ceiling.json
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, f1_score

from ..catalogs import load_candels
from ..context import Ctx, StageSkipped, load_json, save_json
from ..features import galaxy_table
from ..utils import match_with_offset, sky_crossmatch

K15 = ("spheroid", "disk", "irregular", "point_source", "unclassifiable")
COMMON = ("Spheroid", "Disk", "Irregular", "Unclassifiable")
GZH_TO_COMMON = {"Smooth": "Spheroid", "FaceOnSpiral": "Disk", "EdgeOn": "Disk", "Irregular": "Irregular",
                 "Ambiguous": "Unclassifiable"}
K15_TO_COMMON = {"spheroid": "Spheroid", "disk": "Disk", "irregular": "Irregular", "unclassifiable": "Unclassifiable",
                 "point_source": None}


def phase7_region(ctx: Ctx, df) -> str:
    """Which galaxies Phase 7 evaluates on. 'external' = separate CANDELS pointings (proposal design).
    With ONE field there are none, so 'auto' falls back to the TEST split: those galaxies were never
    used for training, tuning or calibration (sky-region split), so the evaluation is still leakage-free;
    it measures transfer to the expert label scheme on held-out sky, with the same depth and PSF."""
    want = str(ctx.cfg["partition"].get("phase7_region", "auto")).lower()
    if want in ("external", "test"):
        return want
    return "external" if (df.split_tag == "external").any() else "test"


def run(ctx: Ctx):
    out = ctx.stage_dir("7a")
    df = galaxy_table(ctx)
    region = phase7_region(ctx, df)
    ext = df[df.split_tag == region].copy()
    if ext.empty:
        raise StageSkipped(f"No galaxies in the '{region}' region for Phase 7. Add a CANDELS field to data/raw and "
                           "list a filename/TARGNAME substring in partition.external_patterns.")
    if region == "test":
        ctx.log.info("[Stage 7a] no external pointing: Phase 7 uses the held-out TEST split "
                     "(partition.phase7_region: auto). Galaxies there were never trained on.")
    metas = list(load_json(ctx.require("1a", "manifest.json")).values())
    k15 = load_candels(ctx, metas)
    if k15 is None or k15.empty:
        raise StageSkipped("No CANDELS (Kartaltepe et al. 2015) table in data/catalogs covering the field.")
    inner = df[df.split_tag.isin([s for s in ("train", "dev", "test") if s != region])]
    _, _, leak = sky_crossmatch(ext.ra, ext.dec, inner.ra, inner.dec, 1.0)
    assert not leak.any(), f"{region} object within 1 arcsec of a training-side object!"

    idx, sep, hit, off = match_with_offset(ext.ra, ext.dec, k15.ra, k15.dec, 0.5)
    F = k15[list(K15)].to_numpy(float)
    main = np.array(K15)[np.nanargmax(np.nan_to_num(F, nan=-1), axis=1)]
    ext["k15_class"] = np.where(hit, main[np.clip(idx, 0, None)], None)
    ext["k15_common"] = ext.k15_class.map(K15_TO_COMMON)
    ext["k15_merger_frac"] = np.where(hit, k15.merger.to_numpy(float)[np.clip(idx, 0, None)], np.nan)
    ext["gzh_common"] = ext.gz_label.map(GZH_TO_COMMON)
    ext["in_gzh_overlap"] = ~ext.is_pseudo_label & ext.k15_common.notna()
    keep = ["object_id", "ra", "dec", "snr", "mag", "k15_class", "k15_common", "k15_merger_frac", "gz_label",
            "gzh_common", "in_gzh_overlap"]
    ext[keep].to_csv(out / "candels_objects.csv", index=False)

    ov = ext[ext.in_gzh_overlap & ext.gzh_common.notna()]
    if len(ov) < 5:
        raise StageSkipped(f"Only {len(ov)} galaxies are labelled by both GZH and CANDELS - no ceiling possible.")
    mat = pd.crosstab(pd.Series(ov.gzh_common.to_numpy(), name="GZH (mapped)"),
                      pd.Series(ov.k15_common.to_numpy(), name="CANDELS expert"))
    mat.to_csv(out / "taxonomy_alignment_matrix.csv")
    present = sorted(set(ov.k15_common) | set(ov.gzh_common))        # only classes that occur are scored
    ceil = {"ceiling_macro_f1": round(float(f1_score(ov.k15_common, ov.gzh_common, labels=present,
                                                     average="macro", zero_division=0)), 4),
            "classes_scored": present,
            "cohen_kappa": round(float(cohen_kappa_score(ov.k15_common, ov.gzh_common)), 4),
            "n_overlap": int(len(ov)), "n_external_galaxies": int(len(ext)), "phase7_region": region,
            "n_k15_matched": int(hit.sum()), "k15_offset_arcsec": off}
    save_json(ceil, out / "taxonomy_ceiling.json")
    ctx.log.info("[Stage 7a] alignment matrix (overlap set):\n" + mat.to_string())
    ctx.log.info(f"[Stage 7a] {ceil}")
    return ceil

In [ ]:
%%writefile src/p2p/stages/stage7b_crosscat.py
"""Stage 7b - Cross-catalogue evaluation (run once).

Purpose : how well does the frozen model generalise to an independent expert scheme?
Method  : the frozen, calibrated ensemble predicts every external galaxy with a
          CANDELS label; galaxies in the GZH overlap (used for the ceiling) and
          expert point sources are excluded. Predictions are mapped to the
          shared taxonomy and scored with Macro-F1 (+ 95% bootstrap interval).
Output  : crosscat_predictions.csv, cross_catalogue_metrics.json
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score

from ..context import Ctx, StageSkipped, save_json
from ..features import TabularPrep, galaxy_table, load_cutouts, make_tensors
from ..inference import calibrated_predict, load_models
from ..nets import MORPH_CLASSES
from ..utils import bootstrap_ci
from .stage7a_taxonomy import COMMON, GZH_TO_COMMON


def run(ctx: Ctx):
    out = ctx.stage_dir("7b")
    p = ctx.path("7a", "candels_objects.csv")
    if not p.exists():
        raise StageSkipped("Stage 7a produced no CANDELS objects.")
    c = pd.read_csv(p)
    c["in_gzh_overlap"] = c.in_gzh_overlap.astype(str).str.lower().eq("true")
    test = c[~c.in_gzh_overlap & c.k15_common.notna()]
    if test.empty:
        raise StageSkipped("No CANDELS-labelled external galaxies outside the GZH overlap.")
    g = galaxy_table(ctx).set_index("object_id").loc[test.object_id].reset_index()
    data = make_tensors(g, load_cutouts(ctx), TabularPrep.load(ctx.require("4a", "tab_prep.joblib")), with_labels=False)
    res = calibrated_predict(ctx, data, load_models(ctx))
    pred = np.array(MORPH_CLASSES)[res["morph"].argmax(1)]
    test = test.assign(pred_class=pred, pred_common=pd.Series(pred).map(GZH_TO_COMMON).to_numpy(),
                       p_merger=res["merger"])
    test.to_csv(out / "crosscat_predictions.csv", index=False)
    labs = {c_: i for i, c_ in enumerate(COMMON)}
    yt = test.k15_common.map(labs).to_numpy()
    yp = test.pred_common.map(labs).to_numpy()
    f1 = lambda a, b: f1_score(a, b, labels=sorted(set(np.asarray(a).tolist()) | set(np.asarray(b).tolist())),
                               average="macro", zero_division=0)
    m = {"xcat_macro_f1": round(float(f1(yt, yp)), 4),
         "xcat_macro_f1_ci95": bootstrap_ci(lambda a, P: f1(a, P), yt, yp, seed=ctx.seed),
         "n_test": int(len(test)), "n_excluded_overlap": int(c.in_gzh_overlap.sum()),
         "n_excluded_point_source": int(c.k15_common.isna().sum()),
         "per_class": classification_report(test.k15_common, test.pred_common, labels=list(COMMON),
                                            output_dict=True, zero_division=0)}
    save_json(m, out / "cross_catalogue_metrics.json")
    ctx.log.info(f"[Stage 7b] cross-catalogue Macro-F1 = {m['xcat_macro_f1']} on n = {m['n_test']}\n" +
                 pd.crosstab(test.k15_common, test.pred_common, rownames=["CANDELS"], colnames=["model"]).to_string())
    return {"xcat_macro_f1": m["xcat_macro_f1"]}

In [ ]:
%%writefile src/p2p/stages/stage7c_generalisation.py
"""Stage 7c - Generalisation disentanglement.

Purpose : report true degradation, not a raw score drop.
Method  : true degradation = Taxonomy Ceiling - Cross-Catalogue Macro-F1.
          The in-domain test score is recomputed on the same coarse taxonomy so
          all three numbers share one scale. The external sample is also split
          into SNR tertiles, and the covariate shift (SNR, size, morphology)
          between the GZH training set and the external test set is listed.
Output  : generalisation_report.json, generalisation_gap.png
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

from ..context import Ctx, StageSkipped, load_json, save_json
from ..features import galaxy_table, supervised_splits
from ..nets import MORPH_CLASSES
from .stage7a_taxonomy import COMMON, GZH_TO_COMMON


def run(ctx: Ctx):
    out = ctx.stage_dir("7c")
    if not ctx.path("7b", "cross_catalogue_metrics.json").exists():
        raise StageSkipped("Stage 7b has no result.")
    ceil = load_json(ctx.require("7a", "taxonomy_ceiling.json"))["ceiling_macro_f1"]
    xcat = load_json(ctx.require("7b", "cross_catalogue_metrics.json"))["xcat_macro_f1"]
    tp = pd.read_csv(ctx.require("6a", "test_predictions.csv"))
    names = np.array(MORPH_CLASSES)
    f1 = lambda a, b: float(f1_score(a, b, labels=sorted(set(a) | set(b)), average="macro", zero_division=0))
    indomain = round(f1(pd.Series(names[tp.y_true]).map(GZH_TO_COMMON), pd.Series(names[tp.y_pred]).map(GZH_TO_COMMON)), 4)
    xp = pd.read_csv(ctx.require("7b", "crosscat_predictions.csv"))
    tert = pd.qcut(xp.snr, 3, labels=["low SNR", "mid SNR", "high SNR"]) if len(xp) >= 9 else None
    by_snr = {str(k): round(f1(g.k15_common, g.pred_common), 4) for k, g in xp.groupby(tert, observed=True)} if tert is not None else {}
    train = supervised_splits(ctx)["train"]
    ext = galaxy_table(ctx).set_index("object_id").loc[xp.object_id]
    shift = {c: {"gzh_train_median": round(float(train[c].median()), 3), "external_median": round(float(ext[c].median()), 3)}
             for c in ("snr", "mag", "r_half", "gini", "asymmetry", "concentration") if c in train and c in ext}
    rep = {"indomain_macro_f1_coarse": indomain, "taxonomy_ceiling_macro_f1": ceil, "cross_catalogue_macro_f1": xcat,
           "true_degradation": round(ceil - xcat, 4), "naive_drop": round(indomain - xcat, 4),
           "label_scheme_component": round(indomain - ceil, 4), "xcat_macro_f1_by_snr_tertile": by_snr,
           "covariate_shift": shift,
           "phase7_region": load_json(ctx.require("7a", "taxonomy_ceiling.json")).get("phase7_region", "external")}
    if rep["phase7_region"] == "test":
        rep["note"] = ("single field: Phase 7 used the held-out test split, so depth and PSF match training; the gap "
                       "measures the label-scheme change and the fainter CANDELS-only galaxies, not a new field")
    save_json(rep, out / "generalisation_report.json")
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5.5, 3.4))
    vals = [indomain, ceil, xcat]
    ax.bar(["In-domain\n(coarse)", "Taxonomy\nceiling", "Cross-\ncatalogue"], vals, color=["#9AA3B5", "#D3D9E5", "#2E5496"])
    for i, v in enumerate(vals):
        ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Macro-F1 (shared taxonomy)")
    ax.set_title(f"Stage 7c: true degradation = {ceil - xcat:+.3f}")
    fig.tight_layout()
    fig.savefig(out / "generalisation_gap.png", dpi=110)
    plt.close(fig)
    ctx.log.info(f"[Stage 7c] {rep}")
    return {"true_degradation": rep["true_degradation"]}

In [ ]:
%%writefile src/p2p/stages/stage8_catalogue.py
"""Stage 8 - Production compilation: the unified, provenance-tracked catalog.csv.

Purpose : one deliverable that a scientist can filter with a single column.
Method  : all tracks (Star, Galaxy, Noise; GZH and external pointings) are merged
          into one row per detected source. Every galaxy gets calibrated class
          probabilities and a merger probability from the frozen ensemble.
          Duplicates from overlapping pointings (closer than 0.2") are merged,
          keeping the higher-SNR detection. Each row gets an IAU-style
          designation (truncated coordinates, a letter appended when two would
          collide). `classification` (C18) is the final Star / Galaxy / Noise class
          after clustering and gating. Columns are written in the fixed order of
          COLUMNS (C1 object_id ... C75 baseline_semantic); the merger result is the
          probability p_merger only (no thresholded duplicate). Columns added
          after C75 (BW) follow EXTRA_COLUMNS: positional errors, per-band
          magnitudes, exposures, the cosmic-ray tests, gaia_match for every row
          and the CANDELS (Kartaltepe 2015) match for every galaxy. Every column has its own rule in COLUMNS
          (group, filled_for): detection measurements and the segregation evidence
          are kept for every row; magnitudes and colours for Star and Galaxy;
          morphology, labels and classification for Galaxy; Gaia and PSF
          calibration for Star; the noise model for Noise - null everywhere else.
          Before writing, the delivery checks of Objective 5 are asserted,
          including that rule and unique designations.
Output  : catalog.csv, catalog_columns.csv (data dictionary), delivery_checks.json
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from ..context import Ctx, load_json, load_objects, save_json
from ..features import TabularPrep, load_cutouts, make_tensors
from ..inference import calibrated_predict, load_models
from ..nets import MORPH_CLASSES
from ..catalogs import load_candels
from ..utils import as_bool, fwhm_fit, iau_designation, match_with_offset, predictive_entropy, unit_vectors

# One row per catalogue column: (name, group, filled_for, description).
# filled_for = the classifications whose rows MAY hold a value; every other row is null (checked before writing).
# A note in brackets narrows it further, e.g. "Star (Gaia-matched)": unmatched stars are null too.
ALL, SG = "all rows", "Star, Galaxy"
COLUMNS = [
    # Order = Sangeeta's column sequence (catalog_columns_Sangeeta.xlsx, 24 Oct 2026), pred_merger removed.
    # Stage 8 asserts that catalog.csv is written in exactly this order (C1 object_id ... C75 baseline_semantic).
    ("object_id", "identity", ALL, "unique source id (Stage 1c)"),  # C1
    ("designation", "identity", ALL, "IAU-style name from truncated RA/Dec; a letter is appended when two detections would share a name"),  # C2
    ("product_type", "provenance", ALL, "drc or drz"),  # C3
    ("ra", "identity", ALL, "right ascension [deg, ICRS]"),  # C4
    ("dec", "identity", ALL, "declination [deg, ICRS]"),  # C5
    ("pointing", "identity", ALL, "pointing id (Stage 1a)"),  # C6
    ("detection_band", "identity", ALL, "filter the source was detected in"),  # C7
    ("tile_id", "identity", ALL, "detection-band _drc file"),  # C8
    ("frame_quality_flag", "provenance", ALL, "1 = clean, 0 = in a QC-flagged tile (Stage 1b)"),  # C9
    ("edge_flag", "provenance", ALL, "cutout truncated by the image edge"),  # C10
    ("anomaly_flag", "provenance", ALL, "1 = fits no population (Stage 2e)"),  # C11
    ("cluster_id_gmm", "segregation", ALL, "GMM component (Stage 2b)"),  # C12
    ("cluster_id_hdbscan", "segregation", ALL, "HDBSCAN cluster, -1 = density noise"),  # C13
    ("gmm_p_star", "segregation", ALL, "GMM posterior of the Star population"),  # C14
    ("gmm_p_galaxy", "segregation", ALL, "GMM posterior of the Galaxy population"),  # C15
    ("gmm_p_noise", "segregation", ALL, "GMM posterior of the Noise population"),  # C16
    ("split_tag", "provenance", ALL, "train / dev / test / external; buffer = inside a collar (no split)"),  # C17
    ("classification", "identity", ALL, "final population after clustering + gating: Star / Galaxy / Noise"),  # C18
    ("morph_flag", "morphology", "Galaxy", "statmorph flag: 0 good, 1 suspect, 2 bad, 4 too faint to measure"),  # C19
    ("pred_class", "classification", "Galaxy (anomaly_flag 0)", "predicted morphology class"),  # C20
    ("p_smooth", "classification", "Galaxy (anomaly_flag 0)", "calibrated probability Smooth"),  # C21
    ("p_spiral", "classification", "Galaxy (anomaly_flag 0)", "calibrated probability FaceOnSpiral"),  # C22
    ("p_edgeon", "classification", "Galaxy (anomaly_flag 0)", "calibrated probability EdgeOn"),  # C23
    ("p_irregular", "classification", "Galaxy (anomaly_flag 0)", "calibrated probability Irregular"),  # C24
    ("p_ambiguous", "classification", "Galaxy (anomaly_flag 0)", "calibrated probability Ambiguous"),  # C25
    ("pred_entropy", "classification", "Galaxy (anomaly_flag 0)", "predictive entropy of the class probabilities"),  # C26
    ("p_merger", "classification", "Galaxy (anomaly_flag 0)", "calibrated merger probability (dual-branch model)"),  # C27
    ("star_fwhm_arcsec", "psf_calibration", "Star", "this star's FWHM, 2-D Gaussian fit, detection band [arcsec]"),  # C28
    ("used_in_psf_match", "psf_calibration", "Star", "star used for the per-band PSFs in cross-filter matching (Stage 1d)"),  # C29
    ("used_in_epsf", "psf_calibration", "Star", "star stacked into the empirical PSF (Stage 5a)"),  # C30
    ("epsf_isolated", "psf_calibration", "Star", "no neighbour near the star"),  # C31
    ("pointing_epsf_fwhm_arcsec", "psf_calibration", "Star", "FWHM of this pointing's empirical PSF [arcsec]"),  # C32
    ("pointing_matched_fwhm_arcsec", "psf_calibration", "Star", "MEASURED FWHM of the PSF all bands were matched to (Stage 1d) [arcsec]; empty = no stars, Gaussian fallback"),  # C33
    ("is_pseudo_label", "labels", "Galaxy", "False = a GZH label exists; True = model prediction only"),  # C34
    ("gz_label", "labels", "Galaxy (GZH-matched)", "Galaxy Zoo: Hubble class"),  # C35
    ("gz_vote_frac", "labels", "Galaxy (GZH-matched)", "GZH agreement along the decision path"),  # C36
    ("gz_merger_flag", "labels", "Galaxy (GZH-matched)", "GZH merger flag"),  # C37
    ("gaia_source_id", "gaia", "Star (Gaia-matched)", "Gaia DR3 source_id (exact integer)"),  # C38
    ("gaia_sep_arcsec", "gaia", "Star (Gaia-matched)", "separation to the Gaia position [arcsec]"),  # C39
    ("mag", "photometry", SG, "AB magnitude, detection band"),  # C40
    ("colour_1", "photometry", SG, "colour {c1} [mag]"),  # C41
    ("colour_2", "photometry", SG, "colour {c2} [mag]"),  # C42
    ("gini", "morphology", "Galaxy (measured)", "Gini coefficient"),  # C43
    ("m20", "morphology", "Galaxy (measured)", "M20 moment"),  # C44
    ("concentration", "morphology", "Galaxy (measured)", "concentration C"),  # C45
    ("asymmetry", "morphology", "Galaxy (measured)", "asymmetry A"),  # C46
    ("smoothness", "morphology", "Galaxy (measured)", "smoothness / clumpiness S"),  # C47
    ("is_transitional", "segregation", "Galaxy", "density-noise object kept as a galaxy (posterior >= 0.8)"),  # C48
    ("psf_pass", "morphology", "Galaxy", "2 = morphology measured with the empirical PSF (Stage 5b)"),  # C49
    ("morph_flag_sersic", "morphology", "Galaxy", "statmorph Sersic-fit flag (0 = good fit)"),  # C50
    ("sersic_n", "morphology", "Galaxy (measured)", "Sersic index"),  # C51
    ("sersic_rhalf", "morphology", "Galaxy (measured)", "Sersic half-light radius [px]"),  # C52
    ("r_half", "morphology", "Galaxy (measured)", "non-parametric half-light radius [px]"),  # C53
    ("ellipticity", "morphology", "Galaxy (measured)", "ellipticity"),  # C54
    ("shape_asymmetry", "morphology", "Galaxy (measured)", "shape asymmetry"),  # C55
    ("multimode", "morphology", "Galaxy (measured)", "multimode statistic"),  # C56
    ("intensity", "morphology", "Galaxy (measured)", "intensity statistic"),  # C57
    ("deviation", "morphology", "Galaxy (measured)", "deviation statistic"),  # C58
    ("parallax", "gaia", "Star (Gaia-matched)", "Gaia parallax [mas]"),  # C59
    ("pmra", "gaia", "Star (Gaia-matched)", "Gaia proper motion RA [mas/yr]"),  # C60
    ("pmdec", "gaia", "Star (Gaia-matched)", "Gaia proper motion Dec [mas/yr]"),  # C61
    ("gaia_g", "gaia", "Star (Gaia-matched)", "Gaia G magnitude"),  # C62
    ("gaia_teff", "gaia", "Star (Gaia-matched)", "Gaia effective temperature [K]"),  # C63
    ("noise_score", "noise", "Noise", "probability-like score that the detection is spurious (Stage 3c)"),  # C64
    ("fpr_at_threshold", "noise", "Noise", "false-positive rate of detections at this source's SNR (Stage 3c)"),  # C65
    ("qc_flagged_bands", "provenance", ALL, "bands whose QC tile was flagged"),  # C66
    ("pipeline_version", "provenance", ALL, "code version"),  # C67
    ("snr", "detection", ALL, "signal-to-noise ratio of the detection, detection band"),  # C68
    ("ndrizim", "provenance", ALL, "NDRIZIM header: input images drizzled into the detection file (ACS/WFC: 2 per exposure, one per chip) - NOT exposures, see n_exposures"),  # C69
    ("flux", "detection", ALL, "flux of the detection [e-/s], detection band"),  # C70
    ("flux_err", "detection", ALL, "flux error [e-/s]"),  # C71
    ("fwhm_over_psf", "segregation", ALL, "source FWHM / PSF FWHM, both 2-D Gaussian fits in the detection band (1 = point-like)"),  # C72
    ("band_consistency", "segregation", ALL, "weakest other-band flux / detection-band flux (~0 = single-band, e.g. a cosmic ray); empty if no other band covers it"),  # C73
    ("reconstruction_error", "segregation", ALL, "Stage 2a autoencoder reconstruction error"),  # C74
    ("baseline_semantic", "segregation", ALL, "rule-based baseline class (Stage 2c)"),  # C75
]
# Columns after C75 (BW), in this order. "{band}" entries are repeated for every band, blue -> red.
EXTRA_COLUMNS = [
    ("ra_err", "astrometry", ALL, "statistical centroid error, RA cos(dec): FWHM / (2.355 SNR) [arcsec] (absolute WCS error not included)"),
    ("dec_err", "astrometry", ALL, "statistical centroid error, Dec: FWHM / (2.355 SNR) [arcsec]"),
    ("mag_{band}", "photometry", SG, "AB magnitude in {band} (detection footprint, cosmic-ray pixels replaced)"),
    ("magerr_{band}", "photometry", SG, "error of mag_{band} [mag]"),
    ("n_exposures", "provenance", ALL, "exposures covering the source in the detection band (drizzle CTX map); 1 = no cosmic-ray rejection possible"),
    ("cr_candidate", "segregation", ALL, "True = no other covering band confirms the source at S/N >= detection.confirm_snr (cosmic-ray candidate); empty = no other band covers it"),
    ("cr_pixels_masked", "segregation", ALL, "cosmic-ray pixels replaced and masked inside the source footprint (Stage 1c)"),
    ("gaia_match", "gaia", ALL, "True = a Gaia DR3 source within semantic_audit.gaia_radius_arcsec, whatever the classification"),
    ("classification_override", "segregation", ALL, "why classification differs from the GMM cluster name (Stage 2e physical gate / transitional rule); empty = cluster name kept"),
    ("k15_class", "cross_catalogue", "Galaxy (CANDELS-matched)", "CANDELS (Kartaltepe et al. 2015) dominant expert class"),
    ("k15_merger_frac", "cross_catalogue", "Galaxy (CANDELS-matched)", "CANDELS expert merger fraction f(M)"),
    ("k15_sep_arcsec", "cross_catalogue", "Galaxy (CANDELS-matched)", "separation to the CANDELS position after the median offset is removed [arcsec]"),
]


def catalogue_columns(bands):
    """The full column list: C1-C75 (COLUMNS), then EXTRA_COLUMNS with {band} expanded."""
    extra = []
    for n, g, f, d in EXTRA_COLUMNS:
        if "{band}" in n:
            extra += [(n.format(band=b), g, f, d.format(band=b)) for b in bands]
        else:
            extra.append((n, g, f, d))
    return COLUMNS + extra

RENAME = {"flag": "morph_flag", "flag_sersic": "morph_flag_sersic"}          # internal name -> catalogue name
ORDER = [n for n, *_ in COLUMNS]                           # the fixed 75-column sequence, C1 ... C75
assert len(ORDER) == 75 and len(set(ORDER)) == 75, "COLUMNS must hold 75 distinct columns"
MORPH_COLS = ["gini", "m20", "concentration", "asymmetry", "smoothness", "sersic_n", "sersic_rhalf", "r_half",
              "shape_asymmetry", "multimode", "intensity", "deviation"]


def allowed(filled_for: str) -> set | None:
    """Classifications a column may be filled for (None = every row)."""
    base = filled_for.split("(")[0].strip()
    return None if base == ALL else {c.strip() for c in base.split(",")}


PCOLS = ["p_smooth", "p_spiral", "p_edgeon", "p_irregular", "p_ambiguous"]


def unique_designations(names: pd.Series, ra: pd.Series) -> pd.Series:
    """Truncated IAU names collide for detections closer than ~1.5": append A, B, C ... (ordered by RA)."""
    names = pd.Series(np.asarray(names, dtype=object), index=ra.index)
    dup = names.duplicated(keep=False)
    if dup.any():
        order = ra[dup].groupby(names[dup]).rank(method="first").astype(int) - 1
        letters = [chr(65 + k % 26) + ("" if k < 26 else str(k // 26)) for k in order]
        names.loc[order.index] = names.loc[order.index] + np.array(letters, dtype=object)
    return names


def deduplicate(df, radius_arcsec):
    pairs = cKDTree(unit_vectors(df.ra, df.dec)).query_pairs(2 * np.sin(np.deg2rad(radius_arcsec / 3600) / 2))
    snr = df.snr.to_numpy()
    drop = {j if snr[i] >= snr[j] else i for i, j in pairs}
    return df.drop(index=df.index[list(drop)]), len(drop)


def run(ctx: Ctx):
    cfg = ctx.cfg["catalogue"]
    out = ctx.stage_dir("8")
    parts = ["sources", "pass1", "clusters", "baseline", "gaia", "segregation", "labels", "routing", "noise", "pass2"]
    parts = [p for p in parts if p != "routing" or ctx.path("3b", "star_routing.csv").exists()]
    df = load_objects(ctx, parts)
    df["psf_pass"] = df.psf_pass.fillna(1).astype(int)

    # ---- calibrated predictions for every galaxy row ----
    gal = (df.cluster_semantic == "Galaxy") & (df.anomaly_flag == 0)
    models = load_models(ctx)
    prep = TabularPrep.load(ctx.require("4a", "tab_prep.joblib"))
    cut = load_cutouts(ctx)
    P, M = [], []
    gidx = df.index[gal]
    for i in range(0, len(gidx), 256):
        r = calibrated_predict(ctx, make_tensors(df.loc[gidx[i:i + 256]], cut, prep, with_labels=False), models)
        P.append(r["morph"])
        M.append(r["merger"])
    if len(gidx):
        P, M = np.concatenate(P), np.concatenate(M)
        for k, c in enumerate(PCOLS):
            df.loc[gal, c] = P[:, k]
        df.loc[gal, "pred_class"] = np.array(MORPH_CLASSES)[P.argmax(1)]
        df.loc[gal, "p_merger"] = M                  # probability only: a 0/1 pred_merger would repeat it
        df.loc[gal, "pred_entropy"] = predictive_entropy(P)

    # ---- identity + provenance ----
    pts = load_json(ctx.require("1a", "pointings.json"))["pointings"]
    man = load_json(ctx.require("1a", "manifest.json"))
    det_file = {p: Path(v["bands"][v["detection_band"]]).name for p, v in pts.items()}
    df["tile_id"] = df.pointing.map(det_file)
    df["product_type"] = df.tile_id.map(lambda f: man.get(f, {}).get("product_type"))
    df["ndrizim"] = df.tile_id.map(lambda f: man.get(f, {}).get("ndrizim"))
    from .. import __version__
    df["pipeline_version"] = f"p2p-{__version__}"
    # empty when HDBSCAN failed (Stage 2e could not apply the rule): keep it empty, do not turn it into False
    df["is_transitional"] = df.is_transitional.map(
        lambda v: None if pd.isna(v) or str(v).strip() == "" else str(v).strip().lower() in ("true", "1", "1.0"))
    df["classification"] = df["cluster_semantic"]    # final Star / Galaxy / Noise after clustering + gating

    # ---- PSF / optical calibration columns for the star track ----
    psfm = load_json(ctx.require("1d", "psf_match.json"))
    scale = df.pointing.map({p: v.get("pixscale_arcsec", np.nan) for p, v in psfm.items()}).astype(float)
    # each star's FWHM with the SAME 2-D Gaussian fit Stage 5a uses for the ePSF, in the detection band of the
    # un-matched cutout, so star_fwhm_arcsec and pointing_epsf_fwhm_arcsec can be compared directly
    from .stage1d_psf_match import crop_center
    info = load_json(ctx.require("1a", "pointings.json"))
    raw = np.load(ctx.require("1c", "cutouts_raw.npy"), mmap_mode="r")
    det_ch = {p: info["bands"].index(v["detection_band"]) for p, v in info["pointings"].items()}
    star = df.cluster_semantic == "Star"
    df["star_fwhm_arcsec"] = np.nan
    df.loc[star, "star_fwhm_arcsec"] = [
        fwhm_fit(crop_center(np.asarray(raw[int(i), det_ch[p]], np.float32), ctx.cfg["psf_match"]["stamp"]))
        for i, p in zip(df.loc[star, "cutout_index"], df.loc[star, "pointing"])]
    df["star_fwhm_arcsec"] = (df["star_fwhm_arcsec"] * scale).round(4)
    match_ids = {int(i) for v in psfm.values() for i in v.get("star_ids", [])}
    df["used_in_psf_match"] = df.object_id.isin(match_ids)
    # the MEASURED matched FWHM (Stage 1d), empty when the target band used the Gaussian fallback
    df["pointing_matched_fwhm_arcsec"] = df.pointing.map(
        {p: (v.get("matched_fwhm_arcsec") if v.get("target_psf_measured") else np.nan) for p, v in psfm.items()}
    ).astype(float).round(4)
    rep = ctx.path("5a", "epsf_report.json")
    if rep.exists():
        ep = {p: v.get("fwhm_px", np.nan) for p, v in load_json(rep).items()}
        df["pointing_epsf_fwhm_arcsec"] = (df.pointing.map(ep).astype(float) * scale).round(4)
    for c in ("used_in_epsf", "epsf_isolated"):
        if c in df:
            df[c] = as_bool(df[c])
    df["is_pseudo_label"] = np.where(df.cluster_semantic == "Galaxy", ~df.gz_label.notna(), None)
    df, n_dup = deduplicate(df, cfg["dedup_radius_arcsec"])
    df["designation"] = unique_designations(iau_designation(df.ra, df.dec, cfg["designation_prefix"]), df.ra)

    # exact Gaia DR3 ids (19 digits do not fit a float64; a merge with missing values would turn them into floats)
    gm = pd.read_csv(ctx.require("2c", "gaia_match.csv"), dtype={"gaia_source_id": "string"})
    ids = gm.set_index("object_id")["gaia_source_id"]
    df["gaia_source_id"] = df.object_id.map(ids).where(lambda v: v.notna() & (v != "-1"), None)
    df["gaia_match"] = as_bool(df.gaia_hit)                       # every row: misclassified Gaia stars stay visible
    df["cr_candidate"] = df.cr_candidate.map(
        lambda v: None if pd.isna(v) or str(v).strip() == "" else str(v).strip().lower() in ("true", "1", "1.0")) \
        if "cr_candidate" in df else None

    # CANDELS (Kartaltepe 2015) match for every galaxy (Stage 7a evaluates only the Phase-7 region)
    for c in ("k15_class", "k15_merger_frac", "k15_sep_arcsec"):
        df[c] = None
    k15 = load_candels(ctx, list(load_json(ctx.require("1a", "manifest.json")).values()))
    galaxy_rows = (df.cluster_semantic == "Galaxy").to_numpy()
    if k15 is not None and len(k15) and galaxy_rows.any():
        from .stage7a_taxonomy import K15
        g = df[galaxy_rows]
        kidx, ksep, khit, koff = match_with_offset(g.ra, g.dec, k15.ra, k15.dec, 0.5)
        F = k15[list(K15)].to_numpy(float)
        main = np.array(K15)[np.nanargmax(np.nan_to_num(F, nan=-1), axis=1)]
        j = np.clip(kidx, 0, None)
        df.loc[galaxy_rows, "k15_class"] = np.where(khit, main[j], None)
        df.loc[galaxy_rows, "k15_merger_frac"] = np.where(khit, k15.merger.to_numpy(float)[j], np.nan)
        df.loc[galaxy_rows, "k15_sep_arcsec"] = np.where(khit, np.round(ksep, 4), np.nan)
        ctx.log.info(f"[Stage 8] CANDELS: {int(khit.sum())} of {int(galaxy_rows.sum())} galaxies matched "
                     f"(offset removed dRA {koff[0]:+.3f}\", dDec {koff[1]:+.3f}\")")
    else:
        ctx.log.info("[Stage 8] no CANDELS table covering the field - k15_* columns empty")
    df = df.rename(columns=RENAME)
    for c in MORPH_COLS:                                          # statmorph -99 = could not measure -> empty
        if c in df:
            v = pd.to_numeric(df[c], errors="coerce")
            df[c] = v.where(~(v <= -98.9), np.nan)

    # a required column that an earlier stage did not produce is written empty, so the C1-C75 layout never shifts
    bands = load_json(ctx.require("1a", "pointings.json"))["bands"]
    cols = catalogue_columns(bands)
    missing = [n for n, *_ in cols if n not in df]
    if missing:
        ctx.log.info(f"[Stage 8] WARNING: no values from earlier stages for {missing}; written as empty columns")
        for n in missing:
            df[n] = None

    # ---- sparse schema, column by column ----
    for name, _, filled_for, _ in cols:
        if name not in df:
            continue
        ok = allowed(filled_for)
        df[name] = df[name].astype(object)
        if ok is not None:
            df.loc[~df.classification.isin(ok), name] = None
        if filled_for.startswith("Star (Gaia-matched)"):
            df.loc[~as_bool(df.gaia_hit), name] = None
        if "anomaly_flag 0" in filled_for:
            df.loc[df.anomaly_flag != 0, name] = None
    ordered = [n for n, *_ in cols]
    cat = df[ordered].sort_values("object_id")
    sparse_ok = all(cat.loc[~cat.classification.isin(allowed(f)), n].isna().all()
                    for n, _, f, _ in cols if allowed(f) is not None)

    # ---- delivery checks (Objective 5) ----
    g = cat[(cat.classification == "Galaxy") & (cat.anomaly_flag == 0)]
    psum = g[PCOLS].astype(float).sum(axis=1)
    checks = {"every_galaxy_has_calibrated_probs": bool(np.allclose(psum, 1.0, atol=1e-4)),
              "every_galaxy_has_merger_prob": bool(g.p_merger.notna().all()),
              "every_galaxy_has_heritage_flag": bool(g.is_pseudo_label.notna().all()),
              "every_measured_galaxy_at_pass_2": bool((g[g.snr >= ctx.cfg["morphology"]["min_snr"]].psf_pass == 2).all()),
              "class_columns_empty_for_other_classifications": bool(sparse_ok),
              "designations_unique": bool(cat.designation.is_unique),
              "column_order_as_specified": list(cat.columns) == ordered and list(cat.columns[:75]) == ORDER}
    pm = g.p_merger.astype(float)
    notes = {"columns": len(cat.columns), "first_extra_column": cat.columns[75] if len(cat.columns) > 75 else None,
             "p_merger_range": [round(float(pm.min()), 4), round(float(pm.max()), 4)] if len(pm) else None}
    if len(pm) and float(pm.max() - pm.min()) < 0.1:
        notes["merger_model"] = (f"p_merger spans only {pm.min():.3f}-{pm.max():.3f}: the merger head is effectively "
                                 "constant - report it as not trainable with this number of GZH mergers")
        ctx.log.info(f"[Stage 8] NOTE: {notes['merger_model']}")
    save_json({**checks, "duplicates_merged": n_dup, "notes": notes}, out / "delivery_checks.json")
    assert all(checks.values()), f"Delivery check failed: {checks}"
    cat.to_csv(out / "catalog.csv", index=False)
    fill = {"c1": f"{bands[0]} - {bands[1]}" if len(bands) > 1 else "(needs 2 bands)",
            "c2": f"{bands[1]} - {bands[2]}" if len(bands) > 2 else "(needs 3 bands)"}
    pd.DataFrame([{"column": n, "group": g_, "filled_for": f, "rows_filled": int(cat[n].notna().sum()),
                   "description": d.format(**fill)}
                  for n, g_, f, d in cols]).to_csv(out / "catalog_columns.csv", index=False)
    heritage = g.is_pseudo_label.astype(str).value_counts().to_dict()
    ctx.log.info(f"[Stage 8] catalog.csv: {len(cat)} rows x {cat.shape[1]} columns ({n_dup} duplicates merged) | "
                 f"{cat.classification.value_counts().to_dict()} | galaxy labels from GZH (False) / model (True): "
                 f"{heritage}")
    ctx.log.info(f"[Stage 8] delivery checks: {checks}")
    return {"rows": len(cat), "checks": checks}

### 2.5 Tools — `tools/`
- `check_catalog_columns.py`
- `download_catalogs.py`
- `download_mast.py`
- `inspect_catalog.py`
- `inspect_fits.py`
- `show_stage.py`

In [ ]:
%%writefile tools/check_catalog_columns.py
#!/usr/bin/env python
"""Check that the catalogue files in data/catalogs have the columns that
config.yaml maps (catalogs.<kind>.columns). Run this BEFORE the pipeline when
you add a new GZH / Gaia / CANDELS file.

    python tools/check_catalog_columns.py [--config config.yaml]
"""
from __future__ import annotations

import argparse
import difflib
import sys
from pathlib import Path

import pandas as pd

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from p2p.catalogs import K15_CLASS_COLUMNS, _candidates, _text_format, find_catalog  # noqa: E402
from p2p.context import Ctx  # noqa: E402


def header(path: Path):
    n = path.name.lower()
    if n.endswith((".fits", ".fit", ".fits.gz")):
        from astropy.table import Table
        return Table.read(path).colnames
    sep, comment = _text_format(path)
    return [c.strip() for c in pd.read_csv(path, nrows=0, sep=sep, comment=comment).columns]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="config.yaml")
    ctx = Ctx(ROOT, ap.parse_args().config)
    for kind in ("gzh", "gaia", "candels"):
        p = find_catalog(ctx, kind)
        print(f"\n=== {kind}: {ctx.rel(p) if p else 'NOT FOUND in ' + str(ctx.catalogs)} ===")
        if p is None:
            continue
        cols = header(p)
        low = {c.strip().lower(): c for c in cols}
        required = {"ra", "dec"} | (set(K15_CLASS_COLUMNS) if kind == "candels" else set())
        for std, want in ctx.cfg["catalogs"][kind]["columns"].items():
            found = next((low[c.lower()] for c in _candidates(want) if c.lower() in low), None)
            if found is not None:
                print(f"  OK       {std:15s} <- {found}")
            else:
                close = difflib.get_close_matches(_candidates(want)[0], cols, n=3, cutoff=0.4)
                tag = "MISSING!" if std in required else "missing "
                print(f"  {tag} {std:15s} <- {want}   closest: {close}"
                      + ("   (REQUIRED - the pipeline stops)" if std in required else "   (optional)"))
        print(f"  ({len(cols)} columns in the file)")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile tools/download_catalogs.py
#!/usr/bin/env python
"""Download the reference catalogues into data/catalogs (needs internet).

    python tools/download_catalogs.py gzh                                   # Galaxy Zoo: Hubble Table 4
    python tools/download_catalogs.py gaia --ra 53.12 --dec -27.80          # Gaia DR3 cone around a pointing
    python tools/download_catalogs.py candels                               # Kartaltepe+2015 (VizieR J/ApJS/221/11/table3)
    python tools/download_catalogs.py all --ra 53.12 --dec -27.80

Gaia can also be left to the pipeline: set catalogs.gaia.query_online: true in config.yaml.
After downloading, run:  python tools/check_catalog_columns.py
"""
from __future__ import annotations

import argparse
import shutil
import urllib.parse
import urllib.request
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
CATS = ROOT / "data" / "catalogs"
GZH_URL = "https://zooniverse-data.s3.amazonaws.com/galaxy-zoo-hubble/gz_hubble_main.csv.gz"


def gzh():
    out = CATS / "gz_hubble_main.csv.gz"
    if out.exists():
        print(f"[gzh] already there: {out.relative_to(ROOT)}")
        return
    print(f"[gzh] downloading {GZH_URL} ...")
    tmp = out.with_suffix(".part")
    with urllib.request.urlopen(GZH_URL) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f, length=1 << 20)
    tmp.rename(out)
    print(f"[gzh] saved {out.relative_to(ROOT)} ({out.stat().st_size / 1e6:.0f} MB)")


def gaia(ra, dec, radius):
    from astroquery.gaia import Gaia
    q = ("SELECT source_id, ra, dec, parallax, pmra, pmdec, phot_g_mean_mag, teff_gspphot "
         "FROM gaiadr3.gaia_source "
         f"WHERE 1 = CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', {ra}, {dec}, {radius}))")
    print(f"[gaia] cone ra={ra} dec={dec} r={radius} deg ...")
    t = Gaia.launch_job(q).get_results()
    out = CATS / f"gaia_dr3_{ra:.3f}_{dec:+.3f}.csv"
    t.to_pandas().to_csv(out, index=False)
    print(f"[gaia] {len(t)} sources -> {out.relative_to(ROOT)}")
    print("       (one pointing per file is fine only if you have ONE pointing; for several pointings use "
          "catalogs.gaia.query_online: true, or one larger radius that covers all of them)")


def candels():
    """Kartaltepe et al. (2015) FRACTIONAL table = VizieR J/ApJS/221/11/table3 (GOODS-S, H < 24.5),
    saved as data/catalogs/candels_k15.csv with the VizieR column labels (f(Sph), f(Dk), ...)."""
    out = CATS / "candels_k15.csv"
    if out.exists():
        print(f"[candels] already there: {out.relative_to(ROOT)}")
        return
    url = ("https://tapvizier.cds.unistra.fr/TAPVizieR/tap/sync?REQUEST=doQuery&LANG=ADQL&FORMAT=csv&QUERY="
           + urllib.parse.quote('SELECT * FROM "J/ApJS/221/11/table3"'))
    print("[candels] downloading VizieR J/ApJS/221/11/table3 (fractional classifications) ...")
    tmp = out.with_suffix(".part")
    with urllib.request.urlopen(url) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f, length=1 << 20)
    tmp.rename(out)
    import pandas as pd
    t = pd.read_csv(out, nrows=5)
    n = sum(1 for _ in open(out)) - 1
    print(f"[candels] {n} rows -> {out.relative_to(ROOT)}\n  columns: {', '.join(t.columns[:20])} ...")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("what", nargs="?", default="all", choices=["gzh", "gaia", "candels", "all"],
                    help="which catalogue (default: all)")
    ap.add_argument("--ra", type=float, help="pointing centre RA [deg] (for gaia)")
    ap.add_argument("--dec", type=float, help="pointing centre Dec [deg] (for gaia)")
    ap.add_argument("--radius", type=float, default=0.06, help="cone radius [deg] (for gaia)")
    a = ap.parse_args()
    CATS.mkdir(parents=True, exist_ok=True)
    if a.what in ("gzh", "all"):
        gzh()
    if a.what == "all" and (a.ra is None or a.dec is None):
        print("[gaia] skipped: no --ra/--dec given. Either rerun with them, or set "
              "catalogs.gaia.query_online: true in config.yaml and the pipeline fetches Gaia itself.")
    elif a.what in ("gaia", "all"):
        if a.ra is None or a.dec is None:
            ap.error("gaia needs --ra and --dec (the centre of your pointing; tools/inspect_fits.py prints it)")
        gaia(a.ra, a.dec, a.radius)
    if a.what in ("candels", "all"):
        candels()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile tools/download_mast.py
#!/usr/bin/env python
"""Download Level-3 HST _drc products from MAST into data/raw (needs internet).

    python tools/download_mast.py --ra 53.12 --dec -27.80 --radius 0.03 \
        --instrument ACS/WFC --filters F435W F606W F814W

One file per filter is kept (the smallest that covers the position), so the
three bands of one pointing arrive together. Files are typically 150-300 MB each.
"""
from __future__ import annotations

import argparse
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ra", type=float, required=True)
    ap.add_argument("--dec", type=float, required=True)
    ap.add_argument("--radius", type=float, default=0.03, help="degrees")
    ap.add_argument("--instrument", default="ACS/WFC")
    ap.add_argument("--filters", nargs="+", default=["F435W", "F606W", "F814W"])
    ap.add_argument("--out", default=str(ROOT / "data" / "raw"))
    ap.add_argument("--dry-run", action="store_true", help="list what would be downloaded")
    a = ap.parse_args()
    from astroquery.mast import Observations
    obs = Observations.query_criteria(coordinates=f"{a.ra} {a.dec}", radius=f"{a.radius} deg",
                                      obs_collection="HST", dataproduct_type="image", calib_level=3,
                                      instrument_name=a.instrument, filters=a.filters)
    print(f"{len(obs)} observations found")
    if len(obs) == 0:
        return
    prods = Observations.filter_products(Observations.get_product_list(obs), productType="SCIENCE",
                                         productSubGroupDescription="DRC", extension="fits")
    prods.sort("size")
    chosen, seen = [], set()
    for i, row in enumerate(prods):
        name = str(row["productFilename"]).lower()
        f = next((x for x in a.filters if x.lower() in name), None)
        if f and f not in seen:
            seen.add(f)
            chosen.append(i)
    if not chosen:          # names without the filter in them: take the first per obs_id
        chosen = list(range(min(len(prods), len(a.filters))))
    sel = prods[chosen]
    for r in sel:
        print(f"  {r['productFilename']}  {int(r['size']) / 1e6:.0f} MB")
    if a.dry_run:
        return
    Path(a.out).mkdir(parents=True, exist_ok=True)
    man = Observations.download_products(sel, download_dir=a.out, flat=True)
    print(man["Local Path", "Status"])


if __name__ == "__main__":
    main()

In [ ]:
%%writefile tools/inspect_catalog.py
#!/usr/bin/env python
"""Summarise the final catalogue.

    python tools/inspect_catalog.py                                   # outputs/stage8_catalogue/catalog.csv
    python tools/inspect_catalog.py path/to/catalog.csv
"""
from __future__ import annotations

import sys

import numpy as np
import pandas as pd


def main(path="outputs/stage8_catalogue/catalog.csv"):
    cat = pd.read_csv(path)
    print(f"{path}: {len(cat):,} rows x {cat.shape[1]} columns\n")
    print("=== rows per population ===")
    print(cat.classification.value_counts().to_string(), "\n")
    print("=== rows per split (buffer = inside a collar) ===")
    print(cat.split_tag.fillna("buffer").value_counts().to_string(), "\n")
    if "cr_candidate" in cat:
        print("=== cosmic-ray checks (Stage 1c) ===")
        print(f"  cosmic-ray candidates  : {int((cat.cr_candidate.astype(str) == 'True').sum())} "
              f"(no other band covers: {int(cat.cr_candidate.isna().sum())})")
        print(f"  footprints with masked cosmic-ray pixels: {int((cat.cr_pixels_masked > 0).sum())}")
    if "classification_override" in cat:
        print("=== classification differs from the cluster name (Stage 2e) ===")
        print(cat.classification_override.fillna("(cluster name kept)").value_counts().to_string(), "\n")
    gal = cat[(cat.classification == "Galaxy") & (cat.anomaly_flag == 0)]
    pc = ["p_smooth", "p_spiral", "p_edgeon", "p_irregular", "p_ambiguous"]
    print("=== galaxies ===")
    print(f"  probabilities sum to 1 : {np.allclose(gal[pc].sum(axis=1), 1, atol=1e-4)}")
    print(f"  label heritage         : {gal.is_pseudo_label.astype(str).value_counts().to_dict()}  "
          "(False = GZH label, True = model inference only)")
    print("  predicted classes      :", gal.pred_class.value_counts().to_dict())
    pm = gal.p_merger.astype(float)
    print(f"  merger probability     : median {pm.median():.3f}, range {pm.min():.3f} - {pm.max():.3f}")
    lab = gal[gal.gz_label.notna()]
    if len(lab):
        print(f"  prediction = GZH label on {(lab.gz_label == lab.pred_class).mean():.1%} of {len(lab)} labelled "
              "galaxies (includes training rows: NOT a test score - see stage6a_test_eval/test_metrics.json)")
    print("\n=== sample rows ===")
    show = ["designation", "classification", "split_tag", "gz_label", "pred_class", "p_merger", "pred_entropy",
            "mag", "snr", "gini", "m20", "sersic_n"]
    print(gal[[c for c in show if c in gal]].head(10).to_string(index=False))


if __name__ == "__main__":
    main(*sys.argv[1:])

In [ ]:
%%writefile tools/inspect_fits.py
#!/usr/bin/env python
"""Look inside an HST drizzled product: extensions, key header values, WCS,
pixel statistics, and a picture of SCI and WHT.

    python tools/inspect_fits.py data/raw/<file>_drc.fits
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from p2p.fitsio import load_image, read_meta, valid_mask  # noqa: E402


def main(path):
    import matplotlib.pyplot as plt
    from astropy.io import fits
    from astropy.visualization import AsinhStretch, ImageNormalize, PercentileInterval

    path = Path(path)
    with fits.open(path) as h:
        print("=== Extensions ===")
        h.info()
    m = read_meta(path)
    print("\n=== What the pipeline reads (Stage 1a) ===")
    for k in ("instrument", "filter", "exptime", "ndrizim", "date_obs", "targname", "bunit", "photflam", "photplam",
              "zp_ab", "pixscale_arcsec", "naxis1", "naxis2", "ra_centre", "dec_centre", "product_type"):
        print(f"  {k:16s} = {m[k]}")
    sci, wht, w = load_image(path)
    val = valid_mask(sci, wht)
    d = sci[val]
    med = np.median(d)
    mad = 1.4826 * np.median(np.abs(d - med))
    print("\n=== Pixels ===")
    print(f"  valid (WHT > 0)  = {val.mean():.1%} of the frame")
    print(f"  sky median       = {med:.5f}   robust sigma = {mad:.5f}  ({m['bunit']})")
    print(f"  min / max        = {d.min():.4g} / {d.max():.4g}")
    norm = ImageNormalize(np.where(val, sci, np.nan), interval=PercentileInterval(99.5), stretch=AsinhStretch(0.05))
    ny, nx = sci.shape
    c = np.s_[ny // 2 - 256:ny // 2 + 256, nx // 2 - 256:nx // 2 + 256]
    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(np.where(val, sci, np.nan), origin="lower", cmap="gray", norm=norm)
    ax[0].set_title(f"SCI {m['filter']} (asinh)")
    ax[1].imshow(sci[c], origin="lower", cmap="gray", norm=norm)
    ax[1].set_title("central 512 px")
    if wht is not None:
        ax[2].imshow(wht, origin="lower", cmap="viridis")
        ax[2].set_title("WHT (0 = no data)")
    for a in ax:
        a.set_xticks([])
        a.set_yticks([])
    fig.suptitle(path.name)
    fig.tight_layout()
    out = path.parent / (path.name.split(".fits")[0] + "_preview.png")
    fig.savefig(out, dpi=90)
    print(f"\nPreview saved: {out}")
    plt.show()


if __name__ == "__main__":
    if len(sys.argv) < 2:
        sys.exit(__doc__)
    main(sys.argv[1])

In [ ]:
%%writefile tools/show_stage.py
#!/usr/bin/env python
"""Explain one stage: its purpose (from the stage file), what it wrote, and the
headline numbers from its JSON outputs.

    python tools/show_stage.py 1c
    python tools/show_stage.py 6a --config my_other_config.yaml
"""
from __future__ import annotations

import argparse
import importlib
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from p2p.context import STAGES, Ctx  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("stage")
    ap.add_argument("--config", default="config.yaml")
    a = ap.parse_args()
    info = {k: (m, d, t) for k, m, d, t in STAGES}
    if a.stage not in info:
        sys.exit(f"unknown stage {a.stage}; choose from {list(info)}")
    mod, folder, title = info[a.stage]
    ctx = Ctx(ROOT, a.config)
    doc = importlib.import_module(f"p2p.stages.{mod}").__doc__
    print(f"Stage {a.stage} - {title}\nsrc/p2p/stages/{mod}.py\n")
    print(doc.strip(), "\n")
    out = ctx.out / folder
    if not out.exists():
        print(f"(no outputs yet in {ctx.rel(out)} - run: python run_pipeline.py --only {a.stage})")
        return
    print(f"Files in {ctx.rel(out)}:")
    for f in sorted(out.iterdir()):
        print(f"  {f.name:45s} {f.stat().st_size / 1e6:8.2f} MB")
    for f in sorted(out.glob("*.json")):
        try:
            d = json.loads(f.read_text())
        except Exception:
            continue
        text = json.dumps(d, indent=1, default=str)
        print(f"\n--- {f.name} ---\n" + (text if len(text) < 2500 else text[:2500] + "\n ... (truncated)"))
    log = ctx.out / "run_log.json"
    if log.exists():
        r = json.loads(log.read_text()).get(a.stage)
        if r:
            print(f"\nlast run: {r['status']} in {r['seconds']} s at {r['finished']}")


if __name__ == "__main__":
    main()

In [ ]:
!{sys.executable} run_pipeline.py --list

## 3. Data

**Images** go in `data/raw/`: HST ACS/WFC `_drc` (or `_drz`) FITS files, three filters per pointing. Several pointings at different RA/Dec are fine: Stage 1a groups files that overlap by ≥ 50 % into one pointing, and Stage 0 splits the sky of every pointing into train / dev / test with a buffer collar.

**Catalogues** go in `data/catalogs/` (the file name must contain the keyword):
- Galaxy Zoo: Hubble (`gz_hubble…`) — required, morphology labels
- Gaia DR3 (`gaia…`) — star ground truth; with several pointings it is easiest to set `catalogs.gaia.query_online: true` in `config.yaml`
- CANDELS Kartaltepe et al. 2015 (`candels…` / `k15…`) — cross-catalogue test

On Colab you can upload files to the Drive folder, or download them with the tools below (needs internet).

In [ ]:
# Option A: upload from your computer (Colab only)
if IN_COLAB:
    from google.colab import files
    # up = files.upload()          # then move FITS to data/raw and tables to data/catalogs
    pass
print(sorted(os.listdir("data/raw")), sorted(os.listdir("data/catalogs")))

In [ ]:
# Option B: download from MAST / Gaia / VizieR. Repeat the MAST line once per pointing (RA, Dec in degrees).
RA, DEC = 53.12, -27.80
# !{sys.executable} tools/download_mast.py --ra {RA} --dec {DEC} --radius 0.03 --filters F435W F606W F814W --dry-run
# !{sys.executable} tools/download_mast.py --ra {RA} --dec {DEC} --radius 0.03 --filters F435W F606W F814W
# !{sys.executable} tools/download_catalogs.py gzh
# !{sys.executable} tools/download_catalogs.py candels
# !{sys.executable} tools/download_catalogs.py gaia --ra {RA} --dec {DEC}

In [ ]:
# Check the inputs before running
!{sys.executable} tools/check_catalog_columns.py
import glob
for f in sorted(glob.glob("data/raw/*.fits")):
    print(f)
# !{sys.executable} tools/inspect_fits.py data/raw/<file>_drc.fits

## 4. Run the pipeline

### 4.1 Helpers
`show_stage(key)` prints the stage's purpose and headline numbers and displays every figure it wrote.

In [ ]:
import json, glob
from IPython.display import Image, display, Markdown

def show_stage(key, max_images=20, width=900):
    """Explain one stage and show its outputs (text summary + all PNG figures)."""
    import subprocess
    r = subprocess.run([sys.executable, "tools/show_stage.py", key], capture_output=True, text=True)
    print(r.stdout + r.stderr)
    sys.path.insert(0, "src")
    from p2p.context import STAGE_DIR
    folder = os.path.join("outputs", STAGE_DIR[key])
    pngs = sorted(glob.glob(os.path.join(folder, "*.png")))
    for p in pngs[:max_images]:
        display(Markdown(f"**{os.path.basename(p)}**"))
        display(Image(filename=p, width=width))
    if len(pngs) > max_images:
        print(f"... {len(pngs) - max_images} more figures in {folder}")

`!{sys.executable} …` runs the scripts with the same Python as this notebook's kernel (a plain `!python` can pick up a different Python in local Jupyter).

### 4.2 Run everything at once
Uncomment to run all 27 stages in order (the order is fixed in `src/p2p/context.py`: Stage 5 runs before Stage 4). `--from <stage>` resumes after an interruption.

In [ ]:
# !{sys.executable} run_pipeline.py
# !{sys.executable} run_pipeline.py --from 2a
# !{sys.executable} run_pipeline.py --from 1a --to 1e

### 4.3 Or run stage by stage
Each stage reads the saved outputs of the earlier ones, so you can stop and continue at any point.

#### Phase 0 — Data governance

**Stage 0 — Sky-region partitioning**

fix the train / dev / test / external split BY SKY REGION before any pixel is analysed, so overlapping pointings cannot leak one object into two buckets.

In [ ]:
!{sys.executable} run_pipeline.py --only 0
show_stage("0")

#### Phase 1 — Ingestion and instrument-aware preprocessing

**Stage 1a — Archive acquisition / local scan**

know exactly which science products the run uses and verify them.

In [ ]:
!{sys.executable} run_pipeline.py --only 1a
show_stage("1a")

**Stage 1b — Frame quality control (CAE)**

flag detector artefacts (satellite trails, bad drizzle regions, chip edges, hot-pixel clusters) that survive drizzle combination.

In [ ]:
!{sys.executable} run_pipeline.py --only 1b
show_stage("1b")

**Stage 1c — Detection, deblending, aligned cutouts**

locate every source and register all filters into one cutout.

In [ ]:
!{sys.executable} run_pipeline.py --only 1c
show_stage("1c")

**Stage 1d — Cross-filter PSF homogenisation**

make every band of a cutout share one resolution, so chromatic PSF differences are not mistaken for galaxy structure.

In [ ]:
!{sys.executable} run_pipeline.py --only 1d
show_stage("1d")

**Stage 1e — Pass-1 photometry and morphology**

a provisional table of physical parameters for every source.

In [ ]:
!{sys.executable} run_pipeline.py --only 1e
show_stage("1e")

#### Phase 2 — Unsupervised object segregation

**Stage 2a — Representation learning + fusion**

one feature space that captures both pixel structure and physics.

In [ ]:
!{sys.executable} run_pipeline.py --only 2a
show_stage("2a")

**Stage 2b — Unsupervised clustering**

group sources into populations without any label.

In [ ]:
!{sys.executable} run_pipeline.py --only 2b
show_stage("2b")

**Stage 2c — Baseline segregation + Gaia match**

a simple, transparent Star / Galaxy / Noise separator - the yardstick the learned segregation (Stages 2a-2d) must beat (Objective 2).

In [ ]:
!{sys.executable} run_pipeline.py --only 2c
show_stage("2c")

**Stage 2d — Semantic audit**

turn abstract GMM cluster IDs into Star / Galaxy / Noise, and measure how good the learned segregation is against Gaia.

In [ ]:
!{sys.executable} run_pipeline.py --only 2d
show_stage("2d")

**Stage 2e — Continuum handling + outlier gating**

do not throw away real galaxies that sit between populations, and do not silently drop true anomalies either.

In [ ]:
!{sys.executable} run_pipeline.py --only 2e
show_stage("2e")

#### Phase 3 — Label anchoring and three-way routing

**Stage 3a — GZH label anchoring**

attach crowd-sourced morphology labels to the galaxy track.

In [ ]:
!{sys.executable} run_pipeline.py --only 3a
show_stage("3a")

**Stage 3b — Star calibration routing**

point sources skip morphology classification and become PSF tracers.

In [ ]:
!{sys.executable} run_pipeline.py --only 3b
show_stage("3b")

**Stage 3c — Noise false-positive model**

characterise spurious detections - the Noise population has no external catalogue, so this model is also its validation.

In [ ]:
!{sys.executable} run_pipeline.py --only 3c
show_stage("3c")

#### Phase 5 — Optical calibration feedback (runs before Phase 4)

**Stage 5a — Empirical PSF construction**

a localised, high-quality PSF built from the pipeline's own stars.

In [ ]:
!{sys.executable} run_pipeline.py --only 5a
show_stage("5a")

**Stage 5b — Pass-2 morphology refinement**

physics-calibrated morphology for the galaxy track.

In [ ]:
!{sys.executable} run_pipeline.py --only 5b
show_stage("5b")

#### Phase 4 — Supervised multi-task learning and ensemble

**Stage 4a — Dual-branch multi-task training**

the core supervised model of the proposal.

In [ ]:
!{sys.executable} run_pipeline.py --only 4a
show_stage("4a")

**Stage 4b — Ensemble consensus**

combine models that make different errors.

In [ ]:
!{sys.executable} run_pipeline.py --only 4b
show_stage("4b")

**Stage 4c — Probability calibration**

make predicted confidence match empirical accuracy.

In [ ]:
!{sys.executable} run_pipeline.py --only 4c
show_stage("4c")

**Stage 4d — Controlled ablation**

measure the value the hybrid design adds (Objective 3).

In [ ]:
!{sys.executable} run_pipeline.py --only 4d
show_stage("4d")

#### Phase 6 — In-domain evaluation and explainability

**Stage 6a — In-domain test (run once)**

the final, honest in-domain score.

> Run this stage **once**, after all tuning on train/dev is finished: it opens the held-out test region.

In [ ]:
!{sys.executable} run_pipeline.py --only 6a
show_stage("6a")

**Stage 6b — Uncertainty audit**

does the model hesitate where humans disagree?

In [ ]:
!{sys.executable} run_pipeline.py --only 6b
show_stage("6b")

**Stage 6c — Explainable AI (Grad-CAM, SHAP)**

show what drives each prediction.

In [ ]:
!{sys.executable} run_pipeline.py --only 6c
show_stage("6c")

#### Phase 7 — Cross-catalogue robustness

**Stage 7a — Taxonomy alignment + ceiling**

separate label-scheme differences from model failure.

In [ ]:
!{sys.executable} run_pipeline.py --only 7a
show_stage("7a")

**Stage 7b — Cross-catalogue evaluation**

how well does the frozen model generalise to an independent expert scheme?

In [ ]:
!{sys.executable} run_pipeline.py --only 7b
show_stage("7b")

**Stage 7c — Generalisation disentanglement**

report true degradation, not a raw score drop.

In [ ]:
!{sys.executable} run_pipeline.py --only 7c
show_stage("7c")

#### Phase 8 — Unified catalogue

**Stage 8 — Unified catalogue (catalog.csv)**

one deliverable that a scientist can filter with a single column.

In [ ]:
!{sys.executable} run_pipeline.py --only 8
show_stage("8")

## 5. Results — `catalog.csv`

In [ ]:
!{sys.executable} tools/inspect_catalog.py

In [ ]:
import pandas as pd
cat = pd.read_csv("outputs/stage8_catalogue/catalog.csv")
cols = pd.read_csv("outputs/stage8_catalogue/catalog_columns.csv")
print(cat.shape)
display(cat.head(10))
display(cols)
print(json.dumps(json.load(open("outputs/stage8_catalogue/delivery_checks.json")), indent=2))

In [ ]:
# Run log: status and duration of every stage
log = json.load(open("outputs/run_log.json"))
pd.DataFrame.from_dict(log, orient="index")

## Notes
- **Re-running on new data**: put the new FITS files in `data/raw/` (remove the old ones), then run from Stage 0. Outputs are overwritten in `outputs/`; copy the folder first if you want to keep a run.
- **Several pointings**: give Gaia per pointing (`tools/download_catalogs.py gaia --ra … --dec …` for each) or set `catalogs.gaia.query_online: true`. To use one pointing as the external CANDELS field, list a filename substring in `partition.external_patterns`.
- **Colab sessions end** after inactivity; because the project folder is on Drive, re-run Section 1 and continue with `--from <stage>`.